# TRIPLE CLASIFICACIÓN


In [1]:
import os
import nibabel as nib
import matplotlib.pyplot as plt
import numpy as np
import re
import pandas as pd
import cv2

def rangeDisplay01(im, flag_GLOBAL=True):
    """
    Normaliza los valores de una imagen en el rango [0, 1].
    
    Parámetros:
    im (ndarray): La imagen a normalizar.
    flag_GLOBAL (bool): Indica si la normalización es global (True) o por bandas (False).
    
    Retorna:
    ndarray: La imagen normalizada.
    """
    # Convertir la imagen a tipo float para asegurar precisión en los cálculos.
    im = im.astype(float)
    
    if flag_GLOBAL:
        # Normalización global de toda la imagen.
        im = (im - im.min()) / (im.max() - im.min())
    else:
        # Normalización por bandas para imágenes multiespectrales.
        for band in range(im.shape[2]):
            im[:, :, band] = (im[:, :, band] - im[:, :, band].min()) / (im[:, :, band].max() - im[:, :, band].min())
            # Nota: plt.imshow con datos RGB requiere rango [0..1] para floats y [0..255] para enteros.
    
    return im

def buscar_archivos_con_regex(directorio, patrones):
    """
    Busca archivos en un directorio que coincidan con una lista de patrones regex.
    
    Parámetros:
    directorio (str): Ruta del directorio donde buscar.
    patrones (list of str): Lista de patrones regex para buscar en los nombres de los archivos.
    
    Retorna:
    list of str: Lista de rutas de los archivos que coinciden con los patrones.
    """
    archivos_coincidentes = []
    # Recorrer el directorio y subdirectorios.
    for raiz, directorios, archivos in os.walk(directorio):
        for archivo in archivos:
            # Obtener la ruta completa del archivo.
            ruta_archivo = os.path.join(raiz, archivo)
            for patron in patrones:
                # Comprobar si el nombre del archivo coincide con algún patrón.
                if re.match(patron, archivo):
                    archivos_coincidentes.append(ruta_archivo)
                    break  # Salir del bucle de patrones si hay coincidencia.
    return archivos_coincidentes

# Definir el directorio y patrones de búsqueda.
directorio = './NIFTI PREPROC'
patrones = [r'.*1CT.*', r'.*FDG.*', r'.*A90.*']  # Patrones para buscar en los nombres de los archivos

# Llamar a la función de búsqueda con los patrones definidos.
archivos_encontrados = buscar_archivos_con_regex(directorio, patrones)

# Mostrar los archivos encontrados, si los hay.
if archivos_encontrados:
    print("Archivos encontrados:")
    for archivo in archivos_encontrados:
        print(archivo)
else:
    print("No se encontraron archivos que coincidan con los patrones especificados.")



Archivos encontrados:
./NIFTI PREPROC\mwc1CT_P01_HVN.nii
./NIFTI PREPROC\mwc1CT_P02_HVN.nii
./NIFTI PREPROC\mwc1CT_P03_HVN.nii
./NIFTI PREPROC\mwc1CT_P04_HVN.nii
./NIFTI PREPROC\mwc1CT_P05_HVN.nii
./NIFTI PREPROC\mwc1CT_P06_HVN.nii
./NIFTI PREPROC\mwc1CT_P07_HVN.nii
./NIFTI PREPROC\mwc1CT_P08_HVN.nii
./NIFTI PREPROC\mwc1CT_P09_HVN.nii
./NIFTI PREPROC\mwc1CT_P10_HVN.nii
./NIFTI PREPROC\mwc1CT_P11_HVN.nii
./NIFTI PREPROC\mwc1CT_P12_HVN.nii
./NIFTI PREPROC\mwc1CT_P13_HVN.nii
./NIFTI PREPROC\mwc1CT_P14_HVN.nii
./NIFTI PREPROC\mwc1CT_P15_HVN.nii
./NIFTI PREPROC\mwc1CT_P16_HVN.nii
./NIFTI PREPROC\mwc1CT_P17_HVN.nii
./NIFTI PREPROC\mwc1CT_P18_HVN.nii
./NIFTI PREPROC\mwc1CT_P19_HVN.nii
./NIFTI PREPROC\mwc1CT_P20_HVN.nii
./NIFTI PREPROC\mwc1CT_P21_HVN.nii
./NIFTI PREPROC\mwc1CT_P22_HVN.nii
./NIFTI PREPROC\mwc1CT_P23_HVN.nii
./NIFTI PREPROC\mwc1CT_P24_HVN.nii
./NIFTI PREPROC\mwc1CT_P25_HVN.nii
./NIFTI PREPROC\mwc1CT_P26_HVN.nii
./NIFTI PREPROC\mwc1CT_P27_HVN.nii
./NIFTI PREPROC\mwc1CT_P28_HVN.ni

In [2]:
import os
import nibabel as nib
import numpy as np
import shutil
import cv2

# Lista para almacenar las imágenes cargadas
imagenes = []
for ruta in archivos_encontrados:
    # Cargar la imagen NIfTI
    imagen_nifti = nib.load(ruta)
    
    # Obtener los datos de la imagen como una matriz numpy
    imagen_nifti = imagen_nifti.get_fdata()
    
    # Sustituir los valores NaN por 0
    imagen_nifti = np.nan_to_num(imagen_nifti, nan=0.0)
    
    # Añadir la imagen procesada a la lista
    imagenes.append(imagen_nifti)

# Lista para almacenar las imágenes finales procesadas
imagenes_finales = []
for imagen in imagenes:
    # Calcular la media de los datos en tres partes diferentes
    ## Se hace alternando la información, la primera imagen corresponde a las medias de las capas 0,3,6,9..., la segunda las capas 1,4,7.., etc
    img1 = np.mean(imagen[:, :, 0::3], axis=2)
    img2 = np.mean(imagen[:, :, 1::3], axis=2)
    img3 = np.mean(imagen[:, :, 2::3], axis=2)
    
    # Combinar las medias en una imagen con tres canales
    imagen = np.stack((img1, img2, img3), axis=2)
    
    # Añadir la imagen final a la lista
    imagenes_finales.append(imagen)

# Definir la carpeta de destino para guardar las imágenes
carpeta_destino = './NIFTI_TRIPLE'
if os.path.exists(carpeta_destino):
    # Eliminar la carpeta si ya existe junto con su contenido
    shutil.rmtree(carpeta_destino)
    print(f"La carpeta {carpeta_destino} y su contenido han sido eliminados.")

# Crear la carpeta de destino
os.makedirs(carpeta_destino)

# Iterar sobre las rutas de los archivos y las imágenes finales
for ruta, imagen_final in zip(archivos_encontrados, imagenes_finales):
    # Obtener el nombre del archivo sin la extensión
    nombre_archivo = os.path.basename(ruta).split(".")[0]
    
    # Generar la ruta completa para guardar la imagen en la carpeta de destino
    ruta_destino = os.path.join(carpeta_destino, f"{nombre_archivo}.jpg")
    
    # Normalizar la imagen en el rango [0, 1] (sin normalización global)
    imagen_final = rangeDisplay01(imagen_final, False)
    
    # Convertir la imagen a tipo float32 para la multiplicación
    imagen_final = imagen_final.astype(np.float32)
    
    # Escalar la imagen a [0, 255] para guardarla como JPG
    imagen_final *= 255
    
    # Guardar la imagen en formato JPG, convirtiendo de RGB a BGR
    cv2.imwrite(ruta_destino, cv2.cvtColor(imagen_final, cv2.COLOR_RGB2BGR))
    
    print(f"Imagen guardada: {ruta_destino}")



La carpeta ./NIFTI_TRIPLE y su contenido han sido eliminados.
Imagen guardada: ./NIFTI_TRIPLE\mwc1CT_P01_HVN.jpg
Imagen guardada: ./NIFTI_TRIPLE\mwc1CT_P02_HVN.jpg
Imagen guardada: ./NIFTI_TRIPLE\mwc1CT_P03_HVN.jpg
Imagen guardada: ./NIFTI_TRIPLE\mwc1CT_P04_HVN.jpg
Imagen guardada: ./NIFTI_TRIPLE\mwc1CT_P05_HVN.jpg
Imagen guardada: ./NIFTI_TRIPLE\mwc1CT_P06_HVN.jpg
Imagen guardada: ./NIFTI_TRIPLE\mwc1CT_P07_HVN.jpg
Imagen guardada: ./NIFTI_TRIPLE\mwc1CT_P08_HVN.jpg
Imagen guardada: ./NIFTI_TRIPLE\mwc1CT_P09_HVN.jpg
Imagen guardada: ./NIFTI_TRIPLE\mwc1CT_P10_HVN.jpg
Imagen guardada: ./NIFTI_TRIPLE\mwc1CT_P11_HVN.jpg
Imagen guardada: ./NIFTI_TRIPLE\mwc1CT_P12_HVN.jpg
Imagen guardada: ./NIFTI_TRIPLE\mwc1CT_P13_HVN.jpg
Imagen guardada: ./NIFTI_TRIPLE\mwc1CT_P14_HVN.jpg
Imagen guardada: ./NIFTI_TRIPLE\mwc1CT_P15_HVN.jpg
Imagen guardada: ./NIFTI_TRIPLE\mwc1CT_P16_HVN.jpg
Imagen guardada: ./NIFTI_TRIPLE\mwc1CT_P17_HVN.jpg
Imagen guardada: ./NIFTI_TRIPLE\mwc1CT_P18_HVN.jpg
Imagen guardada: ./N

In [3]:
###Ahora, se va a realizar un dataset con todas las modalidades mezcladas donde la primera columna será la ruta de la imagen y la segunda será la etiqueta,
###si la etiqueta es 2, se descarta, solo queremos 0 o 1
df = pd.read_csv('etiquetas.txt', delimiter='\t')
df = df.dropna(how="all")

directorio = './NIFTI_TRIPLE'  # Cambia esto por la ruta de tu directorio

# Obtener la lista de archivos en el directorio
archivos = os.listdir(directorio)

pares = []
# Iterar sobre la lista de archivos e imprimir sus nombres
for archivo in archivos:
    archivo_sp = archivo.split("_")
    paciente = archivo_sp[1]
    hospital = archivo_sp[2].split(".")[0]
    etiqueta = df.loc[(df['Pac'] == paciente) & (df['Centro'] == hospital), 'Etiqueta'].values
    if etiqueta != 2:
        ruta_completa = os.path.join(directorio, archivo)  # Obtener la ruta completa
        pares.append((ruta_completa,etiqueta[0]))
    
    
df_defi = pd.DataFrame(pares, columns=['Ruta', 'Etiqueta'])
print(df_defi)

                                  Ruta  Etiqueta
0    ./NIFTI_TRIPLE\mwc1CT_P01_HVN.jpg       1.0
1    ./NIFTI_TRIPLE\mwc1CT_P02_HVN.jpg       1.0
2    ./NIFTI_TRIPLE\mwc1CT_P03_HVN.jpg       0.0
3    ./NIFTI_TRIPLE\mwc1CT_P05_HVN.jpg       1.0
4    ./NIFTI_TRIPLE\mwc1CT_P06_HVN.jpg       1.0
..                                 ...       ...
229    ./NIFTI_TRIPLE\wFDG_P71_HVN.jpg       0.0
230    ./NIFTI_TRIPLE\wFDG_P72_H9O.jpg       1.0
231    ./NIFTI_TRIPLE\wFDG_P73_H9O.jpg       0.0
232    ./NIFTI_TRIPLE\wFDG_P74_H9O.jpg       1.0
233    ./NIFTI_TRIPLE\wFDG_P75_H9O.jpg       0.0

[234 rows x 2 columns]


In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import pandas as pd
from torch.utils.data import Dataset
import torch.nn.init as init
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import numpy as np
import os
import random
from PIL import Image
from torchvision import models, transforms, utils
from torchvision.transforms import v2
import re
from skorch import NeuralNetClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split


In [5]:
# Verificar si CUDA está disponible
cuda_available = torch.cuda.is_available()

# Imprimir la versión de PyTorch y la disponibilidad de CUDA
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {cuda_available}")

# Imprimir el nombre de la GPU si CUDA está disponible
if cuda_available:
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
    # Obtener la versión de CUDA
    cuda_version = torch.version.cuda
    print(f"CUDA version: {cuda_version}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

PyTorch version: 2.2.0
CUDA available: True
GPU name: NVIDIA GeForce RTX 3050 Laptop GPU
CUDA version: 12.1
Using device: cuda


## PRUEBA SIN TL SIN OC

A continuación, se muestran todas las funciones de entrenamiento que se han probado. Se comentará exhaustivamente esta primera. Las demás comparten el código excepto algunos fragmentos. Lo más recomendable es ir comparando con esta función y ver cómo se van añadiendo diferentes funcionalidades.

In [6]:
# Función para calcular el accuracy
def accuracy(outputs, labels):
    preds = outputs > 0.5  # Si la salida es mayor que 0.5, se considera una predicción positiva (1), de lo contrario es negativa (0)
    correct = (preds == labels.byte())  # Compara las predicciones con las etiquetas reales para ver cuántas son correctas
    acc = correct.float().sum() / labels.size(0)  # Suma las predicciones correctas y divide por el número total de etiquetas para obtener la precisión
    return acc.item()  # Devuelve la precisión como un valor escalar

# Función para calcular el recall
def recall(outputs, labels):
    preds = outputs > 0.5  # Si la salida es mayor que 0.5, se considera una predicción positiva (1), de lo contrario es negativa (0)

    # Mueve los tensores a la CPU para asegurar la compatibilidad en operaciones subsecuentes
    preds = preds.cpu()
    labels = labels.cpu()

    TP = sum((preds == 1) & (labels == 1))  # Calcula los verdaderos positivos (TP): predicciones positivas que son correctas
    FN = sum((preds == 0) & (labels == 1))  # Calcula los falsos negativos (FN): etiquetas reales positivas que fueron predichas como negativas

    recall_score = TP / (TP + FN) if (TP + FN) > 0 else 0  # Calcula el recall: TP dividido por la suma de TP y FN. Si TP + FN es 0, el recall es 0
    return recall_score  # Devuelve el valor del recall

# Función para calcular la precisión (precision)
def precision(outputs, labels):
    preds = outputs > 0.5  # Si la salida es mayor que 0.5, se considera una predicción positiva (1), de lo contrario es negativa (0)

    # Mueve los tensores a la CPU para asegurar la compatibilidad en operaciones subsecuentes
    preds = preds.cpu()
    labels = labels.cpu()

    TP = sum((preds == 1) & (labels == 1))  # Calcula los verdaderos positivos (TP): predicciones positivas que son correctas
    FP = sum((preds == 1) & (labels == 0))  # Calcula los falsos positivos (FP): predicciones positivas que son incorrectas

    precision_score = TP / (TP + FP) if (TP + FP) > 0 else 0  # Calcula la precisión: TP dividido por la suma de TP y FP. Si TP + FP es 0, la precisión es 0
    return precision_score  # Devuelve el valor de la precisión


#####Parámetros
#####   dataset: el conjunto de datos, siendo una columna la ruta y otra la etiqueta de la imagen
#####   num_folds_outer: el número de folds de test que se ejecutarán
#####   num_epochs: el número total de épocas que se entrenará
#####   op: el optimziador usado
#####   batch_size: tamaño de batch para entrenamiento
#####   lr: learning rate para el optimizador
#####   data_aug: True o False, para si se hace aumento de datos o no
#####   ini: un char representando uno de los optimizadores que se pueden usar
def prueba_k_fold_resnet_no_pre(dataset, num_folds, num_epochs, op, batch_size, lr, data_aug, ini):
    # Transformaciones para preprocesar las imágenes
    # Verifica si el aumento de datos está activado
    if data_aug:
        transform_t = v2.Compose([
            v2.Resize((224, 224)),  # Redimensiona la imagen a 224x224 píxeles
            v2.RandomHorizontalFlip(),  # Aplica un volteo horizontal aleatorio a la imagen
            v2.RandomVerticalFlip(),  # Aplica un volteo vertical aleatorio a la imagen
            v2.ToImage(),  # Convierte los datos en una imagen
            v2.ToDtype(torch.float32, scale=True),  # Convierte la imagen al tipo de datos float32, escalando los valores
            v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),  # Normaliza la imagen usando los valores de media y desviación estándar de ImageNet
        ])

        # Transformaciones para los datos de validación (sin aumento de datos)
        transform_v = v2.Compose([
            v2.Resize((224, 224)),  # Redimensiona la imagen a 224x224 píxeles
            v2.ToImage(),  # Convierte los datos en una imagen
            v2.ToDtype(torch.float32, scale=True),  # Convierte la imagen al tipo de datos float32, escalando los valores
            v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),  # Normaliza la imagen usando los valores de media y desviación estándar de ImageNet
        ])
    else:
        # Transformaciones para los datos de entrenamiento sin aumento de datos
        transform_t = v2.Compose([
            v2.Resize((224, 224)),  # Redimensiona la imagen a 224x224 píxeles
            v2.ToImage(),  # Convierte los datos en una imagen
            v2.ToDtype(torch.float32, scale=True),  # Convierte la imagen al tipo de datos float32, escalando los valores
            v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),  # Normaliza la imagen usando los valores de media y desviación estándar de ImageNet
        ])

        # Transformaciones para los datos de validación (sin aumento de datos)
        transform_v = v2.Compose([
            v2.Resize((224, 224)),  # Redimensiona la imagen a 224x224 píxeles
            v2.ToImage(),  # Convierte los datos en una imagen
            v2.ToDtype(torch.float32, scale=True),  # Convierte la imagen al tipo de datos float32, escalando los valores
            v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),  # Normaliza la imagen usando los valores de media y desviación estándar de ImageNet
        ])
    
    # Modelo personalizado basado en ResNet
    class CustomResNet(nn.Module):
        def __init__(self, num_classes_binary=1):
            super(CustomResNet, self).__init__()

            # Carga el modelo sin pesos de entrenamiento
            resnet = models.resnet18(weights=None)

            # Extrae todas las capas de la ResNet menos la última capa fully connected
            self.features = nn.Sequential(*list(resnet.children())[:-1])
    
            # Define una nueva capa fully connected para la clasificación binaria
            self.fc_binary = nn.Linear(512, num_classes_binary)
            
            # Inicializa los pesos de la red según el método especificado
            if (ini == "G"):
                self.apply(self.init_weights_glorot)  # Inicialización de Glorot (Xavier)
            elif(ini == "H"):
                self.apply(self.init_weights_he)  # Inicialización de He
            elif (ini == "N"):
                self.apply(self.init_weights_normal)  # Inicialización normal
            elif(ini == "U"):
                self.apply(self.init_weights_uniform)  # Inicialización uniforme
            elif (ini == "O"):
                self.apply(self.init_weights_orthogonal)  # Inicialización ortogonal
            elif(ini == "Z"):
                self.apply(self.init_weights_zeros)  # Inicialización a ceros
            elif(ini == "ON"):
                self.apply(self.init_weights_ones)  # Inicialización a unos

        # Inicialización de pesos con el método de Glorot (Xavier)
        def init_weights_glorot(self, m):
            if isinstance(m, nn.Linear):
                init.xavier_uniform_(m.weight)  # Inicializa los pesos de forma uniforme según Glorot
                if m.bias is not None:
                    init.constant_(m.bias, 0)  # Inicializa los sesgos a cero
    
        # Inicialización de pesos con el método de He
        def init_weights_he(self, m):
            if isinstance(m, nn.Linear):
                init.kaiming_uniform_(m.weight, mode='fan_in', nonlinearity='relu')  # Inicializa los pesos según He
                if m.bias is not None:
                    init.constant_(m.bias, 0)  # Inicializa los sesgos a cero
    
        # Inicialización de pesos con distribución normal
        def init_weights_normal(self, m):
            if isinstance(m, nn.Linear):
                init.normal_(m.weight, mean=0, std=0.01)  # Inicializa los pesos con una distribución normal con media 0 y desviación estándar 0.01
                if m.bias is not None:
                    init.constant_(m.bias, 0)  # Inicializa los sesgos a cero
    
        # Inicialización de pesos con distribución uniforme
        def init_weights_uniform(self, m):
            if isinstance(m, nn.Linear):
                init.uniform_(m.weight, a=-0.1, b=0.1)  # Inicializa los pesos de forma uniforme entre -0.1 y 0.1
                if m.bias is not None:
                    init.constant_(m.bias, 0)  # Inicializa los sesgos a cero
    
        # Inicialización de pesos ortogonal
        def init_weights_orthogonal(self, m):
            if isinstance(m, nn.Linear):
                init.orthogonal_(m.weight)  # Inicializa los pesos de forma ortogonal
                if m.bias is not None:
                    init.constant_(m.bias, 0)  # Inicializa los sesgos a cero
    
        # Inicialización de pesos a ceros
        def init_weights_zeros(self, m):
            if isinstance(m, nn.Linear):
                init.zeros_(m.weight)  # Inicializa todos los pesos a cero
                if m.bias is not None:
                    init.zeros_(m.bias)  # Inicializa los sesgos a cero
    
        # Inicialización de pesos a unos
        def init_weights_ones(self, m):
            if isinstance(m, nn.Linear):
                init.ones_(m.weight)  # Inicializa todos los pesos a uno
                if m.bias is not None:
                    init.ones_(m.bias)  # Inicializa los sesgos a uno
    
        # Método forward para el paso de la red
        def forward(self, x):
            x = self.features(x)  # Pasa la entrada a través de la parte de características de la red
            x = x.view(x.size(0), -1)  # Aplana las características a un vector de 1D para la capa fully connected
    
            # Pasa la salida a través de la capa fully connected personalizada para la clasificación binaria
            out_binary = self.fc_binary(x)
    
            return out_binary  # Devuelve la salida para la clasificación binaria
    
    # Define una clase personalizada para el conjunto de datos
    class CustomDataset(Dataset):
        def __init__(self, dataframe, transform=None):
            """
            Inicializa el conjunto de datos personalizado.

            Args:
                dataframe (pd.DataFrame): DataFrame que contiene rutas a las imágenes y etiquetas.
                transform (callable, optional): Transformaciones que se aplicarán a cada imagen.
            """
            self.data = dataframe  # Almacena el DataFrame con rutas y etiquetas
            self.transform = transform  # Almacena las transformaciones, si se proporcionan
    
        def __len__(self):
            """
            Devuelve la longitud del conjunto de datos.

            Returns:
                int: Número de ejemplos en el conjunto de datos.
            """
            return len(self.data)  # Retorna el número de filas en el DataFrame
    
        def __getitem__(self, idx):
            """
            Obtiene una imagen y su etiqueta correspondiente por índice.

            Args:
                idx (int): Índice del ejemplo a obtener.

            Returns:
                tuple: Una imagen transformada y su etiqueta binaria.
            """
            # Obtiene la ruta de la imagen de la primera columna del DataFrame
            img_path = self.data.iloc[idx, 0]

            # Abre la imagen usando PIL y la convierte a RGB
            image = Image.open(img_path).convert('RGB')

            # Obtiene la etiqueta binaria de la segunda columna del DataFrame y la convierte a tensor
            label_binary = torch.tensor(self.data.iloc[idx, 1], dtype=torch.float32)

            # Aplica las transformaciones a la imagen, si están definidas
            if self.transform:
                image = self.transform(image)

            # Retorna la imagen (posiblemente transformada) y la etiqueta binaria
            return image, label_binary
        
    #####5 fold CV
    from sklearn.model_selection import StratifiedKFold
    
    # Definir el número de folds
    criterion_binary = nn.BCEWithLogitsLoss()
    kf = StratifiedKFold(n_splits=num_folds, shuffle=True, random_state=33)
    
    # Listas para almacenar las pérdidas de entrenamiento y validación de cada fold
    train_losses_folds = []
    val_losses_folds = []
    
    train_accs_folds = []
    train_rec_folds =  []
    train_preci_folds = []

    val_accs_folds = []
    val_rec_folds =  []
    val_preci_folds = []
    
    # Itera sobre cada fold generado por StratifiedKFold
    for fold, (train_index, val_index) in enumerate(kf.split(np.zeros(len(dataset)),dataset['Etiqueta'])):
        print(f"Fold {fold + 1}")
    
        # Dividir el conjunto de datos en entrenamiento y validación para este fold
        train_data_fold = dataset.iloc[train_index]
        val_data_fold = dataset.iloc[val_index]
    
        # Crear instancias del conjunto de datos personalizado para este fold
        train_dataset_fold = CustomDataset(dataframe=train_data_fold, transform=transform_t)
        val_dataset_fold = CustomDataset(dataframe=val_data_fold, transform=transform_v)
    
        # Crear DataLoader para cargar los datos en lotes
        train_loader_fold = DataLoader(dataset=train_dataset_fold, batch_size=batch_size, shuffle=True)
        val_loader_fold = DataLoader(dataset=val_dataset_fold, batch_size=len(val_dataset_fold), shuffle=False) 
    
        # Re-inicializar el modelo para cada fold
        model = CustomResNet(num_classes_binary=1).to(device)

        ##En este forma de entrenar, solo un optimizador
        optimizer = op(model.parameters(), lr = lr)

        # Entrenamiento y validación para este fold
        train_losses_fold = []
        val_losses_fold = []
        train_accuracy_fold = []
        val_accuracy_fold = []
        train_recall_fold = []
        val_recall_fold = []
        train_preci_fold = []
        val_preci_fold = []

        for epoch in range(num_epochs):
            # Entrenamiento del modelo
            # Fase de entrenamiento
            model.train()
            loss_total_acumulada = 0.0
            lossT_bin_total = 0.0
            correct_train = 0
            recall_value_train = 0
            precision_value_train = 0
            for images, bin_label in train_loader_fold:
                
                images, bin_label = images.to(device), bin_label.to(device)

                # Reiniciar gradientes
                optimizer.zero_grad()
        
                # Forward pass
                bin_out = model(images)
                
        
        
                # Calcular pérdida para la variable binaria
                lossT_bin = criterion_binary(bin_out.squeeze(1), bin_label)
        
                # Pérdida total
                loss_total = lossT_bin
    
                loss_total_acumulada += loss_total
    
                # Backward pass y optimización
                loss_total.backward()
                optimizer.step()

                #Calculo de otras métricas
                bin_pred = torch.sigmoid(bin_out.squeeze())
                correct_train += accuracy(bin_pred, bin_label)
                recall_value_train += recall(bin_pred, bin_label)
                precision_value_train += precision(bin_pred, bin_label)

        
    
            # Validación del modelo
            model.eval()
            val_loss = 0.0
            correct_valid = 0
            recall_value_valid = 0
            precision_value_valid = 0
            lossV_bin_total = 0.0
            with torch.no_grad():
                for images, bin_label in val_loader_fold:
                    images,  bin_label = images.to(device),  bin_label.to(device)
                    # Forward pass en el conjunto de validación
                    bin_out = model(images)
    
    
                    # Calcular pérdida para la variable binaria
                    lossV_bin = criterion_binary(bin_out.squeeze(1), bin_label)
    
                    # Sumar las pérdidas del conjunto de validación
                    val_loss += lossV_bin.item()

                    bin_pred = torch.sigmoid(bin_out.squeeze())
                    correct_valid += accuracy(bin_pred, bin_label)
                    recall_value_valid += recall(bin_pred, bin_label)
                    precision_value_valid += precision(bin_pred, bin_label)
                    
            
            ### Se promedian los resultados teniendo en cuenta el número de lotes y se actualizan listas
            avg_loss_total_acumulada = loss_total_acumulada / len(train_loader_fold)
            train_losses_fold.append(avg_loss_total_acumulada)
            acc_train = correct_train / len(train_loader_fold)
            recall_train = recall_value_train / len(train_loader_fold)
            precision_train = precision_value_train / len(train_loader_fold)
            train_accuracy_fold.append(acc_train)
            train_recall_fold.append(recall_train)
            train_preci_fold.append(precision_train)
        
            avg_val_loss = val_loss / len(val_loader_fold)
            val_losses_fold.append(avg_val_loss)
            acc_valid = correct_valid / len(val_loader_fold)
            recall_valid = recall_value_valid / len(val_loader_fold)
            precision_valid = precision_value_valid / len(val_loader_fold)
            val_accuracy_fold.append(acc_valid)
            val_recall_fold.append(recall_valid)
            val_preci_fold.append(precision_valid)
        
      
        
        train_accs_folds.append(np.mean(train_accuracy_fold))
        train_rec_folds.append(np.mean(train_recall_fold))
        train_preci_folds.append(np.mean(train_preci_fold))

        val_accs_folds.append(np.mean(val_accuracy_fold))
        val_rec_folds.append(np.mean(val_recall_fold))
        val_preci_folds.append(np.mean(val_preci_fold))

        train_losses_folds.append(train_losses_fold)
        val_losses_folds.append(val_losses_fold)
        train_losses_fold = [i.cpu().detach().numpy() for i in train_losses_fold]
    
    train_losses_folds_cpu = []
    val_losses_folds_cpu = []

    ##Movemos a cpu el contenido de las listas
    for losses in val_losses_folds:
        val_losses_fold_cpu = []
        for loss in losses:
            val_losses_fold_cpu.append(loss)
        val_losses_folds_cpu.append(np.array(val_losses_fold_cpu))
    for losses in train_losses_folds:
        train_losses_fold_cpu = []
        for loss in losses:
            train_losses_fold_cpu.append(float(loss.cpu().detach().numpy()))
        train_losses_folds_cpu.append(np.array(train_losses_fold_cpu))


    avg_train = []
    avg_val = []
    for losses in train_losses_folds_cpu:
        avg_train.append(np.mean(losses))
    for losses in val_losses_folds_cpu:
        avg_val.append(np.mean(losses))

    ##Se devuelve pérdida, accuracy, recall y precision, aunque esta última no se usará
    return np.mean(avg_val), np.mean(val_accs_folds), np.mean(val_rec_folds), np.mean(val_preci_folds)
    

## SIN TL CON OC

In [7]:
def accuracy(outputs, labels):
    preds = outputs > 0.5  # Si la salida es mayor que 0.5, consideramos que es positiva
    correct = (preds == labels.byte())  # Compara las predicciones con las etiquetas reales
    acc = correct.float().sum() / labels.size(0)  # Calcula la precisión
    return acc.item()

def recall(outputs, labels):
    preds = outputs > 0.5

    preds = preds.cpu()
    labels = labels.cpu()
    TP = sum((preds == 1) & (labels == 1))
    FN = sum((preds == 0) & (labels == 1))
    
    recall_score = TP / (TP + FN) if (TP + FN) > 0 else 0
    return recall_score

def precision(outputs, labels):
    preds = outputs > 0.5

    preds = preds.cpu()
    labels = labels.cpu()

    TP = sum((preds == 1) & (labels == 1))
    FP = sum((preds == 1) & (labels == 0))
    
    precision_score = TP / (TP + FP) if (TP + FP) > 0 else 0
    return precision_score

def prueba_k_fold_resnet_no_pre_oc(dataset, num_folds, num_epochs, op, batch_size, lr, data_aug, ini):
    # Transformaciones para preprocesar las imágenes
    if data_aug:
        transform_t = v2.Compose([
            v2.Resize((224, 224)),
            # v2.RandomResizedCrop(224),
            v2.RandomHorizontalFlip(),
            v2.RandomVerticalFlip(),
            # v2.ElasticTransform(alpha=250.0),
            # v2.RandomRotation(45),
            # transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.2),
            v2.ToImage(), 
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

        transform_v = v2.Compose([
            v2.Resize((224, 224)),
            v2.ToImage(), 
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])
    else:
        transform_t = v2.Compose([
            v2.Resize((224, 224)),
            v2.ToImage(), 
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

        transform_v = v2.Compose([
            v2.Resize((224, 224)),
            v2.ToImage(), 
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])
    
    # Modelo personalizado basado en ResNet
    class CustomResNet(nn.Module):
        def __init__(self, num_classes_binary=1):
            super(CustomResNet, self).__init__()
            resnet = models.resnet18(weights=None)# New weights with accuracy 80.858%
            # Extraer las capas de la red ResNet, excluyendo la capa de clasificación
            self.features = nn.Sequential(*list(resnet.children())[:-1])
    
            # Personalizar la capa de salida para tus necesidades
            self.fc_binary = nn.Linear(512, num_classes_binary)
            
            if (ini== "G"):
                # Inicialización de Glorot para las capas fully connected
                self.apply(self.init_weights_glorot)
            elif(ini == "H"):
                self.apply(self.init_weights_he)

        def init_weights_glorot(self, m):
            if isinstance(m, nn.Linear):
                init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    init.constant_(m.bias, 0)

        def init_weights_he(self, m):
            if isinstance(m, nn.Linear):
                init.kaiming_uniform_(m.weight, mode='fan_in', nonlinearity='relu')
                if m.bias is not None:
                    init.constant_(m.bias, 0)
    
        def forward(self, x):
            x = self.features(x)
            x = x.view(x.size(0), -1)
    
    
            # Salida para la variable binaria
            out_binary = self.fc_binary(x)
    
            return out_binary
    
    # Definir una clase para el conjunto de datos personalizado
    ##adaptar esto para meter unas rutsa y obtener dataos y etiquetas
    class CustomDataset(Dataset):
        def __init__(self, dataframe, transform=None):
            self.data = dataframe
            self.transform = transform
    
        def __len__(self):
            return len(self.data)
    
        def __getitem__(self, idx):
            img_path = self.data.iloc[idx, 0]  # Ruta de la imagen en la primera columna
            image = Image.open(img_path).convert('RGB')
            # Etiquetas: una variable binaria
            label_binary = torch.tensor(self.data.iloc[idx, 1], dtype=torch.float32)
    
            if self.transform:
                image = self.transform(image)
    
            return image, label_binary
        
    #####5 fold CV
    from sklearn.model_selection import StratifiedKFold
    
    # Definir el número de folds
    criterion_binary = nn.BCEWithLogitsLoss()
    kf = StratifiedKFold(n_splits=num_folds, shuffle=True, random_state=33)
    
    # Listas para almacenar las pérdidas de entrenamiento y validación de cada fold
    train_losses_folds = []
    val_losses_folds = []
    
    train_accs_folds = []
    train_rec_folds =  []
    train_preci_folds = []

    val_accs_folds = []
    val_rec_folds =  []
    val_preci_folds = []
    # Iterar sobre los folds
    for fold, (train_index, val_index) in enumerate(kf.split(np.zeros(len(dataset)),dataset['Etiqueta'])):
        print(f"Fold {fold + 1}")
    
        # Dividir el conjunto de datos en entrenamiento y validación para este fold
        train_data_fold = dataset.iloc[train_index]
        val_data_fold = dataset.iloc[val_index]
    
        # Crear instancias del conjunto de datos personalizado para este fold
        train_dataset_fold = CustomDataset(dataframe=train_data_fold, transform=transform_t)
        val_dataset_fold = CustomDataset(dataframe=val_data_fold, transform=transform_v)
    
        # Crear DataLoader para cargar los datos en lotes
        train_loader_fold = DataLoader(dataset=train_dataset_fold, batch_size=batch_size, shuffle=True)
        val_loader_fold = DataLoader(dataset=val_dataset_fold, batch_size=len(val_dataset_fold), shuffle=False) 
    
        # Re-inicializar el modelo para cada fold
        model = CustomResNet(num_classes_binary=1).to(device)

        #####!!!!!!!!!!!!!!!!!!!!!!!######

        ####Esta es una de las diferencias con el código de la función anterior, se implementa un scheduler para la política de ciclo único que se llamará después del optimizador
        optimizer = op(model.parameters(), lr = lr)
        scheduler = optim.lr_scheduler.OneCycleLR(optimizer, max_lr=lr, steps_per_epoch=len(train_loader_fold), epochs=num_epochs)
        # Entrenamiento y validación para este fold
        train_losses_fold = []
        val_losses_fold = []
        train_accuracy_fold = []
        val_accuracy_fold = []
        train_recall_fold = []
        val_recall_fold = []
        train_preci_fold = []
        val_preci_fold = []

        for epoch in range(num_epochs):
            # Entrenamiento del modelo
            # Fase de entrenamiento
            model.train()
            loss_total_acumulada = 0.0
            lossT_bin_total = 0.0
            correct_train = 0
            recall_value_train = 0
            precision_value_train = 0
            for images, bin_label in train_loader_fold:
                
                # Reiniciar gradientes
                images, bin_label = images.to(device), bin_label.to(device)
        
                optimizer.zero_grad()
        
                # Forward pass
                bin_out = model(images)
                        
        
                # Calcular pérdida para la variable binaria
                lossT_bin = criterion_binary(bin_out.squeeze(1), bin_label)
        
                # Pérdida total
                loss_total = lossT_bin
    
                loss_total_acumulada += loss_total
    
                # Backward pass y optimización
                loss_total.backward()
                optimizer.step()
                scheduler.step()

                bin_pred = torch.sigmoid(bin_out.squeeze())
                correct_train += accuracy(bin_pred, bin_label)
                recall_value_train += recall(bin_pred, bin_label)
                precision_value_train += precision(bin_pred, bin_label)

        
    
            # Validación del modelo
            model.eval()
            val_loss = 0.0
            correct_valid = 0
            recall_value_valid = 0
            precision_value_valid = 0
            lossV_bin_total = 0.0
            with torch.no_grad():
                for images, bin_label in val_loader_fold:
                
                    images,  bin_label = images.to(device),  bin_label.to(device)
                    # Forward pass en el conjunto de validación
                    bin_out = model(images)
    
    
    
                    # Calcular pérdida para la variable binaria
                    lossV_bin = criterion_binary(bin_out.squeeze(1), bin_label)
    
                    # Sumar las pérdidas del conjunto de validación
                    val_loss += lossV_bin.item()

                    bin_pred = torch.sigmoid(bin_out.squeeze())
                    correct_valid += accuracy(bin_pred, bin_label)
                    recall_value_valid += recall(bin_pred, bin_label)
                    precision_value_valid += precision(bin_pred, bin_label)
                    
            
            
            avg_loss_total_acumulada = loss_total_acumulada / len(train_loader_fold)
            train_losses_fold.append(avg_loss_total_acumulada)
            acc_train = correct_train / len(train_loader_fold)
            recall_train = recall_value_train / len(train_loader_fold)
            precision_train = precision_value_train / len(train_loader_fold)
            train_accuracy_fold.append(acc_train)
            train_recall_fold.append(recall_train)
            train_preci_fold.append(precision_train)
        
            avg_val_loss = val_loss / len(val_loader_fold)
            val_losses_fold.append(avg_val_loss)
            acc_valid = correct_valid / len(val_loader_fold)
            recall_valid = recall_value_valid / len(val_loader_fold)
            precision_valid = precision_value_valid / len(val_loader_fold)
            val_accuracy_fold.append(acc_valid)
            val_recall_fold.append(recall_valid)
            val_preci_fold.append(precision_valid)
            # print(f'Epoch [{epoch+1}/{num_epochs}], Training Loss: {avg_loss_total_acumulada.item()}, Validation Loss: {avg_val_loss}')
        
      
        
        train_accs_folds.append(np.mean(train_accuracy_fold))
        train_rec_folds.append(np.mean(train_recall_fold))
        train_preci_folds.append(np.mean(train_preci_fold))

        val_accs_folds.append(np.mean(val_accuracy_fold))
        val_rec_folds.append(np.mean(val_recall_fold))
        val_preci_folds.append(np.mean(val_preci_fold))

        train_losses_folds.append(train_losses_fold)
        val_losses_folds.append(val_losses_fold)
        train_losses_fold = [i.cpu().detach().numpy() for i in train_losses_fold]
    
    train_losses_folds_cpu = []
    val_losses_folds_cpu = []
    for losses in val_losses_folds:
        val_losses_fold_cpu = []
        for loss in losses:
            val_losses_fold_cpu.append(loss)
        val_losses_folds_cpu.append(np.array(val_losses_fold_cpu))
    for losses in train_losses_folds:
        train_losses_fold_cpu = []
        for loss in losses:
            train_losses_fold_cpu.append(float(loss.cpu().detach().numpy()))
        train_losses_folds_cpu.append(np.array(train_losses_fold_cpu))
    print(len(train_losses_folds_cpu))
    avg_train = []
    avg_val = []
    for losses in train_losses_folds_cpu:
        avg_train.append(np.mean(losses))
    for losses in val_losses_folds_cpu:
        avg_val.append(np.mean(losses))
    
    return np.mean(avg_val), np.mean(val_accs_folds), np.mean(val_rec_folds), np.mean(val_preci_folds)
    

## CON TL SIN OC

In [8]:

def accuracy(outputs, labels):
    preds = outputs > 0.5  # Si la salida es mayor que 0.5, consideramos que es positiva
    correct = (preds == labels.byte())  # Compara las predicciones con las etiquetas reales
    acc = correct.float().sum() / labels.size(0)  # Calcula la precisión
    return acc.item()

def recall(outputs, labels):
    preds = outputs > 0.5

    preds = preds.cpu()
    labels = labels.cpu()
    TP = sum((preds == 1) & (labels == 1))
    FN = sum((preds == 0) & (labels == 1))
    
    recall_score = TP / (TP + FN) if (TP + FN) > 0 else 0
    return recall_score

def precision(outputs, labels):
    preds = outputs > 0.5

    preds = preds.cpu()
    labels = labels.cpu()

    TP = sum((preds == 1) & (labels == 1))
    FP = sum((preds == 1) & (labels == 0))
    
    precision_score = TP / (TP + FP) if (TP + FP) > 0 else 0
    return precision_score


###Las épocas totales se separan en dos cuando se entrena con TL
def prueba_k_fold_resnet_pre(dataset, num_folds, num_epochs_freeze, num_epochs_all, op, batch_size, lr, data_aug, ini):
    # Transformaciones para preprocesar las imágenes
    if data_aug:
        transform_t = v2.Compose([
            v2.Resize((224, 224)),
            # v2.RandomResizedCrop(224),
            v2.RandomHorizontalFlip(),
            v2.RandomVerticalFlip(),
            # v2.ElasticTransform(alpha=250.0),
            # v2.RandomRotation(45),
            # transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.2),
            v2.ToImage(), 
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

        transform_v = v2.Compose([
            v2.Resize((224, 224)),
            v2.ToImage(), 
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])
    else:
        transform_t = v2.Compose([
            v2.Resize((224, 224)),
            v2.ToImage(), 
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

        transform_v = v2.Compose([
            v2.Resize((224, 224)),
            v2.ToImage(), 
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])
    
    # Modelo personalizado basado en ResNet
    class CustomResNet(nn.Module):
        def __init__(self, num_classes_binary=1):
            super(CustomResNet, self).__init__()
            
            ####!!!!!!!!####
            ### Al ser coon preentrenamiento, se cargan los pesos del entrenamiento en ImageNet.
            resnet = models.resnet18(weights='IMAGENET1K_V1')
            # Extraer las capas de la red ResNet, excluyendo la capa de clasificación
            self.features = nn.Sequential(*list(resnet.children())[:-1])

            # Congelar capas preentrenadas
            for param in self.features.parameters():
                param.requires_grad = False

            # Personalizar la capa de salida para tus necesidades
            self.fc_binary = nn.Linear(512, num_classes_binary)
            
            if (ini== "G"):
                # Inicialización de Glorot para las capas fully connected
                self.apply(self.init_weights_glorot)
            elif(ini == "H"):
                self.apply(self.init_weights_he)
            elif (ini== "N"):
                self.apply(self.init_weights_normal)
            elif(ini == "U"):
                self.apply(self.init_weights_uniform)
            elif (ini== "O"):
                self.apply(self.init_weights_orthogonal)
            elif(ini == "Z"):
                self.apply(self.init_weights_zeros)
            elif(ini == "ON"):
                self.apply(self.init_weights_ones)

        def init_weights_glorot(self, m):
            if isinstance(m, nn.Linear):
                init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    init.constant_(m.bias, 0)

        def init_weights_he(self, m):
            if isinstance(m, nn.Linear):
                init.kaiming_uniform_(m.weight, mode='fan_in', nonlinearity='relu')
                if m.bias is not None:
                    init.constant_(m.bias, 0)
        def init_weights_normal(self, m):
            if isinstance(m, nn.Linear):
                init.normal_(m.weight, mean=0, std=0.01) 
                if m.bias is not None:
                    init.constant_(m.bias, 0)

        def init_weights_uniform(self, m):
            if isinstance(m, nn.Linear):
                init.uniform_(m.weight, a=-0.1, b=0.1)  
                if m.bias is not None:
                    init.constant_(m.bias, 0)
    
        def init_weights_orthogonal(self, m):
            if isinstance(m, nn.Linear):
                init.orthogonal_(m.weight) 
                if m.bias is not None:
                    init.constant_(m.bias, 0)
    
        def init_weights_zeros(self, m):
            if isinstance(m, nn.Linear):
                init.zeros_(m.weight)
                if m.bias is not None:
                    init.zeros_(m.bias)
    
        def init_weights_ones(self, m):
            if isinstance(m, nn.Linear):
                init.ones_(m.weight)
                if m.bias is not None:
                    init.ones_(m.bias)
    
        def forward(self, x):
            x = self.features(x)
            x = x.view(x.size(0), -1)
    
    
            # Salida para la variable binaria
            out_binary = self.fc_binary(x)
    
            return out_binary
    
    # Definir una clase para el conjunto de datos personalizado
    ##adaptar esto para meter unas rutsa y obtener dataos y etiquetas
    class CustomDataset(Dataset):
        def __init__(self, dataframe, transform=None):
            self.data = dataframe
            self.transform = transform
    
        def __len__(self):
            return len(self.data)
    
        def __getitem__(self, idx):
            img_path = self.data.iloc[idx, 0]  # Ruta de la imagen en la primera columna
            image = Image.open(img_path).convert('RGB')
            # Etiquetas: una variable binaria
            label_binary = torch.tensor(self.data.iloc[idx, 1], dtype=torch.float32)
    
            if self.transform:
                image = self.transform(image)
    
            return image, label_binary
        
    #####5 fold CV
    from sklearn.model_selection import StratifiedKFold
    
    # Definir el número de folds
    criterion_binary = nn.BCEWithLogitsLoss()
    kf = StratifiedKFold(n_splits=num_folds, shuffle=True, random_state=33)
    
    # Listas para almacenar las pérdidas de entrenamiento y validación de cada fold
    train_losses_folds = []
    val_losses_folds = []
    
    train_accs_folds = []
    train_rec_folds =  []
    train_preci_folds = []

    val_accs_folds = []
    val_rec_folds =  []
    val_preci_folds = []
    # Iterar sobre los folds
    for fold, (train_index, val_index) in enumerate(kf.split(np.zeros(len(dataset)),dataset['Etiqueta'])):
        print(f"Fold {fold + 1}")
    
        # Dividir el conjunto de datos en entrenamiento y validación para este fold
        train_data_fold = dataset.iloc[train_index]
        val_data_fold = dataset.iloc[val_index]
    
        # Crear instancias del conjunto de datos personalizado para este fold
        train_dataset_fold = CustomDataset(dataframe=train_data_fold, transform=transform_t)
        val_dataset_fold = CustomDataset(dataframe=val_data_fold, transform=transform_v)
    
        # Crear DataLoader para cargar los datos en lotes
        train_loader_fold = DataLoader(dataset=train_dataset_fold, batch_size=batch_size, shuffle=True)
        val_loader_fold = DataLoader(dataset=val_dataset_fold, batch_size=len(val_dataset_fold), shuffle=False) 
    
        # Re-inicializar el modelo para cada fold
        model = CustomResNet(num_classes_binary=1).to(device)

        ####!!!!!!!!!!!!!!!!#####
        ###Esto es otra diferencia con la primera función, se tienen dos optimizadores puesto que el entrenamiento se realizará en dos partes,
        ###una con los pesos bloqueados (no se actualizan, excepto los de la capa FC) y otra sin ellos (se actualizan todos). Es por ello que, a uno de ellos 
        ###se le pasan como parámetros todos los parámetros del modelo y al otro solo los de la FC.
        optimizer = op(model.parameters(), lr = lr)
        optimizer2 = op(model.fc_binary.parameters(), lr = lr)
        # Entrenamiento y validación para este fold
        train_losses_fold = []
        val_losses_fold = []
        train_accuracy_fold = []
        val_accuracy_fold = []
        train_recall_fold = []
        val_recall_fold = []
        train_preci_fold = []
        val_preci_fold = []

        for epoch in range(num_epochs_freeze):
            # Entrenamiento del modelo
            # Fase de entrenamiento
            model.train()
            loss_total_acumulada = 0.0
            lossT_bin_total = 0.0
            correct_train = 0
            recall_value_train = 0
            precision_value_train = 0
            for images, bin_label in train_loader_fold:
                
                # Reiniciar gradientes
            
                images, bin_label = images.to(device), bin_label.to(device)

                

                optimizer2.zero_grad()
        
                # Forward pass
                bin_out = model(images)                
        
        
                # Calcular pérdida para la variable binaria
                lossT_bin = criterion_binary(bin_out.squeeze(1), bin_label)
        
                # Pérdida total
                loss_total = lossT_bin
    
                loss_total_acumulada += loss_total
    
                # Backward pass y optimización
                loss_total.backward()
                optimizer2.step()

                bin_pred = torch.sigmoid(bin_out.squeeze())
                correct_train += accuracy(bin_pred, bin_label)
                recall_value_train += recall(bin_pred, bin_label)
                precision_value_train += precision(bin_pred, bin_label)

        
    
            # Validación del modelo
            model.eval()
            val_loss = 0.0
            correct_valid = 0
            recall_value_valid = 0
            precision_value_valid = 0
            lossV_bin_total = 0.0
            with torch.no_grad():
                for images, bin_label in val_loader_fold:
                
                    images,  bin_label = images.to(device),  bin_label.to(device)
                    # Forward pass en el conjunto de validación
                    bin_out = model(images)
    
    
                    # Calcular pérdida para la variable binaria
                    lossV_bin = criterion_binary(bin_out.squeeze(1), bin_label)
    
                    # Sumar las pérdidas del conjunto de validación
                    val_loss += lossV_bin.item()

                    bin_pred = torch.sigmoid(bin_out.squeeze())
                    correct_valid += accuracy(bin_pred, bin_label)
                    recall_value_valid += recall(bin_pred, bin_label)
                    precision_value_valid += precision(bin_pred, bin_label)
                    
            
            ###El proceso de las pérdidas y métricas se hará ahora y después otra vez, tras entrenar con las capas descongeladas
            avg_loss_total_acumulada = loss_total_acumulada / len(train_loader_fold)
            train_losses_fold.append(avg_loss_total_acumulada)
            acc_train = correct_train / len(train_loader_fold)
            recall_train = recall_value_train / len(train_loader_fold)
            precision_train = precision_value_train / len(train_loader_fold)
            train_accuracy_fold.append(acc_train)
            train_recall_fold.append(recall_train)
            train_preci_fold.append(precision_train)

            avg_val_loss = val_loss / len(val_loader_fold)
            val_losses_fold.append(avg_val_loss)
            acc_valid = correct_valid / len(val_loader_fold)
            recall_valid = recall_value_valid / len(val_loader_fold)
            precision_valid = precision_value_valid / len(val_loader_fold)
            val_accuracy_fold.append(acc_valid)
            val_recall_fold.append(recall_valid)
            val_preci_fold.append(precision_valid)

        ##Se descongelan todas
        for param in model.features.parameters():
            param.requires_grad = True
            
        for epoch in range(num_epochs_freeze, num_epochs_freeze + num_epochs_all):
            # Entrenamiento del modelo
            # Fase de entrenamiento
            model.train()
            loss_total_acumulada = 0.0
            lossT_bin_total = 0.0
            correct_train = 0
            recall_value_train = 0
            precision_value_train = 0
            for images, bin_label in train_loader_fold:
                # Reiniciar gradientes

                images, bin_label = images.to(device), bin_label.to(device)

                optimizer.zero_grad()
        
                # Forward pass
                bin_out = model(images)                
        
        
                # Calcular pérdida para la variable binaria
                lossT_bin = criterion_binary(bin_out.squeeze(1), bin_label)
        
                # Pérdida total
                loss_total = lossT_bin
    
                loss_total_acumulada += loss_total
    
                # Backward pass y optimización
                loss_total.backward()
                optimizer.step()

                bin_pred = torch.sigmoid(bin_out.squeeze())
                correct_train += accuracy(bin_pred, bin_label)
                recall_value_train += recall(bin_pred, bin_label)
                precision_value_train += precision(bin_pred, bin_label)

        
    
            # Validación del modelo
            model.eval()
            val_loss = 0.0
            correct_valid = 0
            recall_value_valid = 0
            precision_value_valid = 0
            lossV_bin_total = 0.0
            with torch.no_grad():
                for images, bin_label in val_loader_fold:
                    
                    images,  bin_label = images.to(device),  bin_label.to(device)
                    # Forward pass en el conjunto de validación
                    bin_out = model(images)
        
                    # Calcular pérdida para la variable binaria
                    lossV_bin = criterion_binary(bin_out.squeeze(1), bin_label)
    
                    # Sumar las pérdidas del conjunto de validación
                    val_loss += lossV_bin.item()

                    bin_pred = torch.sigmoid(bin_out.squeeze())
                    correct_valid += accuracy(bin_pred, bin_label)
                    recall_value_valid += recall(bin_pred, bin_label)
                    precision_value_valid += precision(bin_pred, bin_label)

                    
            
            
            avg_loss_total_acumulada = loss_total_acumulada / len(train_loader_fold)
            train_losses_fold.append(avg_loss_total_acumulada)
            acc_train = correct_train / len(train_loader_fold)
            recall_train = recall_value_train / len(train_loader_fold)
            precision_train = precision_value_train / len(train_loader_fold)
            train_accuracy_fold.append(acc_train)
            train_recall_fold.append(recall_train)
            train_preci_fold.append(precision_train)
        
            avg_val_loss = val_loss / len(val_loader_fold)
            val_losses_fold.append(avg_val_loss)
            acc_valid = correct_valid / len(val_loader_fold)
            recall_valid = recall_value_valid / len(val_loader_fold)
            precision_valid = precision_value_valid / len(val_loader_fold)
            val_accuracy_fold.append(acc_valid)
            val_recall_fold.append(recall_valid)
            val_preci_fold.append(precision_valid)
            # print(f'Epoch [{epoch+1}/{num_epochs}], Training Loss: {avg_loss_total_acumulada.item()}, Validation Loss: {avg_val_loss}')

        train_accs_folds.append(np.mean(train_accuracy_fold))
        train_rec_folds.append(np.mean(train_recall_fold))
        train_preci_folds.append(np.mean(train_preci_fold))

        val_accs_folds.append(np.mean(val_accuracy_fold))
        val_rec_folds.append(np.mean(val_recall_fold))
        val_preci_folds.append(np.mean(val_preci_fold))

        train_losses_folds.append(train_losses_fold)
        val_losses_folds.append(val_losses_fold)
        train_losses_fold = [i.cpu().detach().numpy() for i in train_losses_fold]

        
    
    train_losses_folds_cpu = []
    val_losses_folds_cpu = []
    for losses in val_losses_folds:
        val_losses_fold_cpu = []
        for loss in losses:
            val_losses_fold_cpu.append(loss)
        val_losses_folds_cpu.append(np.array(val_losses_fold_cpu))
    for losses in train_losses_folds:
        train_losses_fold_cpu = []
        for loss in losses:
            train_losses_fold_cpu.append(float(loss.cpu().detach().numpy()))
        train_losses_folds_cpu.append(np.array(train_losses_fold_cpu))
    print(len(train_losses_folds_cpu))
    avg_train = []
    avg_val = []
    for losses in train_losses_folds_cpu:
        avg_train.append(np.mean(losses))
    for losses in val_losses_folds_cpu:
        avg_val.append(np.mean(losses))

    
    return np.mean(avg_val), np.mean(val_accs_folds), np.mean(val_rec_folds), np.mean(val_preci_folds)

    

## CON OC Y CON TL

In [9]:

def accuracy(outputs, labels):
    preds = outputs > 0.5  # Si la salida es mayor que 0.5, consideramos que es positiva
    correct = (preds == labels.byte())  # Compara las predicciones con las etiquetas reales
    acc = correct.float().sum() / labels.size(0)  # Calcula la precisión
    return acc.item()

def recall(outputs, labels):
    preds = outputs > 0.5

    preds = preds.cpu()
    labels = labels.cpu()
    TP = sum((preds == 1) & (labels == 1))
    FN = sum((preds == 0) & (labels == 1))
    
    recall_score = TP / (TP + FN) if (TP + FN) > 0 else 0
    return recall_score

def precision(outputs, labels):
    preds = outputs > 0.5

    preds = preds.cpu()
    labels = labels.cpu()

    TP = sum((preds == 1) & (labels == 1))
    FP = sum((preds == 1) & (labels == 0))
    
    precision_score = TP / (TP + FP) if (TP + FP) > 0 else 0
    return precision_score



def prueba_k_fold_resnet_pre_oc(dataset, num_folds, num_epochs_freeze, num_epochs_all, op, batch_size, lr, data_aug, ini):
    # Transformaciones para preprocesar las imágenes
    if data_aug:
        transform_t = v2.Compose([
            v2.Resize((224, 224)),
            v2.RandomHorizontalFlip(),
            v2.RandomVerticalFlip(),
            v2.ToImage(), 
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

        transform_v = v2.Compose([
            v2.Resize((224, 224)),
            v2.ToImage(), 
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])
    else:
        transform_t = v2.Compose([
            v2.Resize((224, 224)),
            v2.ToImage(), 
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

        transform_v = v2.Compose([
            v2.Resize((224, 224)),
            v2.ToImage(), 
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])
    
    # Modelo personalizado basado en ResNet
    class CustomResNet(nn.Module):
        def __init__(self, num_classes_binary=1):
            super(CustomResNet, self).__init__()
            resnet = models.resnet18(weights='IMAGENET1K_V1')
            # Extraer las capas de la red ResNet, excluyendo la capa de clasificación
            self.features = nn.Sequential(*list(resnet.children())[:-1])

            # Congelar capas preentrenadas
            for param in self.features.parameters():
                param.requires_grad = False

            # Personalizar la capa de salida para tus necesidades
            self.fc_binary = nn.Linear(512, num_classes_binary)
            
            if (ini== "G"):
                # Inicialización de Glorot para las capas fully connected
                self.apply(self.init_weights_glorot)
            elif(ini == "H"):
                self.apply(self.init_weights_he)
            elif (ini== "N"):
                self.apply(self.init_weights_normal)
            elif(ini == "U"):
                self.apply(self.init_weights_uniform)
            elif (ini== "O"):
                self.apply(self.init_weights_orthogonal)
            elif(ini == "Z"):
                self.apply(self.init_weights_zeros)
            elif(ini == "ON"):
                self.apply(self.init_weights_ones)
       
        def init_weights_glorot(self, m):
            if isinstance(m, nn.Linear):
                init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    init.constant_(m.bias, 0)

        def init_weights_he(self, m):
            if isinstance(m, nn.Linear):
                init.kaiming_uniform_(m.weight, mode='fan_in', nonlinearity='relu')
                if m.bias is not None:
                    init.constant_(m.bias, 0)
        def init_weights_normal(self, m):
            if isinstance(m, nn.Linear):
                init.normal_(m.weight, mean=0, std=0.01)  
                if m.bias is not None:
                    init.constant_(m.bias, 0)

        def init_weights_uniform(self, m):
            if isinstance(m, nn.Linear):
                init.uniform_(m.weight, a=-0.1, b=0.1)  
                if m.bias is not None:
                    init.constant_(m.bias, 0)
    
        def init_weights_orthogonal(self, m):
            if isinstance(m, nn.Linear):
                init.orthogonal_(m.weight)  
                if m.bias is not None:
                    init.constant_(m.bias, 0)
    
        def init_weights_zeros(self, m):
            if isinstance(m, nn.Linear):
                init.zeros_(m.weight)
                if m.bias is not None:
                    init.zeros_(m.bias)
    
        def init_weights_ones(self, m):
            if isinstance(m, nn.Linear):
                init.ones_(m.weight)
                if m.bias is not None:
                    init.ones_(m.bias)
    
        def forward(self, x):
            x = self.features(x)
            x = x.view(x.size(0), -1)
    
    
            # Salida para la variable binaria
            out_binary = self.fc_binary(x)
    
            return out_binary
    
    # Definir una clase para el conjunto de datos personalizado
    ##adaptar esto para meter unas rutsa y obtener dataos y etiquetas
    class CustomDataset(Dataset):
        def __init__(self, dataframe, transform=None):
            self.data = dataframe
            self.transform = transform
    
        def __len__(self):
            return len(self.data)
    
        def __getitem__(self, idx):
            img_path = self.data.iloc[idx, 0]  # Ruta de la imagen en la primera columna
            image = Image.open(img_path).convert('RGB')
            # Etiquetas: una variable binaria
            label_binary = torch.tensor(self.data.iloc[idx, 1], dtype=torch.float32)
    
            if self.transform:
                image = self.transform(image)
    
            return image, label_binary
        
    #####5 fold CV
    from sklearn.model_selection import StratifiedKFold
    
    # Definir el número de folds
    criterion_binary = nn.BCEWithLogitsLoss()
    kf = StratifiedKFold(n_splits=num_folds, shuffle=True, random_state=33)
    
    # Listas para almacenar las pérdidas de entrenamiento y validación de cada fold
    train_losses_folds = []
    val_losses_folds = []
    
    train_accs_folds = []
    train_rec_folds =  []
    train_preci_folds = []

    val_accs_folds = []
    val_rec_folds =  []
    val_preci_folds = []
    # Iterar sobre los folds
    for fold, (train_index, val_index) in enumerate(kf.split(np.zeros(len(dataset)),dataset['Etiqueta'])):
        print(f"Fold {fold + 1}")
    
        # Dividir el conjunto de datos en entrenamiento y validación para este fold
        train_data_fold = dataset.iloc[train_index]
        val_data_fold = dataset.iloc[val_index]
    
        # Crear instancias del conjunto de datos personalizado para este fold
        train_dataset_fold = CustomDataset(dataframe=train_data_fold, transform=transform_t)
        val_dataset_fold = CustomDataset(dataframe=val_data_fold, transform=transform_v)
    
        # Crear DataLoader para cargar los datos en lotes
        train_loader_fold = DataLoader(dataset=train_dataset_fold, batch_size=batch_size, shuffle=True)
        val_loader_fold = DataLoader(dataset=val_dataset_fold, batch_size=len(val_dataset_fold), shuffle=False) 
    
        # Re-inicializar el modelo para cada fold
        model = CustomResNet(num_classes_binary=1).to(device)

        ##Se crea un optimizador para dos grupos de parámetro, primero va la fully connected porque se le asignará el lr más pequeño
        optimizer = op([{'params': model.fc_binary.parameters()}, {'params': model.features.parameters()}], lr = lr)
        ##cuando estan congeladas las capas, solo quiero optimizar la fully connected
        optimizer2 = op( model.fc_binary.parameters(), lr = lr)

        # layer_names = []
        # for idx, (name, param) in enumerate(model.named_parameters()):
        #     layer_names.append(name)
        #     print(f'{idx}: {name}')
        # layer_names.reverse()
        # print('Segundo:', optimizer.param_groups[1])
        total = len(optimizer.param_groups)
        ##!!!!!!!!!!!!!!!!!!!##
        ##Partiendo de los optimizadores de antes, se crearán dos schedulers, siendo el primero con el lr que se pase como parámetro
        scheduler1 = optim.lr_scheduler.OneCycleLR(optimizer2, max_lr=lr, steps_per_epoch=len(train_loader_fold), epochs=num_epochs_freeze)
       
        lr /= 2
        
        max_lrs = np.linspace(lr/100, lr, total)
       
        max_lrs = max_lrs.tolist()
        scheduler2 = optim.lr_scheduler.OneCycleLR(optimizer, max_lr=max_lrs, steps_per_epoch=len(train_loader_fold), epochs=num_epochs_all)
        # Entrenamiento y validación para este fold
        train_losses_fold = []
        val_losses_fold = []
        train_accuracy_fold = []
        val_accuracy_fold = []
        train_recall_fold = []
        val_recall_fold = []
        train_preci_fold = []
        val_preci_fold = []

        for epoch in range(num_epochs_freeze):
            # Entrenamiento del modelo
            # Fase de entrenamiento
            model.train()
            loss_total_acumulada = 0.0
            lossT_bin_total = 0.0
            correct_train = 0
            recall_value_train = 0
            precision_value_train = 0
            for images, bin_label in train_loader_fold:
                
                images, bin_label = images.to(device), bin_label.to(device)

                

                optimizer2.zero_grad()
        
                # Forward pass
                bin_out = model(images)
        
                # Calcular pérdida para las variables continuas
                
        
        
                # Calcular pérdida para la variable binaria
                lossT_bin = criterion_binary(bin_out.squeeze(1), bin_label)
        
                # Pérdida total
                loss_total = lossT_bin
    
                loss_total_acumulada += loss_total
    
                # Backward pass y optimización
                loss_total.backward()
                optimizer2.step()
                scheduler1.step()
                

                bin_pred = torch.sigmoid(bin_out.squeeze())
                correct_train += accuracy(bin_pred, bin_label)
                recall_value_train += recall(bin_pred, bin_label)
                precision_value_train += precision(bin_pred, bin_label)

        
    
            # Validación del modelo
            model.eval()
            val_loss = 0.0
            correct_valid = 0
            recall_value_valid = 0
            precision_value_valid = 0
            lossV_bin_total = 0.0
            with torch.no_grad():
                for images, bin_label in val_loader_fold:
                
                    images,  bin_label = images.to(device),  bin_label.to(device)
                    # Forward pass en el conjunto de validación
                    bin_out = model(images)
    
                    # Calcular pérdida para la variable binaria
                    lossV_bin = criterion_binary(bin_out.squeeze(1), bin_label)
    
                    # Sumar las pérdidas del conjunto de validación
                    val_loss += lossV_bin.item()

                    bin_pred = torch.sigmoid(bin_out.squeeze())
                    correct_valid += accuracy(bin_pred, bin_label)
                    recall_value_valid += recall(bin_pred, bin_label)
                    precision_value_valid += precision(bin_pred, bin_label)
                    
            
            
            avg_loss_total_acumulada = loss_total_acumulada / len(train_loader_fold)
            train_losses_fold.append(avg_loss_total_acumulada)
            acc_train = correct_train / len(train_loader_fold)
            recall_train = recall_value_train / len(train_loader_fold)
            precision_train = precision_value_train / len(train_loader_fold)
            train_accuracy_fold.append(acc_train)
            train_recall_fold.append(recall_train)
            train_preci_fold.append(precision_train)

            avg_val_loss = val_loss / len(val_loader_fold)
            val_losses_fold.append(avg_val_loss)
            acc_valid = correct_valid / len(val_loader_fold)
            recall_valid = recall_value_valid / len(val_loader_fold)
            precision_valid = precision_value_valid / len(val_loader_fold)
            val_accuracy_fold.append(acc_valid)
            val_recall_fold.append(recall_valid)
            val_preci_fold.append(precision_valid)

        ##Se descongelan todas
        for param in model.features.parameters():
            param.requires_grad = True
            
        for epoch in range(num_epochs_freeze, num_epochs_freeze + num_epochs_all):
            # Entrenamiento del modelo
            # Fase de entrenamiento
            model.train()
            loss_total_acumulada = 0.0
            lossT_bin_total = 0.0
            correct_train = 0
            recall_value_train = 0
            precision_value_train = 0
            for images, bin_label in train_loader_fold:
                

                images, bin_label = images.to(device), bin_label.to(device)

                optimizer.zero_grad()
        
                # Forward pass
                bin_out = model(images)
                        
        
        
                # Calcular pérdida para la variable binaria
                lossT_bin = criterion_binary(bin_out.squeeze(1), bin_label)
        
                # Pérdida total
                loss_total = lossT_bin
    
                loss_total_acumulada += loss_total
    
                # Backward pass y optimización
                loss_total.backward()
                optimizer.step()
                scheduler2.step()

                bin_pred = torch.sigmoid(bin_out.squeeze())
                correct_train += accuracy(bin_pred, bin_label)
                recall_value_train += recall(bin_pred, bin_label)
                precision_value_train += precision(bin_pred, bin_label)

        
    
            # Validación del modelo
            model.eval()
            val_loss = 0.0
            correct_valid = 0
            recall_value_valid = 0
            precision_value_valid = 0
            lossV_bin_total = 0.0
            with torch.no_grad():
                for images, bin_label in val_loader_fold:
                    
                    images,  bin_label = images.to(device),  bin_label.to(device)
                    ##print(len(images)) aqui se hace con 32 imagenes
                    # Forward pass en el conjunto de validación
                    bin_out = model(images)
        
                    # Calcular pérdida para la variable binaria
                    lossV_bin = criterion_binary(bin_out.squeeze(1), bin_label)
    
                    # Sumar las pérdidas del conjunto de validación
                    val_loss += lossV_bin.item()

                    bin_pred = torch.sigmoid(bin_out.squeeze())
                    correct_valid += accuracy(bin_pred, bin_label)
                    recall_value_valid += recall(bin_pred, bin_label)
                    precision_value_valid += precision(bin_pred, bin_label)

                    
            
            
            avg_loss_total_acumulada = loss_total_acumulada / len(train_loader_fold)
            train_losses_fold.append(avg_loss_total_acumulada)
            acc_train = correct_train / len(train_loader_fold)
            recall_train = recall_value_train / len(train_loader_fold)
            precision_train = precision_value_train / len(train_loader_fold)
            train_accuracy_fold.append(acc_train)
            train_recall_fold.append(recall_train)
            train_preci_fold.append(precision_train)
        
            avg_val_loss = val_loss / len(val_loader_fold)
            val_losses_fold.append(avg_val_loss)
            acc_valid = correct_valid / len(val_loader_fold)
            recall_valid = recall_value_valid / len(val_loader_fold)
            precision_valid = precision_value_valid / len(val_loader_fold)
            val_accuracy_fold.append(acc_valid)
            val_recall_fold.append(recall_valid)
            val_preci_fold.append(precision_valid)
            # print(f'Epoch [{epoch+1}/{num_epochs}], Training Loss: {avg_loss_total_acumulada.item()}, Validation Loss: {avg_val_loss}')
        

        train_accs_folds.append(np.mean(train_accuracy_fold))
        train_rec_folds.append(np.mean(train_recall_fold))
        train_preci_folds.append(np.mean(train_preci_fold))

        val_accs_folds.append(np.mean(val_accuracy_fold))
        val_rec_folds.append(np.mean(val_recall_fold))
        val_preci_folds.append(np.mean(val_preci_fold))

        train_losses_folds.append(train_losses_fold)
        val_losses_folds.append(val_losses_fold)
        train_losses_fold = [i.cpu().detach().numpy() for i in train_losses_fold]

        
    
    train_losses_folds_cpu = []
    val_losses_folds_cpu = []
    for losses in val_losses_folds:
        val_losses_fold_cpu = []
        for loss in losses:
            val_losses_fold_cpu.append(loss)
        val_losses_folds_cpu.append(np.array(val_losses_fold_cpu))
    for losses in train_losses_folds:
        train_losses_fold_cpu = []
        for loss in losses:
            train_losses_fold_cpu.append(float(loss.cpu().detach().numpy()))
        train_losses_folds_cpu.append(np.array(train_losses_fold_cpu))
    print(len(train_losses_folds_cpu))
    avg_train = []
    avg_val = []
    for losses in train_losses_folds_cpu:
        avg_train.append(np.mean(losses))
    for losses in val_losses_folds_cpu:
        avg_val.append(np.mean(losses))
    
    return np.mean(avg_val), np.mean(val_accs_folds), np.mean(val_rec_folds), np.mean(val_preci_folds)
    

In [10]:
df_ct = df_defi[df_defi['Ruta'].str.contains('1CT')]
df_fdg = df_defi[df_defi['Ruta'].str.contains('FDG')]
df_a90 = df_defi[df_defi['Ruta'].str.contains('A90')]

print(df_ct)

                                 Ruta  Etiqueta
0   ./NIFTI_TRIPLE\mwc1CT_P01_HVN.jpg       1.0
1   ./NIFTI_TRIPLE\mwc1CT_P02_HVN.jpg       1.0
2   ./NIFTI_TRIPLE\mwc1CT_P03_HVN.jpg       0.0
3   ./NIFTI_TRIPLE\mwc1CT_P05_HVN.jpg       1.0
4   ./NIFTI_TRIPLE\mwc1CT_P06_HVN.jpg       1.0
..                                ...       ...
90  ./NIFTI_TRIPLE\mwc1CT_P72_HVN.jpg       1.0
91  ./NIFTI_TRIPLE\mwc1CT_P73_H9O.jpg       0.0
92  ./NIFTI_TRIPLE\mwc1CT_P73_HVN.jpg       1.0
93  ./NIFTI_TRIPLE\mwc1CT_P74_H9O.jpg       1.0
94  ./NIFTI_TRIPLE\mwc1CT_P75_H9O.jpg       0.0

[95 rows x 2 columns]


## A9O FBB

Ahora se hacen las pruebas para cada optimizador que se quiera probar y cada forma de entrenar de entre las funciones definidas arriba. En este caso, se usan las imágenes A9O. Aquí, se hicieron todas las pruebas pero se han dejado las últimas, ya que se sobreescribieron al interesarnos realmente lo que quedaba en disco, para las siguientes modalidades se han dejado todas las pruebas.

In [11]:

 
def save_study(study, study_name="study"):
    # Crear una carpeta para guardar el estudio si no existe
    study_dir = os.path.join(os.getcwd(), study_name)
    if not os.path.exists(study_dir):
        os.makedirs(study_dir)
    
    # Guardar los resultados del estudio en un archivo CSV
    study_trials = study.trials_dataframe()
    study_trials.to_csv(os.path.join(study_dir, "study_results.csv"), index=False)

def objective(trial):
    # Define los hiperparámetros para optimizar
    # num_epochs_freeze = trial.suggest_categorical('num_epochs_freeze', [1,3,8,10,15])
    # num_epochs_all = trial.suggest_categorical('num_epochs_all', [25,30,35,40,45,50])
    batch_size = trial.suggest_categorical('batch_size', [32, 16, 8, 4, 48, 5])
    lr = trial.suggest_categorical('lr', [1e-5,1e-4,1e-3,1e-2,1e-1])
    data_aug = trial.suggest_categorical('data_aug', [True,False])
    # ini = trial.suggest_categorical('ini', ['G', 'H', 'N', 'U', 'O', 'Z', 'ON', 'None'])
    # optimizer_name = trial.suggest_categorical('optimizer_name', ['RMSprop', 'Adam', 'SGD'])

    # # Convertir el nombre del optimizador en una clase PyTorch
    # if optimizer_name == 'Adam':
    #     optimizer_class = torch.optim.Adam
    # elif optimizer_name == 'SGD':
    #     optimizer_class = torch.optim.SGD
    # elif optimizer_name == 'RMSprop':
    #     optimizer_class = torch.optim.RMSprop

    # Ejecuta la función de entrenamiento con los hiperparámetros seleccionados
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_pre_oc(df_mean, 5, 5, 35, optim.RMSprop, batch_size, lr, data_aug, None)
    avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_pre(df_a90, 5, 5, 35, optim.SGD, batch_size, lr, data_aug, None)
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre(df_a90, 5, 40, optim.RMSprop, batch_size, lr, data_aug, None)
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre_oc(df_a90, 5, 40, optim.RMSprop, batch_size, lr, data_aug, None)

    # Devuelve el valor objetivo (en este caso, la pérdida promedio de validación)
    return avg_val_loss, avg_val_accuracy, avg_val_recall

import optuna
# Supongamos que tienes un dataset llamado 'dataset'
# Realiza una llamada a Optuna para optimizar los hiperparámetros
search_space = {"lr": [1e-5,1e-4,1e-3,1e-2, 1e-1],  'data_aug' : [True,False], 'batch_size' : [32, 16, 8],
                } ##Con este sampler se hacen todas las combinaciones posibles
study = optuna.create_study(directions = ['minimize','maximize','maximize'], sampler=optuna.samplers.GridSampler(search_space))#None, NSGA por defecto para multiobjetivo, sampler=optuna.samplers.GridSampler(search_space)
print(f"Sampler is {study.sampler.__class__.__name__}")
study.optimize(objective)#, n_trials=100 ##no hace falta indicar el número de pruebas que se tiene

# Muestra los mejores hiperparámetros encontrados
print('Best trial:')
trial = study.best_trials
for t in trial:
    print('Value: {}'.format(t.values))
    print('Params: ')
    for key, value in t.params.items():
        print('{}: {}'.format(key, value))

save_study(study, 'a9_tl_sgd')


c:\Users\da01m\anaconda3\envs\tfg_cnn_lv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2024-04-21 10:00:58,967] A new study created in memory with name: no-name-e6575a34-fdbb-42fd-a581-d87e1d2f8d7b


Sampler is GridSampler
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-21 10:02:38,853] Trial 0 finished with values: [380.5229414121841, 0.7340104184299708, 0.5190485715866089] and parameters: {'batch_size': 16, 'lr': 0.1, 'data_aug': True}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-21 10:04:12,595] Trial 1 finished with values: [116.36577099099982, 0.6588194456199805, 0.459035724401474] and parameters: {'batch_size': 8, 'lr': 0.1, 'data_aug': False}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


... (52 outputs left out)

[I 2024-04-21 10:45:19,394] Trial 28 finished with values: [0.7628538891673088, 0.4244590708240866, 0.17661109566688538] and parameters: {'batch_size': 32, 'lr': 1e-05, 'data_aug': True}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-21 10:46:55,970] Trial 29 finished with values: [0.6337603691592812, 0.6530729176849126, 0.38676390051841736] and parameters: {'batch_size': 16, 'lr': 0.0001, 'data_aug': False}. 


5
Best trial:
Value: [0.442463830858469, 0.8132163855433463, 0.8143333196640015]
Params: 
batch_size: 32
lr: 0.01
data_aug: False
Value: [0.4388294505452116, 0.8150000011424223, 0.6707579493522644]
Params: 
batch_size: 8
lr: 0.001
data_aug: True
Value: [0.4191905711591243, 0.8261805564165116, 0.665242075920105]
Params: 
batch_size: 8
lr: 0.001
data_aug: False


In [12]:
def save_study(study, study_name="study"):
    # Crear una carpeta para guardar el estudio si no existe
    study_dir = os.path.join(os.getcwd(), study_name)
    if not os.path.exists(study_dir):
        os.makedirs(study_dir)
    
    # Guardar los resultados del estudio en un archivo CSV
    study_trials = study.trials_dataframe()
    study_trials.to_csv(os.path.join(study_dir, "study_results.csv"), index=False)

def objective(trial):
    # Define los hiperparámetros para optimizar
    # num_epochs_freeze = trial.suggest_categorical('num_epochs_freeze', [1,3,8,10,15])
    # num_epochs_all = trial.suggest_categorical('num_epochs_all', [25,30,35,40,45,50])
    batch_size = trial.suggest_categorical('batch_size', [32, 16, 8, 4, 48, 5])
    lr = trial.suggest_categorical('lr', [1e-5,1e-4,1e-3,1e-2,1e-1])
    data_aug = trial.suggest_categorical('data_aug', [True,False])
    # ini = trial.suggest_categorical('ini', ['G', 'H', 'N', 'U', 'O', 'Z', 'ON', 'None'])
    # optimizer_name = trial.suggest_categorical('optimizer_name', ['RMSprop', 'Adam', 'SGD'])

    # # Convertir el nombre del optimizador en una clase PyTorch
    # if optimizer_name == 'Adam':
    #     optimizer_class = torch.optim.Adam
    # elif optimizer_name == 'SGD':
    #     optimizer_class = torch.optim.SGD
    # elif optimizer_name == 'RMSprop':
    #     optimizer_class = torch.optim.RMSprop

    # Ejecuta la función de entrenamiento con los hiperparámetros seleccionados
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_pre_oc(df_mean, 5, 5, 35, optim.RMSprop, batch_size, lr, data_aug, None)
    avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_pre(df_a90, 5, 5, 35, optim.RMSprop, batch_size, lr, data_aug, None)
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre(df_a90, 5, 40, optim.RMSprop, batch_size, lr, data_aug, None)
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre_oc(df_a90, 5, 40, optim.RMSprop, batch_size, lr, data_aug, None)

    # Devuelve el valor objetivo (en este caso, la pérdida promedio de validación)
    return avg_val_loss, avg_val_accuracy, avg_val_recall

import optuna
# Supongamos que tienes un dataset llamado 'dataset'
# Realiza una llamada a Optuna para optimizar los hiperparámetros
search_space = {"lr": [1e-5,1e-4,1e-3,1e-2, 1e-1],  'data_aug' : [True,False], 'batch_size' : [32, 16, 8],
                } ##Con este sampler se hacen todas las combinaciones posibles
study = optuna.create_study(directions = ['minimize','maximize','maximize'], sampler=optuna.samplers.GridSampler(search_space))#None, NSGA por defecto para multiobjetivo, sampler=optuna.samplers.GridSampler(search_space)
print(f"Sampler is {study.sampler.__class__.__name__}")
study.optimize(objective)#, n_trials=100

# Muestra los mejores hiperparámetros encontrados
print('Best trial:')
trial = study.best_trials
for t in trial:
    print('Value: {}'.format(t.values))
    print('Params: ')
    for key, value in t.params.items():
        print('{}: {}'.format(key, value))

save_study(study, 'a9_tl_rms')

[I 2024-04-21 10:46:56,108] A new study created in memory with name: no-name-4249a712-2b65-421f-9092-80ceb307ef18


Sampler is GridSampler
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-21 10:48:46,066] Trial 0 finished with values: [12.291974276868494, 0.7274305567145347, 0.5573849081993103] and parameters: {'batch_size': 8, 'lr': 0.001, 'data_aug': True}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-21 10:50:21,389] Trial 1 finished with values: [0.4243981602787971, 0.8299122914671898, 0.7841111421585083] and parameters: {'batch_size': 32, 'lr': 1e-05, 'data_aug': True}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


... (52 outputs left out)

[I 2024-04-21 11:34:28,218] Trial 28 finished with values: [0.45268884619077043, 0.8334722235302131, 0.6303372979164124] and parameters: {'batch_size': 8, 'lr': 1e-05, 'data_aug': True}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-21 11:36:06,108] Trial 29 finished with values: [2602327510639.332, 0.6801900634169579, 0.6677221655845642] and parameters: {'batch_size': 32, 'lr': 0.01, 'data_aug': True}. 


5
Best trial:
Value: [0.4243981602787971, 0.8299122914671898, 0.7841111421585083]
Params: 
batch_size: 32
lr: 1e-05
data_aug: True
Value: [0.5881514725461602, 0.8057163850218056, 0.8344444036483765]
Params: 
batch_size: 32
lr: 0.0001
data_aug: False
Value: [0.45377913295291367, 0.8416666685789824, 0.5988194346427917]
Params: 
batch_size: 16
lr: 1e-05
data_aug: True
Value: [0.41468247235442207, 0.8252083345750968, 0.6512738466262817]
Params: 
batch_size: 8
lr: 1e-05
data_aug: False
Value: [0.47243870854377745, 0.7596929888054729, 0.8466666340827942]
Params: 
batch_size: 32
lr: 1e-05
data_aug: False
Value: [0.45268884619077043, 0.8334722235302131, 0.6303372979164124]
Params: 
batch_size: 8
lr: 1e-05
data_aug: True


In [13]:
def save_study(study, study_name="study"):
    # Crear una carpeta para guardar el estudio si no existe
    study_dir = os.path.join(os.getcwd(), study_name)
    if not os.path.exists(study_dir):
        os.makedirs(study_dir)
    
    # Guardar los resultados del estudio en un archivo CSV
    study_trials = study.trials_dataframe()
    study_trials.to_csv(os.path.join(study_dir, "study_results.csv"), index=False)

def objective(trial):
    # Define los hiperparámetros para optimizar
    # num_epochs_freeze = trial.suggest_categorical('num_epochs_freeze', [1,3,8,10,15])
    # num_epochs_all = trial.suggest_categorical('num_epochs_all', [25,30,35,40,45,50])
    batch_size = trial.suggest_categorical('batch_size', [32, 16, 8, 4, 48, 5])
    lr = trial.suggest_categorical('lr', [1e-5,1e-4,1e-3,1e-2,1e-1])
    data_aug = trial.suggest_categorical('data_aug', [True,False])
    # ini = trial.suggest_categorical('ini', ['G', 'H', 'N', 'U', 'O', 'Z', 'ON', 'None'])
    # optimizer_name = trial.suggest_categorical('optimizer_name', ['RMSprop', 'Adam', 'SGD'])

    # # Convertir el nombre del optimizador en una clase PyTorch
    # if optimizer_name == 'Adam':
    #     optimizer_class = torch.optim.Adam
    # elif optimizer_name == 'SGD':
    #     optimizer_class = torch.optim.SGD
    # elif optimizer_name == 'RMSprop':
    #     optimizer_class = torch.optim.RMSprop

    # Ejecuta la función de entrenamiento con los hiperparámetros seleccionados
    avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_pre_oc(df_a90, 5, 5, 35, optim.Adam, batch_size, lr, data_aug, None)
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_pre(df_a90, 5, 5, 35, optim.SGD, batch_size, lr, data_aug, None)
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre(df_a90, 5, 40, optim.RMSprop, batch_size, lr, data_aug, None)
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre_oc(df_a90, 5, 40, optim.RMSprop, batch_size, lr, data_aug, None)

    # Devuelve el valor objetivo (en este caso, la pérdida promedio de validación)
    return avg_val_loss, avg_val_accuracy, avg_val_recall

import optuna
# Supongamos que tienes un dataset llamado 'dataset'
# Realiza una llamada a Optuna para optimizar los hiperparámetros
search_space = {"lr": [1e-5,1e-4,1e-3,1e-2, 1e-1],  'data_aug' : [True,False], 'batch_size' : [32, 16, 8],
                } ##Con este sampler se hacen todas las combinaciones posibles
study = optuna.create_study(directions = ['minimize','maximize','maximize'], sampler=optuna.samplers.GridSampler(search_space))#None, NSGA por defecto para multiobjetivo, sampler=optuna.samplers.GridSampler(search_space)
print(f"Sampler is {study.sampler.__class__.__name__}")
study.optimize(objective)#, n_trials=100

# Muestra los mejores hiperparámetros encontrados
print('Best trial:')
trial = study.best_trials
for t in trial:
    print('Value: {}'.format(t.values))
    print('Params: ')
    for key, value in t.params.items():
        print('{}: {}'.format(key, value))

save_study(study, 'a9_tl_oc_adam')

[I 2024-04-21 11:36:06,209] A new study created in memory with name: no-name-afed8ef2-c999-468c-9670-d8b781683011


Sampler is GridSampler
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-21 11:37:41,965] Trial 0 finished with values: [0.6888838139176368, 0.6130847957730292, 0.625] and parameters: {'batch_size': 32, 'lr': 1e-05, 'data_aug': False}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-21 11:39:17,459] Trial 1 finished with values: [0.5258037021756172, 0.7463450390100479, 0.8552778363227844] and parameters: {'batch_size': 32, 'lr': 0.0001, 'data_aug': False}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


... (52 outputs left out)

[I 2024-04-21 12:25:08,716] Trial 28 finished with values: [0.5244771641819778, 0.8188888900975387, 0.6603332757949829] and parameters: {'batch_size': 8, 'lr': 0.001, 'data_aug': True}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-21 12:26:49,793] Trial 29 finished with values: [0.6829263560473919, 0.5848958349227906, 0.366854190826416] and parameters: {'batch_size': 16, 'lr': 1e-05, 'data_aug': False}. 


5
Best trial:
Value: [0.5258037021756172, 0.7463450390100479, 0.8552778363227844]
Params: 
batch_size: 32
lr: 0.0001
data_aug: False
Value: [0.502978223413229, 0.79444445297122, 0.8118888735771179]
Params: 
batch_size: 32
lr: 0.001
data_aug: True
Value: [0.5150733168423176, 0.7676169654726982, 0.8465555906295776]
Params: 
batch_size: 32
lr: 0.001
data_aug: False
Value: [0.5244771641819778, 0.8188888900975387, 0.6603332757949829]
Params: 
batch_size: 8
lr: 0.001
data_aug: True


In [14]:
def save_study(study, study_name="study"):
    # Crear una carpeta para guardar el estudio si no existe
    study_dir = os.path.join(os.getcwd(), study_name)
    if not os.path.exists(study_dir):
        os.makedirs(study_dir)
    
    # Guardar los resultados del estudio en un archivo CSV
    study_trials = study.trials_dataframe()
    study_trials.to_csv(os.path.join(study_dir, "study_results.csv"), index=False)

def objective(trial):
    # Define los hiperparámetros para optimizar
    # num_epochs_freeze = trial.suggest_categorical('num_epochs_freeze', [1,3,8,10,15])
    # num_epochs_all = trial.suggest_categorical('num_epochs_all', [25,30,35,40,45,50])
    batch_size = trial.suggest_categorical('batch_size', [32, 16, 8, 4, 48, 5])
    lr = trial.suggest_categorical('lr', [1e-5,1e-4,1e-3,1e-2,1e-1])
    data_aug = trial.suggest_categorical('data_aug', [True,False])
    # ini = trial.suggest_categorical('ini', ['G', 'H', 'N', 'U', 'O', 'Z', 'ON', 'None'])
    # optimizer_name = trial.suggest_categorical('optimizer_name', ['RMSprop', 'Adam', 'SGD'])

    # # Convertir el nombre del optimizador en una clase PyTorch
    # if optimizer_name == 'Adam':
    #     optimizer_class = torch.optim.Adam
    # elif optimizer_name == 'SGD':
    #     optimizer_class = torch.optim.SGD
    # elif optimizer_name == 'RMSprop':
    #     optimizer_class = torch.optim.RMSprop

    # Ejecuta la función de entrenamiento con los hiperparámetros seleccionados
    avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_pre_oc(df_a90, 5, 5, 35, optim.SGD, batch_size, lr, data_aug, None)
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_pre(df_a90, 5, 5, 35, optim.SGD, batch_size, lr, data_aug, None)
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre(df_a90, 5, 40, optim.RMSprop, batch_size, lr, data_aug, None)
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre_oc(df_a90, 5, 40, optim.RMSprop, batch_size, lr, data_aug, None)

    # Devuelve el valor objetivo (en este caso, la pérdida promedio de validación)
    return avg_val_loss, avg_val_accuracy, avg_val_recall

import optuna
# Supongamos que tienes un dataset llamado 'dataset'
# Realiza una llamada a Optuna para optimizar los hiperparámetros
search_space = {"lr": [1e-5,1e-4,1e-3,1e-2, 1e-1],  'data_aug' : [True,False], 'batch_size' : [32, 16, 8],
                } ##Con este sampler se hacen todas las combinaciones posibles
study = optuna.create_study(directions = ['minimize','maximize','maximize'], sampler=optuna.samplers.GridSampler(search_space))#None, NSGA por defecto para multiobjetivo, sampler=optuna.samplers.GridSampler(search_space)
print(f"Sampler is {study.sampler.__class__.__name__}")
study.optimize(objective)#, n_trials=100

# Muestra los mejores hiperparámetros encontrados
print('Best trial:')
trial = study.best_trials
for t in trial:
    print('Value: {}'.format(t.values))
    print('Params: ')
    for key, value in t.params.items():
        print('{}: {}'.format(key, value))

save_study(study, 'a9_tl_oc_sgd')

[I 2024-04-21 12:26:50,243] A new study created in memory with name: no-name-a5ae35bc-71fb-417c-8ee8-bd35abb4d5de


Sampler is GridSampler
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-21 12:28:39,309] Trial 0 finished with values: [0.5394976450502872, 0.7910416677594184, 0.6057341694831848] and parameters: {'batch_size': 8, 'lr': 0.001, 'data_aug': True}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-21 12:30:19,310] Trial 1 finished with values: [3.3285791132740266, 0.726666668280959, 0.539770781993866] and parameters: {'batch_size': 16, 'lr': 0.1, 'data_aug': True}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


... (52 outputs left out)

[I 2024-04-21 13:14:55,898] Trial 28 finished with values: [0.7224351799488067, 0.479912284836173, 0.29561111330986023] and parameters: {'batch_size': 32, 'lr': 0.0001, 'data_aug': True}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-21 13:16:34,347] Trial 29 finished with values: [0.8602643360197544, 0.4736458343267441, 0.31532639265060425] and parameters: {'batch_size': 16, 'lr': 0.0001, 'data_aug': True}. 


5
Best trial:
Value: [0.5394976450502872, 0.7910416677594184, 0.6057341694831848]
Params: 
batch_size: 8
lr: 0.001
data_aug: True
Value: [11.909000508263707, 0.7788450345396994, 0.8129444122314453]
Params: 
batch_size: 32
lr: 0.1
data_aug: True
Value: [0.5652777787360053, 0.8031944456696511, 0.6778254508972168]
Params: 
batch_size: 8
lr: 0.01
data_aug: True
Value: [0.45519022159278394, 0.7904532226920128, 0.7881666421890259]
Params: 
batch_size: 32
lr: 0.01
data_aug: False
Value: [346.73571434213784, 0.754005854278803, 0.8200554847717285]
Params: 
batch_size: 32
lr: 0.1
data_aug: False


In [15]:
def save_study(study, study_name="study"):
    # Crear una carpeta para guardar el estudio si no existe
    study_dir = os.path.join(os.getcwd(), study_name)
    if not os.path.exists(study_dir):
        os.makedirs(study_dir)
    
    # Guardar los resultados del estudio en un archivo CSV
    study_trials = study.trials_dataframe()
    study_trials.to_csv(os.path.join(study_dir, "study_results.csv"), index=False)

def objective(trial):
    # Define los hiperparámetros para optimizar
    # num_epochs_freeze = trial.suggest_categorical('num_epochs_freeze', [1,3,8,10,15])
    # num_epochs_all = trial.suggest_categorical('num_epochs_all', [25,30,35,40,45,50])
    batch_size = trial.suggest_categorical('batch_size', [32, 16, 8, 4, 48, 5])
    lr = trial.suggest_categorical('lr', [1e-5,1e-4,1e-3,1e-2,1e-1])
    data_aug = trial.suggest_categorical('data_aug', [True,False])
    # ini = trial.suggest_categorical('ini', ['G', 'H', 'N', 'U', 'O', 'Z', 'ON', 'None'])
    # optimizer_name = trial.suggest_categorical('optimizer_name', ['RMSprop', 'Adam', 'SGD'])

    # # Convertir el nombre del optimizador en una clase PyTorch
    # if optimizer_name == 'Adam':
    #     optimizer_class = torch.optim.Adam
    # elif optimizer_name == 'SGD':
    #     optimizer_class = torch.optim.SGD
    # elif optimizer_name == 'RMSprop':
    #     optimizer_class = torch.optim.RMSprop

    # Ejecuta la función de entrenamiento con los hiperparámetros seleccionados
    avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_pre_oc(df_a90, 5, 5, 35, optim.RMSprop, batch_size, lr, data_aug, None)
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_pre(df_a90, 5, 5, 35, optim.SGD, batch_size, lr, data_aug, None)
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre(df_a90, 5, 40, optim.RMSprop, batch_size, lr, data_aug, None)
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre_oc(df_a90, 5, 40, optim.RMSprop, batch_size, lr, data_aug, None)

    # Devuelve el valor objetivo (en este caso, la pérdida promedio de validación)
    return avg_val_loss, avg_val_accuracy, avg_val_recall

import optuna
# Supongamos que tienes un dataset llamado 'dataset'
# Realiza una llamada a Optuna para optimizar los hiperparámetros
search_space = {"lr": [1e-5,1e-4,1e-3,1e-2, 1e-1],  'data_aug' : [True,False], 'batch_size' : [32, 16, 8],
                } ##Con este sampler se hacen todas las combinaciones posibles
study = optuna.create_study(directions = ['minimize','maximize','maximize'], sampler=optuna.samplers.GridSampler(search_space))#None, NSGA por defecto para multiobjetivo, sampler=optuna.samplers.GridSampler(search_space)
print(f"Sampler is {study.sampler.__class__.__name__}")
study.optimize(objective)#, n_trials=100

# Muestra los mejores hiperparámetros encontrados
print('Best trial:')
trial = study.best_trials
for t in trial:
    print('Value: {}'.format(t.values))
    print('Params: ')
    for key, value in t.params.items():
        print('{}: {}'.format(key, value))

save_study(study, 'a9_tl_oc_rms')

[I 2024-04-21 13:16:34,373] A new study created in memory with name: no-name-edf2857b-524d-4d37-b414-43b5833387ba


Sampler is GridSampler
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-21 13:18:12,871] Trial 0 finished with values: [4.915402383008542, 0.7539583352208138, 0.5548263788223267] and parameters: {'batch_size': 16, 'lr': 0.001, 'data_aug': False}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-21 13:20:02,819] Trial 1 finished with values: [0.48694734624276564, 0.8271527790029843, 0.6138055324554443] and parameters: {'batch_size': 8, 'lr': 1e-05, 'data_aug': True}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


... (52 outputs left out)

[I 2024-04-21 14:02:54,782] Trial 28 finished with values: [399039936265.02576, 0.6606579005718232, 0.7002221941947937] and parameters: {'batch_size': 32, 'lr': 0.1, 'data_aug': True}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-21 14:04:27,087] Trial 29 finished with values: [653954108714.0386, 0.6668229184299708, 0.5573889017105103] and parameters: {'batch_size': 16, 'lr': 0.1, 'data_aug': False}. 


5
Best trial:
Value: [0.48694734624276564, 0.8271527790029843, 0.6138055324554443]
Params: 
batch_size: 8
lr: 1e-05
data_aug: True
Value: [0.47277430124580866, 0.7735818783938885, 0.7745000123977661]
Params: 
batch_size: 32
lr: 1e-05
data_aug: True
Value: [0.6248041906947037, 0.7888888901472091, 0.6462500691413879]
Params: 
batch_size: 8
lr: 0.0001
data_aug: True
Value: [0.7627913466840983, 0.7886403591930866, 0.8447777628898621]
Params: 
batch_size: 32
lr: 0.0001
data_aug: False
Value: [0.5998342803276606, 0.8281944456199805, 0.638353168964386]
Params: 
batch_size: 8
lr: 0.0001
data_aug: False


## FDG

A continuación, se encuentran todas las pruebas con GridSearch hechas para FDG.

In [16]:
def save_study(study, study_name="study"):
    # Crear una carpeta para guardar el estudio si no existe
    study_dir = os.path.join(os.getcwd(), study_name)
    if not os.path.exists(study_dir):
        os.makedirs(study_dir)
    
    # Guardar los resultados del estudio en un archivo CSV
    study_trials = study.trials_dataframe()
    study_trials.to_csv(os.path.join(study_dir, "study_results.csv"), index=False)

def objective(trial):
    # Define los hiperparámetros para optimizar
    # num_epochs_freeze = trial.suggest_categorical('num_epochs_freeze', [1,3,8,10,15])
    # num_epochs_all = trial.suggest_categorical('num_epochs_all', [25,30,35,40,45,50])
    batch_size = trial.suggest_categorical('batch_size', [32, 16, 8, 4, 48, 5])
    lr = trial.suggest_categorical('lr', [1e-5,1e-4,1e-3,1e-2,1e-1])
    data_aug = trial.suggest_categorical('data_aug', [True,False])
    # ini = trial.suggest_categorical('ini', ['G', 'H', 'N', 'U', 'O', 'Z', 'ON', 'None'])
    # optimizer_name = trial.suggest_categorical('optimizer_name', ['RMSprop', 'Adam', 'SGD'])

    # # Convertir el nombre del optimizador en una clase PyTorch
    # if optimizer_name == 'Adam':
    #     optimizer_class = torch.optim.Adam
    # elif optimizer_name == 'SGD':
    #     optimizer_class = torch.optim.SGD
    # elif optimizer_name == 'RMSprop':
    #     optimizer_class = torch.optim.RMSprop

    # Ejecuta la función de entrenamiento con los hiperparámetros seleccionados
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_pre_oc(df_a90, 5, 5, 35, optim.RMSprop, batch_size, lr, data_aug, None)
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_pre(df_a90, 5, 5, 35, optim.SGD, batch_size, lr, data_aug, None)
    avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre(df_fdg, 5, 40, optim.Adam, batch_size, lr, data_aug, None)
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre_oc(df_a90, 5, 40, optim.RMSprop, batch_size, lr, data_aug, None)

    # Devuelve el valor objetivo (en este caso, la pérdida promedio de validación)
    return avg_val_loss, avg_val_accuracy, avg_val_recall

import optuna
# Supongamos que tienes un dataset llamado 'dataset'
# Realiza una llamada a Optuna para optimizar los hiperparámetros
search_space = {"lr": [1e-5,1e-4,1e-3,1e-2, 1e-1],  'data_aug' : [True,False], 'batch_size' : [32, 16, 8],
                } ##Con este sampler se hacen todas las combinaciones posibles
study = optuna.create_study(directions = ['minimize','maximize','maximize'], sampler=optuna.samplers.GridSampler(search_space))#None, NSGA por defecto para multiobjetivo, sampler=optuna.samplers.GridSampler(search_space)
print(f"Sampler is {study.sampler.__class__.__name__}")
study.optimize(objective)#, n_trials=100

# Muestra los mejores hiperparámetros encontrados
print('Best trial:')
trial = study.best_trials
for t in trial:
    print('Value: {}'.format(t.values))
    print('Params: ')
    for key, value in t.params.items():
        print('{}: {}'.format(key, value))

save_study(study, 'fdg_adam')

[I 2024-04-21 14:04:27,113] A new study created in memory with name: no-name-3459dfb9-30a2-467b-b23d-ec7a507d791f


Sampler is GridSampler
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-21 14:05:20,413] Trial 0 finished with values: [0.6909407708048821, 0.5778889027237892, 0.2865000069141388] and parameters: {'batch_size': 16, 'lr': 1e-05, 'data_aug': True}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-21 14:06:13,220] Trial 1 finished with values: [1.4738187435269352, 0.5577222342789173, 0.48774999380111694] and parameters: {'batch_size': 16, 'lr': 0.0001, 'data_aug': True}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


... (52 outputs left out)

[I 2024-04-21 14:30:26,248] Trial 28 finished with values: [3.5127815778553484, 0.48922223106026647, 0.690750002861023] and parameters: {'batch_size': 16, 'lr': 0.001, 'data_aug': True}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-21 14:31:18,113] Trial 29 finished with values: [258580875263.31363, 0.5295000129193068, 0.4610000252723694] and parameters: {'batch_size': 16, 'lr': 0.1, 'data_aug': False}. 


5
Best trial:
Value: [0.6909407708048821, 0.5778889027237892, 0.2865000069141388]
Params: 
batch_size: 16
lr: 1e-05
data_aug: True
Value: [1.4738187435269352, 0.5577222342789173, 0.48774999380111694]
Params: 
batch_size: 16
lr: 0.0001
data_aug: True
Value: [1.0778371149301529, 0.5500000119209291, 0.4619999825954437]
Params: 
batch_size: 32
lr: 0.0001
data_aug: False
Value: [0.7137237688899041, 0.5211111207306385, 0.48499998450279236]
Params: 
batch_size: 32
lr: 1e-05
data_aug: True
Value: [0.7127330389618873, 0.5515555687248707, 0.4367499351501465]
Params: 
batch_size: 32
lr: 1e-05
data_aug: False
Value: [0.7195436653494836, 0.6340000119805336, 0.3959999978542328]
Params: 
batch_size: 16
lr: 1e-05
data_aug: False
Value: [1.8060155132412912, 0.5241666777431965, 0.5952500104904175]
Params: 
batch_size: 32
lr: 0.0001
data_aug: True
Value: [6.950025264620781, 0.5297777868807316, 0.6274999976158142]
Params: 
batch_size: 16
lr: 0.001
data_aug: False
Value: [3.5127815778553484, 0.489222231060

In [17]:
def save_study(study, study_name="study"):
    # Crear una carpeta para guardar el estudio si no existe
    study_dir = os.path.join(os.getcwd(), study_name)
    if not os.path.exists(study_dir):
        os.makedirs(study_dir)
    
    # Guardar los resultados del estudio en un archivo CSV
    study_trials = study.trials_dataframe()
    study_trials.to_csv(os.path.join(study_dir, "study_results.csv"), index=False)

def objective(trial):
    # Define los hiperparámetros para optimizar
    # num_epochs_freeze = trial.suggest_categorical('num_epochs_freeze', [1,3,8,10,15])
    # num_epochs_all = trial.suggest_categorical('num_epochs_all', [25,30,35,40,45,50])
    batch_size = trial.suggest_categorical('batch_size', [32, 16, 8, 4, 48, 5])
    lr = trial.suggest_categorical('lr', [1e-5,1e-4,1e-3,1e-2,1e-1])
    data_aug = trial.suggest_categorical('data_aug', [True,False])
    # ini = trial.suggest_categorical('ini', ['G', 'H', 'N', 'U', 'O', 'Z', 'ON', 'None'])
    # optimizer_name = trial.suggest_categorical('optimizer_name', ['RMSprop', 'Adam', 'SGD'])

    # # Convertir el nombre del optimizador en una clase PyTorch
    # if optimizer_name == 'Adam':
    #     optimizer_class = torch.optim.Adam
    # elif optimizer_name == 'SGD':
    #     optimizer_class = torch.optim.SGD
    # elif optimizer_name == 'RMSprop':
    #     optimizer_class = torch.optim.RMSprop

    # Ejecuta la función de entrenamiento con los hiperparámetros seleccionados
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_pre_oc(df_a90, 5, 5, 35, optim.RMSprop, batch_size, lr, data_aug, None)
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_pre(df_a90, 5, 5, 35, optim.SGD, batch_size, lr, data_aug, None)
    avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre(df_fdg, 5, 40, optim.SGD, batch_size, lr, data_aug, None)
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre_oc(df_a90, 5, 40, optim.RMSprop, batch_size, lr, data_aug, None)

    # Devuelve el valor objetivo (en este caso, la pérdida promedio de validación)
    return avg_val_loss, avg_val_accuracy, avg_val_recall

import optuna
# Supongamos que tienes un dataset llamado 'dataset'
# Realiza una llamada a Optuna para optimizar los hiperparámetros
search_space = {"lr": [1e-5,1e-4,1e-3,1e-2, 1e-1],  'data_aug' : [True,False], 'batch_size' : [32, 16, 8],
                } ##Con este sampler se hacen todas las combinaciones posibles
study = optuna.create_study(directions = ['minimize','maximize','maximize'], sampler=optuna.samplers.GridSampler(search_space))#None, NSGA por defecto para multiobjetivo, sampler=optuna.samplers.GridSampler(search_space)
print(f"Sampler is {study.sampler.__class__.__name__}")
study.optimize(objective)#, n_trials=100

# Muestra los mejores hiperparámetros encontrados
print('Best trial:')
trial = study.best_trials
for t in trial:
    print('Value: {}'.format(t.values))
    print('Params: ')
    for key, value in t.params.items():
        print('{}: {}'.format(key, value))

save_study(study, 'fdg_sgd')

[I 2024-04-21 14:31:18,138] A new study created in memory with name: no-name-7282e5ef-5ee0-46ab-abb2-675ca5a5ae7d


Sampler is GridSampler
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-21 14:32:12,241] Trial 0 finished with values: [0.8368352147750556, 0.5728125, 0.4047916531562805] and parameters: {'batch_size': 8, 'lr': 0.01, 'data_aug': False}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-21 14:33:03,015] Trial 1 finished with values: [5.862182708829642, 0.5195000117272139, 0.47049999237060547] and parameters: {'batch_size': 16, 'lr': 0.1, 'data_aug': True}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


... (52 outputs left out)

[I 2024-04-21 14:56:09,464] Trial 28 finished with values: [6.555154018394077, 0.52, 0.47833338379859924] and parameters: {'batch_size': 8, 'lr': 0.1, 'data_aug': True}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-21 14:56:59,358] Trial 29 finished with values: [0.6923655831813812, 0.5378333461284637, 0.27000001072883606] and parameters: {'batch_size': 16, 'lr': 0.0001, 'data_aug': False}. 


5
Best trial:
Value: [0.8368352147750556, 0.5728125, 0.4047916531562805]
Params: 
batch_size: 8
lr: 0.01
data_aug: False
Value: [0.7006497630476952, 0.5131666760146618, 0.36250001192092896]
Params: 
batch_size: 32
lr: 1e-05
data_aug: True
Value: [0.728425708413124, 0.5005555675923825, 0.5949999690055847]
Params: 
batch_size: 32
lr: 1e-05
data_aug: False
Value: [0.7632854366302491, 0.543333344757557, 0.46050000190734863]
Params: 
batch_size: 16
lr: 0.01
data_aug: False
Value: [0.7271591091156007, 0.5000000119209289, 0.6000000238418579]
Params: 
batch_size: 16
lr: 1e-05
data_aug: False
Value: [0.7404040437936782, 0.5196666799485683, 0.4364999830722809]
Params: 
batch_size: 32
lr: 0.01
data_aug: True
Value: [0.7401821574568748, 0.5184444576501847, 0.39399999380111694]
Params: 
batch_size: 32
lr: 0.01
data_aug: False
Value: [0.6957178729772566, 0.5016111212968827, 0.48249998688697815]
Params: 
batch_size: 32
lr: 0.0001
data_aug: False
Value: [0.6880897694826127, 0.5492777956277133, 0.10875

In [18]:
def save_study(study, study_name="study"):
    # Crear una carpeta para guardar el estudio si no existe
    study_dir = os.path.join(os.getcwd(), study_name)
    if not os.path.exists(study_dir):
        os.makedirs(study_dir)
    
    # Guardar los resultados del estudio en un archivo CSV
    study_trials = study.trials_dataframe()
    study_trials.to_csv(os.path.join(study_dir, "study_results.csv"), index=False)

def objective(trial):
    # Define los hiperparámetros para optimizar
    # num_epochs_freeze = trial.suggest_categorical('num_epochs_freeze', [1,3,8,10,15])
    # num_epochs_all = trial.suggest_categorical('num_epochs_all', [25,30,35,40,45,50])
    batch_size = trial.suggest_categorical('batch_size', [32, 16, 8, 4, 48, 5])
    lr = trial.suggest_categorical('lr', [1e-5,1e-4,1e-3,1e-2,1e-1])
    data_aug = trial.suggest_categorical('data_aug', [True,False])
    # ini = trial.suggest_categorical('ini', ['G', 'H', 'N', 'U', 'O', 'Z', 'ON', 'None'])
    # optimizer_name = trial.suggest_categorical('optimizer_name', ['RMSprop', 'Adam', 'SGD'])

    # # Convertir el nombre del optimizador en una clase PyTorch
    # if optimizer_name == 'Adam':
    #     optimizer_class = torch.optim.Adam
    # elif optimizer_name == 'SGD':
    #     optimizer_class = torch.optim.SGD
    # elif optimizer_name == 'RMSprop':
    #     optimizer_class = torch.optim.RMSprop

    # Ejecuta la función de entrenamiento con los hiperparámetros seleccionados
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_pre_oc(df_a90, 5, 5, 35, optim.RMSprop, batch_size, lr, data_aug, None)
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_pre(df_a90, 5, 5, 35, optim.SGD, batch_size, lr, data_aug, None)
    avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre(df_fdg, 5, 40, optim.RMSprop, batch_size, lr, data_aug, None)
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre_oc(df_a90, 5, 40, optim.RMSprop, batch_size, lr, data_aug, None)

    # Devuelve el valor objetivo (en este caso, la pérdida promedio de validación)
    return avg_val_loss, avg_val_accuracy, avg_val_recall

import optuna
# Supongamos que tienes un dataset llamado 'dataset'
# Realiza una llamada a Optuna para optimizar los hiperparámetros
search_space = {"lr": [1e-5,1e-4,1e-3,1e-2, 1e-1],  'data_aug' : [True,False], 'batch_size' : [32, 16, 8],
                } ##Con este sampler se hacen todas las combinaciones posibles
study = optuna.create_study(directions = ['minimize','maximize','maximize'], sampler=optuna.samplers.GridSampler(search_space))#None, NSGA por defecto para multiobjetivo, sampler=optuna.samplers.GridSampler(search_space)
print(f"Sampler is {study.sampler.__class__.__name__}")
study.optimize(objective)#, n_trials=100

# Muestra los mejores hiperparámetros encontrados
print('Best trial:')
trial = study.best_trials
for t in trial:
    print('Value: {}'.format(t.values))
    print('Params: ')
    for key, value in t.params.items():
        print('{}: {}'.format(key, value))

save_study(study, 'fdg_rms')

[I 2024-04-21 14:56:59,382] A new study created in memory with name: no-name-1a2742f5-a8ae-4531-bf8f-49670786263e


Sampler is GridSampler
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-21 14:57:51,758] Trial 0 finished with values: [501326834.79504406, 0.523833346515894, 0.36649999022483826] and parameters: {'batch_size': 16, 'lr': 0.01, 'data_aug': True}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-21 14:58:49,168] Trial 1 finished with values: [349.12248225429755, 0.53125, 0.31854167580604553] and parameters: {'batch_size': 8, 'lr': 0.001, 'data_aug': True}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


... (52 outputs left out)

[I 2024-04-21 15:22:33,893] Trial 28 finished with values: [0.7619564288854599, 0.5790555676817893, 0.4677499830722809] and parameters: {'batch_size': 16, 'lr': 1e-05, 'data_aug': True}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-21 15:23:31,389] Trial 29 finished with values: [1.1973129077909228, 0.5765625, 0.4170833230018616] and parameters: {'batch_size': 8, 'lr': 0.0001, 'data_aug': True}. 


5
Best trial:
Value: [1.3912574373185636, 0.5585000105202198, 0.5142499804496765]
Params: 
batch_size: 16
lr: 0.0001
data_aug: False
Value: [1.1408174851536752, 0.6134375000000001, 0.39916664361953735]
Params: 
batch_size: 8
lr: 1e-05
data_aug: False
Value: [0.7439372324943542, 0.5590000125020743, 0.46024999022483826]
Params: 
batch_size: 32
lr: 1e-05
data_aug: False
Value: [143285.0890716946, 0.5188888988643885, 0.5354999899864197]
Params: 
batch_size: 16
lr: 0.001
data_aug: False
Value: [0.7792005705833435, 0.6001111242175103, 0.4282499849796295]
Params: 
batch_size: 16
lr: 1e-05
data_aug: False
Value: [0.7619564288854599, 0.5790555676817893, 0.4677499830722809]
Params: 
batch_size: 16
lr: 1e-05
data_aug: True


In [19]:
def save_study(study, study_name="study"):
    # Crear una carpeta para guardar el estudio si no existe
    study_dir = os.path.join(os.getcwd(), study_name)
    if not os.path.exists(study_dir):
        os.makedirs(study_dir)
    
    # Guardar los resultados del estudio en un archivo CSV
    study_trials = study.trials_dataframe()
    study_trials.to_csv(os.path.join(study_dir, "study_results.csv"), index=False)

def objective(trial):
    # Define los hiperparámetros para optimizar
    # num_epochs_freeze = trial.suggest_categorical('num_epochs_freeze', [1,3,8,10,15])
    # num_epochs_all = trial.suggest_categorical('num_epochs_all', [25,30,35,40,45,50])
    batch_size = trial.suggest_categorical('batch_size', [32, 16, 8, 4, 48, 5])
    lr = trial.suggest_categorical('lr', [1e-5,1e-4,1e-3,1e-2,1e-1])
    data_aug = trial.suggest_categorical('data_aug', [True,False])
    # ini = trial.suggest_categorical('ini', ['G', 'H', 'N', 'U', 'O', 'Z', 'ON', 'None'])
    # optimizer_name = trial.suggest_categorical('optimizer_name', ['RMSprop', 'Adam', 'SGD'])

    # # Convertir el nombre del optimizador en una clase PyTorch
    # if optimizer_name == 'Adam':
    #     optimizer_class = torch.optim.Adam
    # elif optimizer_name == 'SGD':
    #     optimizer_class = torch.optim.SGD
    # elif optimizer_name == 'RMSprop':
    #     optimizer_class = torch.optim.RMSprop

    # Ejecuta la función de entrenamiento con los hiperparámetros seleccionados
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_pre_oc(df_a90, 5, 5, 35, optim.RMSprop, batch_size, lr, data_aug, None)
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_pre(df_a90, 5, 5, 35, optim.SGD, batch_size, lr, data_aug, None)
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre(df_fdg, 5, 40, optim.RMSprop, batch_size, lr, data_aug, None)
    avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre_oc(df_fdg, 5, 40, optim.Adam, batch_size, lr, data_aug, None)

    # Devuelve el valor objetivo (en este caso, la pérdida promedio de validación)
    return avg_val_loss, avg_val_accuracy, avg_val_recall

import optuna
# Supongamos que tienes un dataset llamado 'dataset'
# Realiza una llamada a Optuna para optimizar los hiperparámetros
search_space = {"lr": [1e-5,1e-4,1e-3,1e-2, 1e-1],  'data_aug' : [True,False], 'batch_size' : [32, 16, 8],
                } ##Con este sampler se hacen todas las combinaciones posibles
study = optuna.create_study(directions = ['minimize','maximize','maximize'], sampler=optuna.samplers.GridSampler(search_space))#None, NSGA por defecto para multiobjetivo, sampler=optuna.samplers.GridSampler(search_space)
print(f"Sampler is {study.sampler.__class__.__name__}")
study.optimize(objective)#, n_trials=100

# Muestra los mejores hiperparámetros encontrados
print('Best trial:')
trial = study.best_trials
for t in trial:
    print('Value: {}'.format(t.values))
    print('Params: ')
    for key, value in t.params.items():
        print('{}: {}'.format(key, value))

save_study(study, 'fdg_oc_adam')

[I 2024-04-21 15:23:31,418] A new study created in memory with name: no-name-e5450ba1-d7c8-42e7-b6b7-e1d20f9b168c


Sampler is GridSampler
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-21 15:24:23,302] Trial 0 finished with values: [422.8447216123342, 0.4963333448767663, 0.4757499694824219] and parameters: {'batch_size': 32, 'lr': 0.01, 'data_aug': True}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-21 15:25:20,894] Trial 1 finished with values: [0.7869456302374601, 0.55375, 0.27666670083999634] and parameters: {'batch_size': 8, 'lr': 1e-05, 'data_aug': False}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


... (52 outputs left out)

[I 2024-04-21 15:49:36,054] Trial 28 finished with values: [65.27193265736103, 0.554777788668871, 0.5005000233650208] and parameters: {'batch_size': 16, 'lr': 0.01, 'data_aug': False}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-21 15:50:27,695] Trial 29 finished with values: [0.937127884030342, 0.5381111231446266, 0.4934999942779541] and parameters: {'batch_size': 32, 'lr': 0.0001, 'data_aug': True}. 


5
Best trial:
Value: [0.6976015575230121, 0.5731249999999999, 0.34062498807907104]
Params: 
batch_size: 8
lr: 1e-05
data_aug: True
Value: [2.432711094766855, 0.5325000095367433, 0.6184999942779541]
Params: 
batch_size: 16
lr: 0.001
data_aug: True
Value: [8.692838355451823, 0.5662777862697841, 0.590499997138977]
Params: 
batch_size: 32
lr: 0.001
data_aug: False
Value: [1.1110081100463867, 0.5741666809469461, 0.4662500321865082]
Params: 
batch_size: 16
lr: 0.0001
data_aug: False
Value: [1.1305889230263664, 0.5721875000000001, 0.47333335876464844]
Params: 
batch_size: 8
lr: 0.0001
data_aug: True
Value: [0.6882706469297408, 0.5707222355902195, 0.5677499771118164]
Params: 
batch_size: 32
lr: 1e-05
data_aug: False


In [20]:
def save_study(study, study_name="study"):
    # Crear una carpeta para guardar el estudio si no existe
    study_dir = os.path.join(os.getcwd(), study_name)
    if not os.path.exists(study_dir):
        os.makedirs(study_dir)
    
    # Guardar los resultados del estudio en un archivo CSV
    study_trials = study.trials_dataframe()
    study_trials.to_csv(os.path.join(study_dir, "study_results.csv"), index=False)

def objective(trial):
    # Define los hiperparámetros para optimizar
    # num_epochs_freeze = trial.suggest_categorical('num_epochs_freeze', [1,3,8,10,15])
    # num_epochs_all = trial.suggest_categorical('num_epochs_all', [25,30,35,40,45,50])
    batch_size = trial.suggest_categorical('batch_size', [32, 16, 8, 4, 48, 5])
    lr = trial.suggest_categorical('lr', [1e-5,1e-4,1e-3,1e-2,1e-1])
    data_aug = trial.suggest_categorical('data_aug', [True,False])
    # ini = trial.suggest_categorical('ini', ['G', 'H', 'N', 'U', 'O', 'Z', 'ON', 'None'])
    # optimizer_name = trial.suggest_categorical('optimizer_name', ['RMSprop', 'Adam', 'SGD'])

    # # Convertir el nombre del optimizador en una clase PyTorch
    # if optimizer_name == 'Adam':
    #     optimizer_class = torch.optim.Adam
    # elif optimizer_name == 'SGD':
    #     optimizer_class = torch.optim.SGD
    # elif optimizer_name == 'RMSprop':
    #     optimizer_class = torch.optim.RMSprop

    # Ejecuta la función de entrenamiento con los hiperparámetros seleccionados
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_pre_oc(df_a90, 5, 5, 35, optim.RMSprop, batch_size, lr, data_aug, None)
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_pre(df_a90, 5, 5, 35, optim.SGD, batch_size, lr, data_aug, None)
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre(df_fdg, 5, 40, optim.RMSprop, batch_size, lr, data_aug, None)
    avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre_oc(df_fdg, 5, 40, optim.SGD, batch_size, lr, data_aug, None)

    # Devuelve el valor objetivo (en este caso, la pérdida promedio de validación)
    return avg_val_loss, avg_val_accuracy, avg_val_recall

import optuna
# Supongamos que tienes un dataset llamado 'dataset'
# Realiza una llamada a Optuna para optimizar los hiperparámetros
search_space = {"lr": [1e-5,1e-4,1e-3,1e-2, 1e-1],  'data_aug' : [True,False], 'batch_size' : [32, 16, 8],
                } ##Con este sampler se hacen todas las combinaciones posibles
study = optuna.create_study(directions = ['minimize','maximize','maximize'], sampler=optuna.samplers.GridSampler(search_space))#None, NSGA por defecto para multiobjetivo, sampler=optuna.samplers.GridSampler(search_space)
print(f"Sampler is {study.sampler.__class__.__name__}")
study.optimize(objective)#, n_trials=100

# Muestra los mejores hiperparámetros encontrados
print('Best trial:')
trial = study.best_trials
for t in trial:
    print('Value: {}'.format(t.values))
    print('Params: ')
    for key, value in t.params.items():
        print('{}: {}'.format(key, value))

save_study(study, 'fdg_oc_sgd')

[I 2024-04-21 15:50:27,719] A new study created in memory with name: no-name-3dca392d-3d93-4dc8-bd23-4635a2f791ef


Sampler is GridSampler
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-21 15:51:19,252] Trial 0 finished with values: [0.6988937625288963, 0.49411112383008005, 0.328499972820282] and parameters: {'batch_size': 16, 'lr': 1e-05, 'data_aug': True}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-21 15:52:09,667] Trial 1 finished with values: [0.6876335710287094, 0.5437777934968471, 0.2632499933242798] and parameters: {'batch_size': 16, 'lr': 0.001, 'data_aug': False}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


... (52 outputs left out)

[I 2024-04-21 16:15:43,030] Trial 28 finished with values: [1.6634100440411814, 0.5296875, 0.5631250143051147] and parameters: {'batch_size': 8, 'lr': 0.01, 'data_aug': True}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-21 16:16:34,588] Trial 29 finished with values: [0.6981403949856758, 0.5271666828542948, 0.19200000166893005] and parameters: {'batch_size': 16, 'lr': 0.001, 'data_aug': True}. 


5
Best trial:
Value: [1.59480022162199, 0.5257777854800224, 0.7085000276565552]
Params: 
batch_size: 32
lr: 0.01
data_aug: False
Value: [0.6943517231941223, 0.5311111231148243, 0.5962499976158142]
Params: 
batch_size: 32
lr: 1e-05
data_aug: False
Value: [0.6857367339730263, 0.5484444611519576, 0.17125000059604645]
Params: 
batch_size: 32
lr: 0.0001
data_aug: False
Value: [112.48511765325009, 0.5328125, 0.46833333373069763]
Params: 
batch_size: 8
lr: 0.1
data_aug: False
Value: [0.6873126521706581, 0.5660555711388587, 0.40799999237060547]
Params: 
batch_size: 16
lr: 1e-05
data_aug: False


In [21]:
def save_study(study, study_name="study"):
    # Crear una carpeta para guardar el estudio si no existe
    study_dir = os.path.join(os.getcwd(), study_name)
    if not os.path.exists(study_dir):
        os.makedirs(study_dir)
    
    # Guardar los resultados del estudio en un archivo CSV
    study_trials = study.trials_dataframe()
    study_trials.to_csv(os.path.join(study_dir, "study_results.csv"), index=False)

def objective(trial):
    # Define los hiperparámetros para optimizar
    # num_epochs_freeze = trial.suggest_categorical('num_epochs_freeze', [1,3,8,10,15])
    # num_epochs_all = trial.suggest_categorical('num_epochs_all', [25,30,35,40,45,50])
    batch_size = trial.suggest_categorical('batch_size', [32, 16, 8, 4, 48, 5])
    lr = trial.suggest_categorical('lr', [1e-5,1e-4,1e-3,1e-2,1e-1])
    data_aug = trial.suggest_categorical('data_aug', [True,False])
    # ini = trial.suggest_categorical('ini', ['G', 'H', 'N', 'U', 'O', 'Z', 'ON', 'None'])
    # optimizer_name = trial.suggest_categorical('optimizer_name', ['RMSprop', 'Adam', 'SGD'])

    # # Convertir el nombre del optimizador en una clase PyTorch
    # if optimizer_name == 'Adam':
    #     optimizer_class = torch.optim.Adam
    # elif optimizer_name == 'SGD':
    #     optimizer_class = torch.optim.SGD
    # elif optimizer_name == 'RMSprop':
    #     optimizer_class = torch.optim.RMSprop

    # Ejecuta la función de entrenamiento con los hiperparámetros seleccionados
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_pre_oc(df_a90, 5, 5, 35, optim.RMSprop, batch_size, lr, data_aug, None)
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_pre(df_a90, 5, 5, 35, optim.SGD, batch_size, lr, data_aug, None)
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre(df_fdg, 5, 40, optim.RMSprop, batch_size, lr, data_aug, None)
    avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre_oc(df_fdg, 5, 40, optim.RMSprop, batch_size, lr, data_aug, None)

    # Devuelve el valor objetivo (en este caso, la pérdida promedio de validación)
    return avg_val_loss, avg_val_accuracy, avg_val_recall

import optuna
# Supongamos que tienes un dataset llamado 'dataset'
# Realiza una llamada a Optuna para optimizar los hiperparámetros
search_space = {"lr": [1e-5,1e-4,1e-3,1e-2, 1e-1],  'data_aug' : [True,False], 'batch_size' : [32, 16, 8],
                } ##Con este sampler se hacen todas las combinaciones posibles
study = optuna.create_study(directions = ['minimize','maximize','maximize'], sampler=optuna.samplers.GridSampler(search_space))#None, NSGA por defecto para multiobjetivo, sampler=optuna.samplers.GridSampler(search_space)
print(f"Sampler is {study.sampler.__class__.__name__}")
study.optimize(objective)#, n_trials=100

# Muestra los mejores hiperparámetros encontrados
print('Best trial:')
trial = study.best_trials
for t in trial:
    print('Value: {}'.format(t.values))
    print('Params: ')
    for key, value in t.params.items():
        print('{}: {}'.format(key, value))

save_study(study, 'fdg_oc_rms')

[I 2024-04-21 16:16:34,613] A new study created in memory with name: no-name-afb62a4e-c1ca-4f74-a574-fd5cea45277d


Sampler is GridSampler
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-21 16:17:33,456] Trial 0 finished with values: [1.2640945458701027, 0.5650000000000001, 0.43708333373069763] and parameters: {'batch_size': 8, 'lr': 1e-05, 'data_aug': True}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-21 16:18:24,722] Trial 1 finished with values: [1.4002187275886535, 0.5657777885347605, 0.3487499952316284] and parameters: {'batch_size': 32, 'lr': 1e-05, 'data_aug': True}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


... (52 outputs left out)

[I 2024-04-21 16:42:37,524] Trial 28 finished with values: [15185301.054955471, 0.5201666788756848, 0.413750022649765] and parameters: {'batch_size': 16, 'lr': 0.01, 'data_aug': False}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-21 16:43:28,031] Trial 29 finished with values: [1.271181233227253, 0.5671111220121384, 0.4909999966621399] and parameters: {'batch_size': 32, 'lr': 1e-05, 'data_aug': False}. 


5
Best trial:
Value: [1.3261384388804438, 0.5759444558620453, 0.5577499866485596]
Params: 
batch_size: 16
lr: 1e-05
data_aug: True
Value: [61.74938324317336, 0.4959444528818131, 0.6182500123977661]
Params: 
batch_size: 32
lr: 0.0001
data_aug: False
Value: [1.2417032329738142, 0.5743333432078362, 0.4752500057220459]
Params: 
batch_size: 16
lr: 1e-05
data_aug: False
Value: [1.4726458543537226, 0.608125, 0.5089583396911621]
Params: 
batch_size: 8
lr: 1e-05
data_aug: False
Value: [1.271181233227253, 0.5671111220121384, 0.4909999966621399]
Params: 
batch_size: 32
lr: 1e-05
data_aug: False


In [22]:
def save_study(study, study_name="study"):
    # Crear una carpeta para guardar el estudio si no existe
    study_dir = os.path.join(os.getcwd(), study_name)
    if not os.path.exists(study_dir):
        os.makedirs(study_dir)
    
    # Guardar los resultados del estudio en un archivo CSV
    study_trials = study.trials_dataframe()
    study_trials.to_csv(os.path.join(study_dir, "study_results.csv"), index=False)

def objective(trial):
    # Define los hiperparámetros para optimizar
    # num_epochs_freeze = trial.suggest_categorical('num_epochs_freeze', [1,3,8,10,15])
    # num_epochs_all = trial.suggest_categorical('num_epochs_all', [25,30,35,40,45,50])
    batch_size = trial.suggest_categorical('batch_size', [32, 16, 8, 4, 48, 5])
    lr = trial.suggest_categorical('lr', [1e-5,1e-4,1e-3,1e-2,1e-1])
    data_aug = trial.suggest_categorical('data_aug', [True,False])
    # ini = trial.suggest_categorical('ini', ['G', 'H', 'N', 'U', 'O', 'Z', 'ON', 'None'])
    # optimizer_name = trial.suggest_categorical('optimizer_name', ['RMSprop', 'Adam', 'SGD'])

    # # Convertir el nombre del optimizador en una clase PyTorch
    # if optimizer_name == 'Adam':
    #     optimizer_class = torch.optim.Adam
    # elif optimizer_name == 'SGD':
    #     optimizer_class = torch.optim.SGD
    # elif optimizer_name == 'RMSprop':
    #     optimizer_class = torch.optim.RMSprop

    # Ejecuta la función de entrenamiento con los hiperparámetros seleccionados
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_pre_oc(df_a90, 5, 5, 35, optim.RMSprop, batch_size, lr, data_aug, None)
    avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_pre(df_fdg, 5, 5, 35, optim.Adam, batch_size, lr, data_aug, None)
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre(df_fdg, 5, 40, optim.RMSprop, batch_size, lr, data_aug, None)
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre_oc(df_fdg, 5, 40, optim.Adam, batch_size, lr, data_aug, None)

    # Devuelve el valor objetivo (en este caso, la pérdida promedio de validación)
    return avg_val_loss, avg_val_accuracy, avg_val_recall

import optuna
# Supongamos que tienes un dataset llamado 'dataset'
# Realiza una llamada a Optuna para optimizar los hiperparámetros
search_space = {"lr": [1e-5,1e-4,1e-3,1e-2, 1e-1],  'data_aug' : [True,False], 'batch_size' : [32, 16, 8],
                } ##Con este sampler se hacen todas las combinaciones posibles
study = optuna.create_study(directions = ['minimize','maximize','maximize'], sampler=optuna.samplers.GridSampler(search_space))#None, NSGA por defecto para multiobjetivo, sampler=optuna.samplers.GridSampler(search_space)
print(f"Sampler is {study.sampler.__class__.__name__}")
study.optimize(objective)#, n_trials=100

# Muestra los mejores hiperparámetros encontrados
print('Best trial:')
trial = study.best_trials
for t in trial:
    print('Value: {}'.format(t.values))
    print('Params: ')
    for key, value in t.params.items():
        print('{}: {}'.format(key, value))

save_study(study, 'fdg_tl_adam')

[I 2024-04-21 16:43:28,052] A new study created in memory with name: no-name-e5b07bec-3877-455b-8778-6d49effb79c2


Sampler is GridSampler
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-21 16:44:17,072] Trial 0 finished with values: [1.0055179046094418, 0.5560000129044056, 0.3580000102519989] and parameters: {'batch_size': 32, 'lr': 0.0001, 'data_aug': True}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-21 16:45:04,986] Trial 1 finished with values: [0.9759606373310088, 0.5488333468139172, 0.36275002360343933] and parameters: {'batch_size': 32, 'lr': 0.0001, 'data_aug': False}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


... (52 outputs left out)

[I 2024-04-21 17:08:15,417] Trial 28 finished with values: [63756.44099960455, 0.5109375, 0.39000001549720764] and parameters: {'batch_size': 8, 'lr': 0.01, 'data_aug': False}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-21 17:09:04,242] Trial 29 finished with values: [254331.73921096086, 0.5147777912020685, 0.3620000183582306] and parameters: {'batch_size': 32, 'lr': 0.01, 'data_aug': True}. 


5
Best trial:
Value: [1.0055179046094418, 0.5560000129044056, 0.3580000102519989]
Params: 
batch_size: 32
lr: 0.0001
data_aug: True
Value: [0.9759606373310088, 0.5488333468139172, 0.36275002360343933]
Params: 
batch_size: 32
lr: 0.0001
data_aug: False
Value: [0.9175029178708792, 0.501875, 0.4347916543483734]
Params: 
batch_size: 8
lr: 1e-05
data_aug: True
Value: [2.43796678930521, 0.555222233235836, 0.46024999022483826]
Params: 
batch_size: 16
lr: 0.001
data_aug: False
Value: [1.0694450397789477, 0.5627222314476967, 0.3995000123977661]
Params: 
batch_size: 16
lr: 0.0001
data_aug: False
Value: [1.2186165306705514, 0.55125, 0.41750001907348633]
Params: 
batch_size: 8
lr: 0.0001
data_aug: True
Value: [1.848218308946311, 0.5290625, 0.4404166638851166]
Params: 
batch_size: 8
lr: 0.001
data_aug: True
Value: [0.7436251521110535, 0.5340000119060277, 0.34549999237060547]
Params: 
batch_size: 32
lr: 1e-05
data_aug: True
Value: [2.1142792218923567, 0.5213333445787429, 0.5282500386238098]
Params: 

In [23]:
def save_study(study, study_name="study"):
    # Crear una carpeta para guardar el estudio si no existe
    study_dir = os.path.join(os.getcwd(), study_name)
    if not os.path.exists(study_dir):
        os.makedirs(study_dir)
    
    # Guardar los resultados del estudio en un archivo CSV
    study_trials = study.trials_dataframe()
    study_trials.to_csv(os.path.join(study_dir, "study_results.csv"), index=False)

def objective(trial):
    # Define los hiperparámetros para optimizar
    # num_epochs_freeze = trial.suggest_categorical('num_epochs_freeze', [1,3,8,10,15])
    # num_epochs_all = trial.suggest_categorical('num_epochs_all', [25,30,35,40,45,50])
    batch_size = trial.suggest_categorical('batch_size', [32, 16, 8, 4, 48, 5])
    lr = trial.suggest_categorical('lr', [1e-5,1e-4,1e-3,1e-2,1e-1])
    data_aug = trial.suggest_categorical('data_aug', [True,False])
    # ini = trial.suggest_categorical('ini', ['G', 'H', 'N', 'U', 'O', 'Z', 'ON', 'None'])
    # optimizer_name = trial.suggest_categorical('optimizer_name', ['RMSprop', 'Adam', 'SGD'])

    # # Convertir el nombre del optimizador en una clase PyTorch
    # if optimizer_name == 'Adam':
    #     optimizer_class = torch.optim.Adam
    # elif optimizer_name == 'SGD':
    #     optimizer_class = torch.optim.SGD
    # elif optimizer_name == 'RMSprop':
    #     optimizer_class = torch.optim.RMSprop

    # Ejecuta la función de entrenamiento con los hiperparámetros seleccionados
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_pre_oc(df_a90, 5, 5, 35, optim.RMSprop, batch_size, lr, data_aug, None)
    avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_pre(df_fdg, 5, 5, 35, optim.SGD, batch_size, lr, data_aug, None)
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre(df_fdg, 5, 40, optim.RMSprop, batch_size, lr, data_aug, None)
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre_oc(df_fdg, 5, 40, optim.Adam, batch_size, lr, data_aug, None)

    # Devuelve el valor objetivo (en este caso, la pérdida promedio de validación)
    return avg_val_loss, avg_val_accuracy, avg_val_recall

import optuna
# Supongamos que tienes un dataset llamado 'dataset'
# Realiza una llamada a Optuna para optimizar los hiperparámetros
search_space = {"lr": [1e-5,1e-4,1e-3,1e-2, 1e-1],  'data_aug' : [True,False], 'batch_size' : [32, 16, 8],
                } ##Con este sampler se hacen todas las combinaciones posibles
study = optuna.create_study(directions = ['minimize','maximize','maximize'], sampler=optuna.samplers.GridSampler(search_space))#None, NSGA por defecto para multiobjetivo, sampler=optuna.samplers.GridSampler(search_space)
print(f"Sampler is {study.sampler.__class__.__name__}")
study.optimize(objective)#, n_trials=100

# Muestra los mejores hiperparámetros encontrados
print('Best trial:')
trial = study.best_trials
for t in trial:
    print('Value: {}'.format(t.values))
    print('Params: ')
    for key, value in t.params.items():
        print('{}: {}'.format(key, value))

save_study(study, 'fdg_tl_sgd')

[I 2024-04-21 17:09:04,265] A new study created in memory with name: no-name-d64b80ea-e874-4c33-8b55-79064e759cb0


Sampler is GridSampler
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-21 17:09:51,832] Trial 0 finished with values: [0.7063322633504867, 0.5338889020308851, 0.43174999952316284] and parameters: {'batch_size': 32, 'lr': 0.0001, 'data_aug': True}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-21 17:10:40,172] Trial 1 finished with values: [0.7224056473374366, 0.5235000114515423, 0.2887499928474426] and parameters: {'batch_size': 16, 'lr': 0.001, 'data_aug': True}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


... (52 outputs left out)

[I 2024-04-21 17:32:42,860] Trial 28 finished with values: [89.9295998352766, 0.5067222347855568, 0.42625004053115845] and parameters: {'batch_size': 16, 'lr': 0.1, 'data_aug': True}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-21 17:33:35,205] Trial 29 finished with values: [0.7749732133746147, 0.385, 0.1262499988079071] and parameters: {'batch_size': 8, 'lr': 0.0001, 'data_aug': True}. 


5
Best trial:
Value: [0.7063322633504867, 0.5338889020308851, 0.43174999952316284]
Params: 
batch_size: 32
lr: 0.0001
data_aug: True
Value: [0.7821047529578209, 0.5972222340852023, 0.42624998092651367]
Params: 
batch_size: 32
lr: 0.01
data_aug: False
Value: [0.7112615656852723, 0.5858889058977367, 0.24950000643730164]
Params: 
batch_size: 16
lr: 0.0001
data_aug: True
Value: [0.7076616188138724, 0.54625, 0.5747917294502258]
Params: 
batch_size: 8
lr: 1e-05
data_aug: True
Value: [0.6935226478427647, 0.5528125, 0.3583333492279053]
Params: 
batch_size: 8
lr: 0.0001
data_aug: False


In [24]:
def save_study(study, study_name="study"):
    # Crear una carpeta para guardar el estudio si no existe
    study_dir = os.path.join(os.getcwd(), study_name)
    if not os.path.exists(study_dir):
        os.makedirs(study_dir)
    
    # Guardar los resultados del estudio en un archivo CSV
    study_trials = study.trials_dataframe()
    study_trials.to_csv(os.path.join(study_dir, "study_results.csv"), index=False)

def objective(trial):
    # Define los hiperparámetros para optimizar
    # num_epochs_freeze = trial.suggest_categorical('num_epochs_freeze', [1,3,8,10,15])
    # num_epochs_all = trial.suggest_categorical('num_epochs_all', [25,30,35,40,45,50])
    batch_size = trial.suggest_categorical('batch_size', [32, 16, 8, 4, 48, 5])
    lr = trial.suggest_categorical('lr', [1e-5,1e-4,1e-3,1e-2,1e-1])
    data_aug = trial.suggest_categorical('data_aug', [True,False])
    # ini = trial.suggest_categorical('ini', ['G', 'H', 'N', 'U', 'O', 'Z', 'ON', 'None'])
    # optimizer_name = trial.suggest_categorical('optimizer_name', ['RMSprop', 'Adam', 'SGD'])

    # # Convertir el nombre del optimizador en una clase PyTorch
    # if optimizer_name == 'Adam':
    #     optimizer_class = torch.optim.Adam
    # elif optimizer_name == 'SGD':
    #     optimizer_class = torch.optim.SGD
    # elif optimizer_name == 'RMSprop':
    #     optimizer_class = torch.optim.RMSprop

    # Ejecuta la función de entrenamiento con los hiperparámetros seleccionados
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_pre_oc(df_a90, 5, 5, 35, optim.RMSprop, batch_size, lr, data_aug, None)
    avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_pre(df_fdg, 5, 5, 35, optim.RMSprop, batch_size, lr, data_aug, None)
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre(df_fdg, 5, 40, optim.RMSprop, batch_size, lr, data_aug, None)
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre_oc(df_fdg, 5, 40, optim.Adam, batch_size, lr, data_aug, None)

    # Devuelve el valor objetivo (en este caso, la pérdida promedio de validación)
    return avg_val_loss, avg_val_accuracy, avg_val_recall

import optuna
# Supongamos que tienes un dataset llamado 'dataset'
# Realiza una llamada a Optuna para optimizar los hiperparámetros
search_space = {"lr": [1e-5,1e-4,1e-3,1e-2, 1e-1],  'data_aug' : [True,False], 'batch_size' : [32, 16, 8],
                } ##Con este sampler se hacen todas las combinaciones posibles
study = optuna.create_study(directions = ['minimize','maximize','maximize'], sampler=optuna.samplers.GridSampler(search_space))#None, NSGA por defecto para multiobjetivo, sampler=optuna.samplers.GridSampler(search_space)
print(f"Sampler is {study.sampler.__class__.__name__}")
study.optimize(objective)#, n_trials=100

# Muestra los mejores hiperparámetros encontrados
print('Best trial:')
trial = study.best_trials
for t in trial:
    print('Value: {}'.format(t.values))
    print('Params: ')
    for key, value in t.params.items():
        print('{}: {}'.format(key, value))

save_study(study, 'fdg_tl_rms')

[I 2024-04-21 17:33:35,234] A new study created in memory with name: no-name-d53ab11d-a209-4a37-ab41-651d64600e6c


Sampler is GridSampler
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-21 17:34:25,062] Trial 0 finished with values: [34805.29412291855, 0.5051666788384319, 0.4322500228881836] and parameters: {'batch_size': 16, 'lr': 0.001, 'data_aug': True}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-21 17:35:13,292] Trial 1 finished with values: [2.4841284610895635e+17, 0.5015000117570162, 0.5115000009536743] and parameters: {'batch_size': 32, 'lr': 0.1, 'data_aug': True}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


... (52 outputs left out)

[I 2024-04-21 17:57:59,468] Trial 28 finished with values: [1.2983387777209283, 0.5264444561302662, 0.4244999885559082] and parameters: {'batch_size': 32, 'lr': 0.0001, 'data_aug': True}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-21 17:58:47,754] Trial 29 finished with values: [7.887860600636668e+18, 0.5172777909785509, 0.41575002670288086] and parameters: {'batch_size': 32, 'lr': 0.01, 'data_aug': True}. 


5
Best trial:
Value: [2.4841284610895635e+17, 0.5015000117570162, 0.5115000009536743]
Params: 
batch_size: 32
lr: 0.1
data_aug: True
Value: [0.7755758926272392, 0.5540000102669, 0.3657500147819519]
Params: 
batch_size: 32
lr: 1e-05
data_aug: True
Value: [1.2126248845458032, 0.5805000130087137, 0.4805000424385071]
Params: 
batch_size: 16
lr: 0.0001
data_aug: False
Value: [0.9567042168974875, 0.5785000112652778, 0.4727500379085541]
Params: 
batch_size: 32
lr: 0.0001
data_aug: False
Value: [489.0368014417589, 0.49261112146079533, 0.5130000114440918]
Params: 
batch_size: 32
lr: 0.001
data_aug: True
Value: [0.8151170650124551, 0.4736666753143072, 0.41575002670288086]
Params: 
batch_size: 32
lr: 1e-05
data_aug: False


In [25]:
def save_study(study, study_name="study"):
    # Crear una carpeta para guardar el estudio si no existe
    study_dir = os.path.join(os.getcwd(), study_name)
    if not os.path.exists(study_dir):
        os.makedirs(study_dir)
    
    # Guardar los resultados del estudio en un archivo CSV
    study_trials = study.trials_dataframe()
    study_trials.to_csv(os.path.join(study_dir, "study_results.csv"), index=False)

def objective(trial):
    # Define los hiperparámetros para optimizar
    # num_epochs_freeze = trial.suggest_categorical('num_epochs_freeze', [1,3,8,10,15])
    # num_epochs_all = trial.suggest_categorical('num_epochs_all', [25,30,35,40,45,50])
    batch_size = trial.suggest_categorical('batch_size', [32, 16, 8, 4, 48, 5])
    lr = trial.suggest_categorical('lr', [1e-5,1e-4,1e-3,1e-2,1e-1])
    data_aug = trial.suggest_categorical('data_aug', [True,False])
    # ini = trial.suggest_categorical('ini', ['G', 'H', 'N', 'U', 'O', 'Z', 'ON', 'None'])
    # optimizer_name = trial.suggest_categorical('optimizer_name', ['RMSprop', 'Adam', 'SGD'])

    # # Convertir el nombre del optimizador en una clase PyTorch
    # if optimizer_name == 'Adam':
    #     optimizer_class = torch.optim.Adam
    # elif optimizer_name == 'SGD':
    #     optimizer_class = torch.optim.SGD
    # elif optimizer_name == 'RMSprop':
    #     optimizer_class = torch.optim.RMSprop

    # Ejecuta la función de entrenamiento con los hiperparámetros seleccionados
    avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_pre_oc(df_fdg, 5, 5, 35, optim.Adam, batch_size, lr, data_aug, None)
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_pre(df_fdg, 5, 5, 35, optim.Adam, batch_size, lr, data_aug, None)
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre(df_fdg, 5, 40, optim.RMSprop, batch_size, lr, data_aug, None)
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre_oc(df_fdg, 5, 40, optim.Adam, batch_size, lr, data_aug, None)

    # Devuelve el valor objetivo (en este caso, la pérdida promedio de validación)
    return avg_val_loss, avg_val_accuracy, avg_val_recall

import optuna
# Supongamos que tienes un dataset llamado 'dataset'
# Realiza una llamada a Optuna para optimizar los hiperparámetros
search_space = {"lr": [1e-5,1e-4,1e-3,1e-2, 1e-1],  'data_aug' : [True,False], 'batch_size' : [32, 16, 8],
                } ##Con este sampler se hacen todas las combinaciones posibles
study = optuna.create_study(directions = ['minimize','maximize','maximize'], sampler=optuna.samplers.GridSampler(search_space))#None, NSGA por defecto para multiobjetivo, sampler=optuna.samplers.GridSampler(search_space)
print(f"Sampler is {study.sampler.__class__.__name__}")
study.optimize(objective)#, n_trials=100

# Muestra los mejores hiperparámetros encontrados
print('Best trial:')
trial = study.best_trials
for t in trial:
    print('Value: {}'.format(t.values))
    print('Params: ')
    for key, value in t.params.items():
        print('{}: {}'.format(key, value))

save_study(study, 'fdg_tl_oc_adam')

[I 2024-04-21 17:58:47,779] A new study created in memory with name: no-name-8a68f1c8-f772-4db3-b8ea-762ade920cc5


Sampler is GridSampler
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-21 17:59:42,735] Trial 0 finished with values: [15.449646294978304, 0.5171875, 0.3385416865348816] and parameters: {'batch_size': 8, 'lr': 0.1, 'data_aug': False}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-21 18:00:37,602] Trial 1 finished with values: [2.166074887952709, 0.5721875, 0.49291667342185974] and parameters: {'batch_size': 8, 'lr': 0.01, 'data_aug': False}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


... (52 outputs left out)

[I 2024-04-21 18:23:32,571] Trial 28 finished with values: [0.9978302021324634, 0.5032222294807435, 0.2017499953508377] and parameters: {'batch_size': 16, 'lr': 0.001, 'data_aug': False}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-21 18:24:27,478] Trial 29 finished with values: [0.8600546284765006, 0.47375, 0.21124999225139618] and parameters: {'batch_size': 8, 'lr': 0.0001, 'data_aug': False}. 


5
Best trial:
Value: [2.166074887952709, 0.5721875, 0.49291667342185974]
Params: 
batch_size: 8
lr: 0.01
data_aug: False
Value: [0.7600600990653039, 0.5213333441317081, 0.5547499656677246]
Params: 
batch_size: 32
lr: 0.0001
data_aug: False
Value: [0.7924869319796563, 0.4791666732728482, 0.6662500500679016]
Params: 
batch_size: 32
lr: 1e-05
data_aug: True
Value: [0.739865079820156, 0.556166674643755, 0.5122500658035278]
Params: 
batch_size: 16
lr: 0.0001
data_aug: False
Value: [0.6988363319635391, 0.5618333458900452, 0.4257500171661377]
Params: 
batch_size: 16
lr: 1e-05
data_aug: True
Value: [2.6373375564813615, 0.5287222319096327, 0.5694999694824219]
Params: 
batch_size: 16
lr: 0.01
data_aug: False
Value: [0.7215266031399369, 0.555, 0.5114583373069763]
Params: 
batch_size: 8
lr: 1e-05
data_aug: True
Value: [7239.1511029815665, 0.4977777869999408, 0.5884999632835388]
Params: 
batch_size: 32
lr: 0.1
data_aug: False
Value: [11.152806157171728, 0.5384444552659988, 0.5350000262260437]
Param

In [26]:
def save_study(study, study_name="study"):
    # Crear una carpeta para guardar el estudio si no existe
    study_dir = os.path.join(os.getcwd(), study_name)
    if not os.path.exists(study_dir):
        os.makedirs(study_dir)
    
    # Guardar los resultados del estudio en un archivo CSV
    study_trials = study.trials_dataframe()
    study_trials.to_csv(os.path.join(study_dir, "study_results.csv"), index=False)

def objective(trial):
    # Define los hiperparámetros para optimizar
    # num_epochs_freeze = trial.suggest_categorical('num_epochs_freeze', [1,3,8,10,15])
    # num_epochs_all = trial.suggest_categorical('num_epochs_all', [25,30,35,40,45,50])
    batch_size = trial.suggest_categorical('batch_size', [32, 16, 8, 4, 48, 5])
    lr = trial.suggest_categorical('lr', [1e-5,1e-4,1e-3,1e-2,1e-1])
    data_aug = trial.suggest_categorical('data_aug', [True,False])
    # ini = trial.suggest_categorical('ini', ['G', 'H', 'N', 'U', 'O', 'Z', 'ON', 'None'])
    # optimizer_name = trial.suggest_categorical('optimizer_name', ['RMSprop', 'Adam', 'SGD'])

    # # Convertir el nombre del optimizador en una clase PyTorch
    # if optimizer_name == 'Adam':
    #     optimizer_class = torch.optim.Adam
    # elif optimizer_name == 'SGD':
    #     optimizer_class = torch.optim.SGD
    # elif optimizer_name == 'RMSprop':
    #     optimizer_class = torch.optim.RMSprop

    # Ejecuta la función de entrenamiento con los hiperparámetros seleccionados
    avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_pre_oc(df_fdg, 5, 5, 35, optim.SGD, batch_size, lr, data_aug, None)
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_pre(df_fdg, 5, 5, 35, optim.Adam, batch_size, lr, data_aug, None)
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre(df_fdg, 5, 40, optim.RMSprop, batch_size, lr, data_aug, None)
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre_oc(df_fdg, 5, 40, optim.Adam, batch_size, lr, data_aug, None)

    # Devuelve el valor objetivo (en este caso, la pérdida promedio de validación)
    return avg_val_loss, avg_val_accuracy, avg_val_recall

import optuna
# Supongamos que tienes un dataset llamado 'dataset'
# Realiza una llamada a Optuna para optimizar los hiperparámetros
search_space = {"lr": [1e-5,1e-4,1e-3,1e-2, 1e-1],  'data_aug' : [True,False], 'batch_size' : [32, 16, 8],
                } ##Con este sampler se hacen todas las combinaciones posibles
study = optuna.create_study(directions = ['minimize','maximize','maximize'], sampler=optuna.samplers.GridSampler(search_space))#None, NSGA por defecto para multiobjetivo, sampler=optuna.samplers.GridSampler(search_space)
print(f"Sampler is {study.sampler.__class__.__name__}")
study.optimize(objective)#, n_trials=100

# Muestra los mejores hiperparámetros encontrados
print('Best trial:')
trial = study.best_trials
for t in trial:
    print('Value: {}'.format(t.values))
    print('Params: ')
    for key, value in t.params.items():
        print('{}: {}'.format(key, value))

save_study(study, 'fdg_tl_oc_sgd')

[I 2024-04-21 18:24:27,501] A new study created in memory with name: no-name-4eadc07d-4cdb-41b1-b945-fe4cd2387bcb


Sampler is GridSampler
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-21 18:25:20,595] Trial 0 finished with values: [0.7643042137473822, 0.478125, 0.3960416913032532] and parameters: {'batch_size': 8, 'lr': 0.001, 'data_aug': False}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-21 18:26:08,793] Trial 1 finished with values: [7.9111277568340315, 0.5256111218035221, 0.4749999940395355] and parameters: {'batch_size': 16, 'lr': 0.1, 'data_aug': False}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


... (52 outputs left out)

[I 2024-04-21 18:48:32,267] Trial 28 finished with values: [1.027058223336935, 0.47531249999999997, 0.3212500214576721] and parameters: {'batch_size': 8, 'lr': 0.01, 'data_aug': False}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-21 18:49:20,356] Trial 29 finished with values: [0.7197875386476517, 0.5036111198365688, 0.42000001668930054] and parameters: {'batch_size': 16, 'lr': 0.0001, 'data_aug': False}. 


5
Best trial:
Value: [7.9111277568340315, 0.5256111218035221, 0.4749999940395355]
Params: 
batch_size: 16
lr: 0.1
data_aug: False
Value: [0.7816167273372412, 0.47687500000000005, 0.46895831823349]
Params: 
batch_size: 8
lr: 1e-05
data_aug: True
Value: [0.836649019420147, 0.44061111666262154, 0.8224999308586121]
Params: 
batch_size: 16
lr: 1e-05
data_aug: False
Value: [0.7299839070439338, 0.4749444556981325, 0.49175000190734863]
Params: 
batch_size: 32
lr: 1e-05
data_aug: True
Value: [1.3718017826974394, 0.5336111206561327, 0.4309999942779541]
Params: 
batch_size: 32
lr: 0.1
data_aug: False
Value: [5.710958990012878, 0.5393749999999999, 0.40687504410743713]
Params: 
batch_size: 8
lr: 0.1
data_aug: True
Value: [0.7892505252361297, 0.5246666768193246, 0.4699999690055847]
Params: 
batch_size: 32
lr: 1e-05
data_aug: False
Value: [87.69185337662697, 0.5553333442658186, 0.4192500114440918]
Params: 
batch_size: 32
lr: 0.1
data_aug: True
Value: [0.8154386886954308, 0.45906250000000004, 0.624583

In [27]:
def save_study(study, study_name="study"):
    # Crear una carpeta para guardar el estudio si no existe
    study_dir = os.path.join(os.getcwd(), study_name)
    if not os.path.exists(study_dir):
        os.makedirs(study_dir)
    
    # Guardar los resultados del estudio en un archivo CSV
    study_trials = study.trials_dataframe()
    study_trials.to_csv(os.path.join(study_dir, "study_results.csv"), index=False)

def objective(trial):
    # Define los hiperparámetros para optimizar
    # num_epochs_freeze = trial.suggest_categorical('num_epochs_freeze', [1,3,8,10,15])
    # num_epochs_all = trial.suggest_categorical('num_epochs_all', [25,30,35,40,45,50])
    batch_size = trial.suggest_categorical('batch_size', [32, 16, 8, 4, 48, 5])
    lr = trial.suggest_categorical('lr', [1e-5,1e-4,1e-3,1e-2,1e-1])
    data_aug = trial.suggest_categorical('data_aug', [True,False])
    # ini = trial.suggest_categorical('ini', ['G', 'H', 'N', 'U', 'O', 'Z', 'ON', 'None'])
    # optimizer_name = trial.suggest_categorical('optimizer_name', ['RMSprop', 'Adam', 'SGD'])

    # # Convertir el nombre del optimizador en una clase PyTorch
    # if optimizer_name == 'Adam':
    #     optimizer_class = torch.optim.Adam
    # elif optimizer_name == 'SGD':
    #     optimizer_class = torch.optim.SGD
    # elif optimizer_name == 'RMSprop':
    #     optimizer_class = torch.optim.RMSprop

    # Ejecuta la función de entrenamiento con los hiperparámetros seleccionados
    avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_pre_oc(df_fdg, 5, 5, 35, optim.RMSprop, batch_size, lr, data_aug, None)
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_pre(df_fdg, 5, 5, 35, optim.Adam, batch_size, lr, data_aug, None)
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre(df_fdg, 5, 40, optim.RMSprop, batch_size, lr, data_aug, None)
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre_oc(df_fdg, 5, 40, optim.Adam, batch_size, lr, data_aug, None)

    # Devuelve el valor objetivo (en este caso, la pérdida promedio de validación)
    return avg_val_loss, avg_val_accuracy, avg_val_recall

import optuna
# Supongamos que tienes un dataset llamado 'dataset'
# Realiza una llamada a Optuna para optimizar los hiperparámetros
search_space = {"lr": [1e-5,1e-4,1e-3,1e-2, 1e-1],  'data_aug' : [True,False], 'batch_size' : [32, 16, 8],
                } ##Con este sampler se hacen todas las combinaciones posibles
study = optuna.create_study(directions = ['minimize','maximize','maximize'], sampler=optuna.samplers.GridSampler(search_space))#None, NSGA por defecto para multiobjetivo, sampler=optuna.samplers.GridSampler(search_space)
print(f"Sampler is {study.sampler.__class__.__name__}")
study.optimize(objective)#, n_trials=100

# Muestra los mejores hiperparámetros encontrados
print('Best trial:')
trial = study.best_trials
for t in trial:
    print('Value: {}'.format(t.values))
    print('Params: ')
    for key, value in t.params.items():
        print('{}: {}'.format(key, value))

save_study(study, 'fdg_tl_oc_rms')

[I 2024-04-21 18:49:20,384] A new study created in memory with name: no-name-966589f8-d0bb-43e3-974b-aa2d0882765a


Sampler is GridSampler
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-21 18:50:10,962] Trial 0 finished with values: [1.498955674469471, 0.502833341434598, 0.33674997091293335] and parameters: {'batch_size': 16, 'lr': 0.0001, 'data_aug': True}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-21 18:51:01,415] Trial 1 finished with values: [0.7539564767479897, 0.5308333417028189, 0.3997500240802765] and parameters: {'batch_size': 16, 'lr': 1e-05, 'data_aug': True}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


... (52 outputs left out)

[I 2024-04-21 19:14:07,189] Trial 28 finished with values: [1.4230629280209541, 0.5310000097751617, 0.47874999046325684] and parameters: {'batch_size': 32, 'lr': 0.0001, 'data_aug': False}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-21 19:14:56,700] Trial 29 finished with values: [36.90799644589424, 0.5286666756123305, 0.546750009059906] and parameters: {'batch_size': 16, 'lr': 0.001, 'data_aug': False}. 


5
Best trial:
Value: [0.7539564767479897, 0.5308333417028189, 0.3997500240802765]
Params: 
batch_size: 16
lr: 1e-05
data_aug: True
Value: [22738.98451487559, 0.5493750000000001, 0.35083332657814026]
Params: 
batch_size: 8
lr: 0.01
data_aug: False
Value: [559.7125731208921, 0.55111112549901, 0.30550000071525574]
Params: 
batch_size: 32
lr: 0.001
data_aug: False
Value: [0.7786585476994514, 0.5278889016062022, 0.41725000739097595]
Params: 
batch_size: 32
lr: 1e-05
data_aug: True
Value: [12874970415.683704, 0.5409375, 0.5052083730697632]
Params: 
batch_size: 8
lr: 0.1
data_aug: False
Value: [2.7162536645477484, 0.5328125, 0.4739583134651184]
Params: 
batch_size: 8
lr: 0.001
data_aug: True
Value: [3387907.1092844084, 0.49850001033395525, 0.6000000238418579]
Params: 
batch_size: 32
lr: 0.01
data_aug: False
Value: [0.7734366858005524, 0.4960555629432202, 0.4652499556541443]
Params: 
batch_size: 32
lr: 1e-05
data_aug: False
Value: [1.4230629280209541, 0.5310000097751617, 0.47874999046325684]
P

## CT

Ahora se muestran todas las pruebas hechas para CT.

In [28]:
def save_study(study, study_name="study"):
    # Crear una carpeta para guardar el estudio si no existe
    study_dir = os.path.join(os.getcwd(), study_name)
    if not os.path.exists(study_dir):
        os.makedirs(study_dir)
    
    # Guardar los resultados del estudio en un archivo CSV
    study_trials = study.trials_dataframe()
    study_trials.to_csv(os.path.join(study_dir, "study_results.csv"), index=False)

def objective(trial):
    # Define los hiperparámetros para optimizar
    # num_epochs_freeze = trial.suggest_categorical('num_epochs_freeze', [1,3,8,10,15])
    # num_epochs_all = trial.suggest_categorical('num_epochs_all', [25,30,35,40,45,50])
    batch_size = trial.suggest_categorical('batch_size', [32, 16, 8, 4, 48, 5])
    lr = trial.suggest_categorical('lr', [1e-5,1e-4,1e-3,1e-2,1e-1])
    data_aug = trial.suggest_categorical('data_aug', [True,False])
    # ini = trial.suggest_categorical('ini', ['G', 'H', 'N', 'U', 'O', 'Z', 'ON', 'None'])
    # optimizer_name = trial.suggest_categorical('optimizer_name', ['RMSprop', 'Adam', 'SGD'])

    # # Convertir el nombre del optimizador en una clase PyTorch
    # if optimizer_name == 'Adam':
    #     optimizer_class = torch.optim.Adam
    # elif optimizer_name == 'SGD':
    #     optimizer_class = torch.optim.SGD
    # elif optimizer_name == 'RMSprop':
    #     optimizer_class = torch.optim.RMSprop

    # Ejecuta la función de entrenamiento con los hiperparámetros seleccionados
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_pre_oc(df_a90, 5, 5, 35, optim.RMSprop, batch_size, lr, data_aug, None)
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_pre(df_a90, 5, 5, 35, optim.SGD, batch_size, lr, data_aug, None)
    avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre(df_ct, 5, 40, optim.Adam, batch_size, lr, data_aug, None)
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre_oc(df_a90, 5, 40, optim.RMSprop, batch_size, lr, data_aug, None)

    # Devuelve el valor objetivo (en este caso, la pérdida promedio de validación)
    return avg_val_loss, avg_val_accuracy, avg_val_recall

import optuna
# Supongamos que tienes un dataset llamado 'dataset'
# Realiza una llamada a Optuna para optimizar los hiperparámetros
search_space = {"lr": [1e-5,1e-4,1e-3,1e-2, 1e-1],  'data_aug' : [True,False], 'batch_size' : [32, 16, 8],
                } ##Con este sampler se hacen todas las combinaciones posibles
study = optuna.create_study(directions = ['minimize','maximize','maximize'], sampler=optuna.samplers.GridSampler(search_space))#None, NSGA por defecto para multiobjetivo, sampler=optuna.samplers.GridSampler(search_space)
print(f"Sampler is {study.sampler.__class__.__name__}")
study.optimize(objective)#, n_trials=100

# Muestra los mejores hiperparámetros encontrados
print('Best trial:')
trial = study.best_trials
for t in trial:
    print('Value: {}'.format(t.values))
    print('Params: ')
    for key, value in t.params.items():
        print('{}: {}'.format(key, value))

save_study(study, 'ct_adam')

[I 2024-04-21 19:14:56,729] A new study created in memory with name: no-name-795494e0-6000-4a92-be9e-b1f473d79473


Sampler is GridSampler
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-21 19:16:32,100] Trial 0 finished with values: [2058525.8576228633, 0.5065789580345154, 0.5908888578414917] and parameters: {'batch_size': 32, 'lr': 0.01, 'data_aug': False}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-21 19:18:24,893] Trial 1 finished with values: [0.8622508381307125, 0.5179861141741275, 0.37186112999916077] and parameters: {'batch_size': 8, 'lr': 1e-05, 'data_aug': False}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


... (52 outputs left out)

[I 2024-04-21 20:04:40,771] Trial 28 finished with values: [0.7334678623080253, 0.5244736894965171, 0.5212777853012085] and parameters: {'batch_size': 32, 'lr': 1e-05, 'data_aug': False}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-21 20:06:33,231] Trial 29 finished with values: [543.570304111186, 0.4590277810891469, 0.47761112451553345] and parameters: {'batch_size': 8, 'lr': 0.01, 'data_aug': False}. 


5
Best trial:
Value: [4.52698566099008, 0.5216666704416275, 0.5400555729866028]
Params: 
batch_size: 8
lr: 0.001
data_aug: True
Value: [2.8208384898304937, 0.5213157996535301, 0.6485555768013]
Params: 
batch_size: 32
lr: 0.001
data_aug: True
Value: [0.802523006647825, 0.4089583376795053, 0.5218055248260498]
Params: 
batch_size: 16
lr: 1e-05
data_aug: True
Value: [2.2772675823161626, 0.5038541722297669, 0.5268749594688416]
Params: 
batch_size: 16
lr: 0.001
data_aug: True
Value: [0.7066164538264273, 0.510789480060339, 0.4586111009120941]
Params: 
batch_size: 32
lr: 1e-05
data_aug: True
Value: [0.7334678623080253, 0.5244736894965171, 0.5212777853012085]
Params: 
batch_size: 32
lr: 1e-05
data_aug: False


In [29]:
def save_study(study, study_name="study"):
    # Crear una carpeta para guardar el estudio si no existe
    study_dir = os.path.join(os.getcwd(), study_name)
    if not os.path.exists(study_dir):
        os.makedirs(study_dir)
    
    # Guardar los resultados del estudio en un archivo CSV
    study_trials = study.trials_dataframe()
    study_trials.to_csv(os.path.join(study_dir, "study_results.csv"), index=False)

def objective(trial):
    # Define los hiperparámetros para optimizar
    # num_epochs_freeze = trial.suggest_categorical('num_epochs_freeze', [1,3,8,10,15])
    # num_epochs_all = trial.suggest_categorical('num_epochs_all', [25,30,35,40,45,50])
    batch_size = trial.suggest_categorical('batch_size', [32, 16, 8, 4, 48, 5])
    lr = trial.suggest_categorical('lr', [1e-5,1e-4,1e-3,1e-2,1e-1])
    data_aug = trial.suggest_categorical('data_aug', [True,False])
    # ini = trial.suggest_categorical('ini', ['G', 'H', 'N', 'U', 'O', 'Z', 'ON', 'None'])
    # optimizer_name = trial.suggest_categorical('optimizer_name', ['RMSprop', 'Adam', 'SGD'])

    # # Convertir el nombre del optimizador en una clase PyTorch
    # if optimizer_name == 'Adam':
    #     optimizer_class = torch.optim.Adam
    # elif optimizer_name == 'SGD':
    #     optimizer_class = torch.optim.SGD
    # elif optimizer_name == 'RMSprop':
    #     optimizer_class = torch.optim.RMSprop

    # Ejecuta la función de entrenamiento con los hiperparámetros seleccionados
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_pre_oc(df_a90, 5, 5, 35, optim.RMSprop, batch_size, lr, data_aug, None)
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_pre(df_a90, 5, 5, 35, optim.SGD, batch_size, lr, data_aug, None)
    avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre(df_ct, 5, 40, optim.SGD, batch_size, lr, data_aug, None)
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre_oc(df_a90, 5, 40, optim.RMSprop, batch_size, lr, data_aug, None)

    # Devuelve el valor objetivo (en este caso, la pérdida promedio de validación)
    return avg_val_loss, avg_val_accuracy, avg_val_recall

import optuna
# Supongamos que tienes un dataset llamado 'dataset'
# Realiza una llamada a Optuna para optimizar los hiperparámetros
search_space = {"lr": [1e-5,1e-4,1e-3,1e-2, 1e-1],  'data_aug' : [True,False], 'batch_size' : [32, 16, 8],
                } ##Con este sampler se hacen todas las combinaciones posibles
study = optuna.create_study(directions = ['minimize','maximize','maximize'], sampler=optuna.samplers.GridSampler(search_space))#None, NSGA por defecto para multiobjetivo, sampler=optuna.samplers.GridSampler(search_space)
print(f"Sampler is {study.sampler.__class__.__name__}")
study.optimize(objective)#, n_trials=100

# Muestra los mejores hiperparámetros encontrados
print('Best trial:')
trial = study.best_trials
for t in trial:
    print('Value: {}'.format(t.values))
    print('Params: ')
    for key, value in t.params.items():
        print('{}: {}'.format(key, value))

save_study(study, 'ct_sgd')

[I 2024-04-21 20:06:33,259] A new study created in memory with name: no-name-85e848d9-f476-46b9-aef8-271e040f7e55


Sampler is GridSampler
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-21 20:08:11,228] Trial 0 finished with values: [3.489165517147631, 0.45723958909511564, 0.3889235854148865] and parameters: {'batch_size': 16, 'lr': 0.1, 'data_aug': True}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-21 20:09:43,769] Trial 1 finished with values: [0.6977647531032563, 0.4857894836366176, 0.7344444394111633] and parameters: {'batch_size': 32, 'lr': 0.001, 'data_aug': False}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


... (52 outputs left out)

[I 2024-04-21 20:54:28,247] Trial 28 finished with values: [5.561033083864022, 0.466250005364418, 0.4018055498600006] and parameters: {'batch_size': 16, 'lr': 0.1, 'data_aug': False}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-21 20:56:04,772] Trial 29 finished with values: [0.6950454431772233, 0.5000000052154063, 0.7164235711097717] and parameters: {'batch_size': 16, 'lr': 0.001, 'data_aug': False}. 


5
Best trial:
Value: [0.7451625749468803, 0.5089473803341388, 0.6792222857475281]
Params: 
batch_size: 32
lr: 0.01
data_aug: True
Value: [0.7333389759063722, 0.5245833362142245, 0.643916666507721]
Params: 
batch_size: 8
lr: 1e-05
data_aug: False
Value: [0.6855990905563037, 0.5285416712363561, 0.22441668808460236]
Params: 
batch_size: 8
lr: 1e-05
data_aug: True
Value: [0.6974948011338712, 0.5128645886480807, 0.6093403100967407]
Params: 
batch_size: 16
lr: 0.0001
data_aug: False
Value: [0.6941082584857939, 0.4986842192709446, 0.7988888621330261]
Params: 
batch_size: 32
lr: 0.001
data_aug: True
Value: [0.6950454431772233, 0.5000000052154063, 0.7164235711097717]
Params: 
batch_size: 16
lr: 0.001
data_aug: False


In [30]:
def save_study(study, study_name="study"):
    # Crear una carpeta para guardar el estudio si no existe
    study_dir = os.path.join(os.getcwd(), study_name)
    if not os.path.exists(study_dir):
        os.makedirs(study_dir)
    
    # Guardar los resultados del estudio en un archivo CSV
    study_trials = study.trials_dataframe()
    study_trials.to_csv(os.path.join(study_dir, "study_results.csv"), index=False)

def objective(trial):
    # Define los hiperparámetros para optimizar
    # num_epochs_freeze = trial.suggest_categorical('num_epochs_freeze', [1,3,8,10,15])
    # num_epochs_all = trial.suggest_categorical('num_epochs_all', [25,30,35,40,45,50])
    batch_size = trial.suggest_categorical('batch_size', [32, 16, 8, 4, 48, 5])
    lr = trial.suggest_categorical('lr', [1e-5,1e-4,1e-3,1e-2,1e-1])
    data_aug = trial.suggest_categorical('data_aug', [True,False])
    # ini = trial.suggest_categorical('ini', ['G', 'H', 'N', 'U', 'O', 'Z', 'ON', 'None'])
    # optimizer_name = trial.suggest_categorical('optimizer_name', ['RMSprop', 'Adam', 'SGD'])

    # # Convertir el nombre del optimizador en una clase PyTorch
    # if optimizer_name == 'Adam':
    #     optimizer_class = torch.optim.Adam
    # elif optimizer_name == 'SGD':
    #     optimizer_class = torch.optim.SGD
    # elif optimizer_name == 'RMSprop':
    #     optimizer_class = torch.optim.RMSprop

    # Ejecuta la función de entrenamiento con los hiperparámetros seleccionados
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_pre_oc(df_a90, 5, 5, 35, optim.RMSprop, batch_size, lr, data_aug, None)
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_pre(df_a90, 5, 5, 35, optim.SGD, batch_size, lr, data_aug, None)
    avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre(df_ct, 5, 40, optim.RMSprop, batch_size, lr, data_aug, None)
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre_oc(df_a90, 5, 40, optim.RMSprop, batch_size, lr, data_aug, None)

    # Devuelve el valor objetivo (en este caso, la pérdida promedio de validación)
    return avg_val_loss, avg_val_accuracy, avg_val_recall

import optuna
# Supongamos que tienes un dataset llamado 'dataset'
# Realiza una llamada a Optuna para optimizar los hiperparámetros
search_space = {"lr": [1e-5,1e-4,1e-3,1e-2, 1e-1],  'data_aug' : [True,False], 'batch_size' : [32, 16, 8],
                } ##Con este sampler se hacen todas las combinaciones posibles
study = optuna.create_study(directions = ['minimize','maximize','maximize'], sampler=optuna.samplers.GridSampler(search_space))#None, NSGA por defecto para multiobjetivo, sampler=optuna.samplers.GridSampler(search_space)
print(f"Sampler is {study.sampler.__class__.__name__}")
study.optimize(objective)#, n_trials=100

# Muestra los mejores hiperparámetros encontrados
print('Best trial:')
trial = study.best_trials
for t in trial:
    print('Value: {}'.format(t.values))
    print('Params: ')
    for key, value in t.params.items():
        print('{}: {}'.format(key, value))

save_study(study, 'ct_rms')

[I 2024-04-21 20:56:04,801] A new study created in memory with name: no-name-f91a79db-a6de-412e-8a38-8e784fd3cc97


Sampler is GridSampler
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-21 20:57:40,543] Trial 0 finished with values: [106383.49456211836, 0.47657895758748053, 0.5834444761276245] and parameters: {'batch_size': 32, 'lr': 0.001, 'data_aug': True}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-21 20:59:14,604] Trial 1 finished with values: [1.2302261370420458, 0.5213157904148101, 0.3837222158908844] and parameters: {'batch_size': 32, 'lr': 0.0001, 'data_aug': False}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


... (52 outputs left out)

[I 2024-04-21 21:45:25,825] Trial 28 finished with values: [2967628344.180289, 0.5063157998025417, 0.5915000438690186] and parameters: {'batch_size': 32, 'lr': 0.01, 'data_aug': True}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-21 21:47:01,342] Trial 29 finished with values: [1.3115391221642494, 0.5344736889004706, 0.47583332657814026] and parameters: {'batch_size': 32, 'lr': 0.0001, 'data_aug': True}. 


5
Best trial:
Value: [106383.49456211836, 0.47657895758748053, 0.5834444761276245]
Params: 
batch_size: 32
lr: 0.001
data_aug: True
Value: [0.7926968058943747, 0.5315789520740509, 0.3563889265060425]
Params: 
batch_size: 32
lr: 1e-05
data_aug: True
Value: [662349899.6227921, 0.4915789575874806, 0.5227777361869812]
Params: 
batch_size: 32
lr: 0.1
data_aug: False
Value: [0.92082239245375, 0.5443055592974028, 0.4318055212497711]
Params: 
batch_size: 8
lr: 1e-05
data_aug: False
Value: [0.8239507973194122, 0.5202631613612175, 0.3878333270549774]
Params: 
batch_size: 32
lr: 1e-05
data_aug: False
Value: [75807.80012716829, 0.4659375058859586, 0.5408680438995361]
Params: 
batch_size: 16
lr: 0.1
data_aug: True
Value: [4.896746238267127, 0.4765972259143988, 0.5017500519752502]
Params: 
batch_size: 8
lr: 0.001
data_aug: True
Value: [3.947797806660334, 0.452152781188488, 0.49016666412353516]
Params: 
batch_size: 8
lr: 0.001
data_aug: False
Value: [100685.69522650927, 0.4802631652355195, 0.53350001

In [31]:
def save_study(study, study_name="study"):
    # Crear una carpeta para guardar el estudio si no existe
    study_dir = os.path.join(os.getcwd(), study_name)
    if not os.path.exists(study_dir):
        os.makedirs(study_dir)
    
    # Guardar los resultados del estudio en un archivo CSV
    study_trials = study.trials_dataframe()
    study_trials.to_csv(os.path.join(study_dir, "study_results.csv"), index=False)

def objective(trial):
    # Define los hiperparámetros para optimizar
    # num_epochs_freeze = trial.suggest_categorical('num_epochs_freeze', [1,3,8,10,15])
    # num_epochs_all = trial.suggest_categorical('num_epochs_all', [25,30,35,40,45,50])
    batch_size = trial.suggest_categorical('batch_size', [32, 16, 8, 4, 48, 5])
    lr = trial.suggest_categorical('lr', [1e-5,1e-4,1e-3,1e-2,1e-1])
    data_aug = trial.suggest_categorical('data_aug', [True,False])
    # ini = trial.suggest_categorical('ini', ['G', 'H', 'N', 'U', 'O', 'Z', 'ON', 'None'])
    # optimizer_name = trial.suggest_categorical('optimizer_name', ['RMSprop', 'Adam', 'SGD'])

    # # Convertir el nombre del optimizador en una clase PyTorch
    # if optimizer_name == 'Adam':
    #     optimizer_class = torch.optim.Adam
    # elif optimizer_name == 'SGD':
    #     optimizer_class = torch.optim.SGD
    # elif optimizer_name == 'RMSprop':
    #     optimizer_class = torch.optim.RMSprop

    # Ejecuta la función de entrenamiento con los hiperparámetros seleccionados
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_pre_oc(df_a90, 5, 5, 35, optim.RMSprop, batch_size, lr, data_aug, None)
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_pre(df_a90, 5, 5, 35, optim.SGD, batch_size, lr, data_aug, None)
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre(df_ct, 5, 40, optim.SGD, batch_size, lr, data_aug, None)
    avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre_oc(df_ct, 5, 40, optim.Adam, batch_size, lr, data_aug, None)

    # Devuelve el valor objetivo (en este caso, la pérdida promedio de validación)
    return avg_val_loss, avg_val_accuracy, avg_val_recall

import optuna
# Supongamos que tienes un dataset llamado 'dataset'
# Realiza una llamada a Optuna para optimizar los hiperparámetros
search_space = {"lr": [1e-5,1e-4,1e-3,1e-2, 1e-1],  'data_aug' : [True,False], 'batch_size' : [32, 16, 8],
                } ##Con este sampler se hacen todas las combinaciones posibles
study = optuna.create_study(directions = ['minimize','maximize','maximize'], sampler=optuna.samplers.GridSampler(search_space))#None, NSGA por defecto para multiobjetivo, sampler=optuna.samplers.GridSampler(search_space)
print(f"Sampler is {study.sampler.__class__.__name__}")
study.optimize(objective)#, n_trials=100

# Muestra los mejores hiperparámetros encontrados
print('Best trial:')
trial = study.best_trials
for t in trial:
    print('Value: {}'.format(t.values))
    print('Params: ')
    for key, value in t.params.items():
        print('{}: {}'.format(key, value))

save_study(study, 'ct_oc_adam')

[I 2024-04-21 21:47:01,365] A new study created in memory with name: no-name-47243e3d-9c36-4994-be1b-e90576f128b2


Sampler is GridSampler
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-21 21:48:37,751] Trial 0 finished with values: [30290.659214884043, 0.49710527524352077, 0.5864999890327454] and parameters: {'batch_size': 32, 'lr': 0.1, 'data_aug': True}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-21 21:50:19,376] Trial 1 finished with values: [1.3184034173563124, 0.4822395888715983, 0.4782639145851135] and parameters: {'batch_size': 16, 'lr': 0.0001, 'data_aug': True}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


... (52 outputs left out)

[I 2024-04-21 22:37:15,150] Trial 28 finished with values: [1.4920558023794244, 0.47659722536802296, 0.4568888545036316] and parameters: {'batch_size': 8, 'lr': 0.0001, 'data_aug': False}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-21 22:38:57,283] Trial 29 finished with values: [9874.764915331567, 0.42875000461935997, 0.457256942987442] and parameters: {'batch_size': 16, 'lr': 0.1, 'data_aug': True}. 


5
Best trial:
Value: [1.7687471449375152, 0.5344736902415752, 0.595277726650238]
Params: 
batch_size: 32
lr: 0.001
data_aug: True
Value: [2.668399049341679, 0.552368424832821, 0.5447222590446472]
Params: 
batch_size: 32
lr: 0.001
data_aug: False
Value: [31.788341869413852, 0.5294736908376216, 0.6463888883590698]
Params: 
batch_size: 32
lr: 0.01
data_aug: False
Value: [0.7002277365326881, 0.504736847281456, 0.6980000734329224]
Params: 
batch_size: 32
lr: 1e-05
data_aug: True
Value: [0.9955303949117662, 0.5392105320096017, 0.4815555512905121]
Params: 
batch_size: 32
lr: 0.0001
data_aug: True
Value: [0.7020361980795861, 0.5165789532661438, 0.35805556178092957]
Params: 
batch_size: 32
lr: 1e-05
data_aug: False
Value: [0.9294445821642876, 0.53236842289567, 0.47022223472595215]
Params: 
batch_size: 32
lr: 0.0001
data_aug: False


In [32]:
def save_study(study, study_name="study"):
    # Crear una carpeta para guardar el estudio si no existe
    study_dir = os.path.join(os.getcwd(), study_name)
    if not os.path.exists(study_dir):
        os.makedirs(study_dir)
    
    # Guardar los resultados del estudio en un archivo CSV
    study_trials = study.trials_dataframe()
    study_trials.to_csv(os.path.join(study_dir, "study_results.csv"), index=False)

def objective(trial):
    # Define los hiperparámetros para optimizar
    # num_epochs_freeze = trial.suggest_categorical('num_epochs_freeze', [1,3,8,10,15])
    # num_epochs_all = trial.suggest_categorical('num_epochs_all', [25,30,35,40,45,50])
    batch_size = trial.suggest_categorical('batch_size', [32, 16, 8, 4, 48, 5])
    lr = trial.suggest_categorical('lr', [1e-5,1e-4,1e-3,1e-2,1e-1])
    data_aug = trial.suggest_categorical('data_aug', [True,False])
    # ini = trial.suggest_categorical('ini', ['G', 'H', 'N', 'U', 'O', 'Z', 'ON', 'None'])
    # optimizer_name = trial.suggest_categorical('optimizer_name', ['RMSprop', 'Adam', 'SGD'])

    # # Convertir el nombre del optimizador en una clase PyTorch
    # if optimizer_name == 'Adam':
    #     optimizer_class = torch.optim.Adam
    # elif optimizer_name == 'SGD':
    #     optimizer_class = torch.optim.SGD
    # elif optimizer_name == 'RMSprop':
    #     optimizer_class = torch.optim.RMSprop

    # Ejecuta la función de entrenamiento con los hiperparámetros seleccionados
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_pre_oc(df_a90, 5, 5, 35, optim.RMSprop, batch_size, lr, data_aug, None)
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_pre(df_a90, 5, 5, 35, optim.SGD, batch_size, lr, data_aug, None)
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre(df_ct, 5, 40, optim.SGD, batch_size, lr, data_aug, None)
    avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre_oc(df_ct, 5, 40, optim.SGD, batch_size, lr, data_aug, None)

    # Devuelve el valor objetivo (en este caso, la pérdida promedio de validación)
    return avg_val_loss, avg_val_accuracy, avg_val_recall

import optuna
# Supongamos que tienes un dataset llamado 'dataset'
# Realiza una llamada a Optuna para optimizar los hiperparámetros
search_space = {"lr": [1e-5,1e-4,1e-3,1e-2, 1e-1],  'data_aug' : [True,False], 'batch_size' : [32, 16, 8],
                } ##Con este sampler se hacen todas las combinaciones posibles
study = optuna.create_study(directions = ['minimize','maximize','maximize'], sampler=optuna.samplers.GridSampler(search_space))#None, NSGA por defecto para multiobjetivo, sampler=optuna.samplers.GridSampler(search_space)
print(f"Sampler is {study.sampler.__class__.__name__}")
study.optimize(objective)#, n_trials=100

# Muestra los mejores hiperparámetros encontrados
print('Best trial:')
trial = study.best_trials
for t in trial:
    print('Value: {}'.format(t.values))
    print('Params: ')
    for key, value in t.params.items():
        print('{}: {}'.format(key, value))

save_study(study, 'ct_oc_sgd')

[I 2024-04-21 22:38:57,306] A new study created in memory with name: no-name-dc69839b-af38-48ce-af46-6c3423014004


Sampler is GridSampler
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-21 22:40:32,741] Trial 0 finished with values: [0.6986755073070525, 0.4965789531171321, 0.715999960899353] and parameters: {'batch_size': 32, 'lr': 0.0001, 'data_aug': True}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-21 22:42:06,088] Trial 1 finished with values: [0.7178443217277527, 0.47500001043081286, 0.15627777576446533] and parameters: {'batch_size': 32, 'lr': 1e-05, 'data_aug': False}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


... (52 outputs left out)

[I 2024-04-21 23:27:36,504] Trial 28 finished with values: [0.7080114349722862, 0.5004861147205035, 0.5805555582046509] and parameters: {'batch_size': 8, 'lr': 1e-05, 'data_aug': True}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-21 23:29:15,959] Trial 29 finished with values: [22.265964840017258, 0.4294270879030228, 0.4159027934074402] and parameters: {'batch_size': 16, 'lr': 0.1, 'data_aug': True}. 


5
Best trial:
Value: [0.6986755073070525, 0.4965789531171321, 0.715999960899353]
Params: 
batch_size: 32
lr: 0.0001
data_aug: True
Value: [0.6938243055343628, 0.531315795481205, 0.6662222146987915]
Params: 
batch_size: 32
lr: 0.001
data_aug: True
Value: [1.2685868822038173, 0.5376315855979918, 0.6747222542762756]
Params: 
batch_size: 32
lr: 0.01
data_aug: False
Value: [0.7025297206640243, 0.4978947442770004, 0.7230000495910645]
Params: 
batch_size: 32
lr: 0.0001
data_aug: False
Value: [0.7017488747835159, 0.5102083371082942, 0.6813055276870728]
Params: 
batch_size: 8
lr: 0.0001
data_aug: True
Value: [0.9656014090776445, 0.5242105342447758, 0.7293332815170288]
Params: 
batch_size: 32
lr: 0.01
data_aug: True
Value: [0.6967695090174675, 0.499210532605648, 0.6907222270965576]
Params: 
batch_size: 32
lr: 0.001
data_aug: False
Value: [0.7170923050244651, 0.45368055870135626, 0.7233333587646484]
Params: 
batch_size: 8
lr: 1e-05
data_aug: False


In [12]:
def save_study(study, study_name="study"):
    # Crear una carpeta para guardar el estudio si no existe
    study_dir = os.path.join(os.getcwd(), study_name)
    if not os.path.exists(study_dir):
        os.makedirs(study_dir)
    
    # Guardar los resultados del estudio en un archivo CSV
    study_trials = study.trials_dataframe()
    study_trials.to_csv(os.path.join(study_dir, "study_results.csv"), index=False)

def objective(trial):
    # Define los hiperparámetros para optimizar
    # num_epochs_freeze = trial.suggest_categorical('num_epochs_freeze', [1,3,8,10,15])
    # num_epochs_all = trial.suggest_categorical('num_epochs_all', [25,30,35,40,45,50])
    batch_size = trial.suggest_categorical('batch_size', [32, 16, 8, 4, 48, 5])
    lr = trial.suggest_categorical('lr', [1e-5,1e-4,1e-3,1e-2,1e-1])
    data_aug = trial.suggest_categorical('data_aug', [True,False])
    # ini = trial.suggest_categorical('ini', ['G', 'H', 'N', 'U', 'O', 'Z', 'ON', 'None'])
    # optimizer_name = trial.suggest_categorical('optimizer_name', ['RMSprop', 'Adam', 'SGD'])

    # # Convertir el nombre del optimizador en una clase PyTorch
    # if optimizer_name == 'Adam':
    #     optimizer_class = torch.optim.Adam
    # elif optimizer_name == 'SGD':
    #     optimizer_class = torch.optim.SGD
    # elif optimizer_name == 'RMSprop':
    #     optimizer_class = torch.optim.RMSprop

    # Ejecuta la función de entrenamiento con los hiperparámetros seleccionados
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_pre_oc(df_a90, 5, 5, 35, optim.RMSprop, batch_size, lr, data_aug, None)
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_pre(df_a90, 5, 5, 35, optim.SGD, batch_size, lr, data_aug, None)
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre(df_ct, 5, 40, optim.SGD, batch_size, lr, data_aug, None)
    avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre_oc(df_ct, 5, 40, optim.RMSprop, batch_size, lr, data_aug, None)

    # Devuelve el valor objetivo (en este caso, la pérdida promedio de validación)
    return avg_val_loss, avg_val_accuracy, avg_val_recall

import optuna
# Supongamos que tienes un dataset llamado 'dataset'
# Realiza una llamada a Optuna para optimizar los hiperparámetros
search_space = {"lr": [1e-5,1e-4,1e-3,1e-2, 1e-1],  'data_aug' : [True,False], 'batch_size' : [32, 16, 8],
                } ##Con este sampler se hacen todas las combinaciones posibles
study = optuna.create_study(directions = ['minimize','maximize','maximize'], sampler=optuna.samplers.GridSampler(search_space))#None, NSGA por defecto para multiobjetivo, sampler=optuna.samplers.GridSampler(search_space)
print(f"Sampler is {study.sampler.__class__.__name__}")
study.optimize(objective)#, n_trials=100

# Muestra los mejores hiperparámetros encontrados
print('Best trial:')
trial = study.best_trials
for t in trial:
    print('Value: {}'.format(t.values))
    print('Params: ')
    for key, value in t.params.items():
        print('{}: {}'.format(key, value))

save_study(study, 'ct_oc_rms')

[I 2024-04-22 09:57:21,513] A new study created in memory with name: no-name-8c38016e-7a07-42f2-a3fc-7c439d1f7d12


Sampler is GridSampler
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-22 09:59:19,577] Trial 0 finished with values: [1.4084429194017622, 0.49923611477017393, 0.4324999749660492] and parameters: {'batch_size': 8, 'lr': 1e-05, 'data_aug': False}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-22 10:00:52,695] Trial 1 finished with values: [14110497586068.95, 0.47421053737401964, 0.4786666929721832] and parameters: {'batch_size': 32, 'lr': 0.1, 'data_aug': False}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


... (52 outputs left out)

[I 2024-04-22 10:47:21,392] Trial 28 finished with values: [1.276936760991812, 0.4502083380520344, 0.285381942987442] and parameters: {'batch_size': 16, 'lr': 1e-05, 'data_aug': False}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-22 10:49:07,226] Trial 29 finished with values: [405.19222234215374, 0.4385937552154064, 0.4279513955116272] and parameters: {'batch_size': 16, 'lr': 0.001, 'data_aug': True}. 


5
Best trial:
Value: [1619.8871349200606, 0.47894737988710395, 0.6413888931274414]
Params: 
batch_size: 32
lr: 0.001
data_aug: False
Value: [1.2900819751868646, 0.5265277818342049, 0.4859444499015808]
Params: 
batch_size: 8
lr: 1e-05
data_aug: True
Value: [1.4593063228825727, 0.5027777814865112, 0.5209721922874451]
Params: 
batch_size: 8
lr: 0.0001
data_aug: True
Value: [6.148903629183769, 0.5189473764598369, 0.6174443960189819]
Params: 
batch_size: 32
lr: 0.0001
data_aug: True
Value: [1.2130023211240768, 0.5273684227466584, 0.4289444386959076]
Params: 
batch_size: 32
lr: 1e-05
data_aug: False
Value: [1.2228538823127746, 0.5197368468344212, 0.4415555000305176]
Params: 
batch_size: 32
lr: 1e-05
data_aug: True
Value: [4.077573686242103, 0.5044736905395985, 0.5586110949516296]
Params: 
batch_size: 32
lr: 0.0001
data_aug: False


In [13]:
def save_study(study, study_name="study"):
    # Crear una carpeta para guardar el estudio si no existe
    study_dir = os.path.join(os.getcwd(), study_name)
    if not os.path.exists(study_dir):
        os.makedirs(study_dir)
    
    # Guardar los resultados del estudio en un archivo CSV
    study_trials = study.trials_dataframe()
    study_trials.to_csv(os.path.join(study_dir, "study_results.csv"), index=False)

def objective(trial):
    # Define los hiperparámetros para optimizar
    # num_epochs_freeze = trial.suggest_categorical('num_epochs_freeze', [1,3,8,10,15])
    # num_epochs_all = trial.suggest_categorical('num_epochs_all', [25,30,35,40,45,50])
    batch_size = trial.suggest_categorical('batch_size', [32, 16, 8, 4, 48, 5])
    lr = trial.suggest_categorical('lr', [1e-5,1e-4,1e-3,1e-2,1e-1])
    data_aug = trial.suggest_categorical('data_aug', [True,False])
    # ini = trial.suggest_categorical('ini', ['G', 'H', 'N', 'U', 'O', 'Z', 'ON', 'None'])
    # optimizer_name = trial.suggest_categorical('optimizer_name', ['RMSprop', 'Adam', 'SGD'])

    # # Convertir el nombre del optimizador en una clase PyTorch
    # if optimizer_name == 'Adam':
    #     optimizer_class = torch.optim.Adam
    # elif optimizer_name == 'SGD':
    #     optimizer_class = torch.optim.SGD
    # elif optimizer_name == 'RMSprop':
    #     optimizer_class = torch.optim.RMSprop

    # Ejecuta la función de entrenamiento con los hiperparámetros seleccionados
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_pre_oc(df_a90, 5, 5, 35, optim.RMSprop, batch_size, lr, data_aug, None)
    avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_pre(df_ct, 5, 5, 35, optim.Adam, batch_size, lr, data_aug, None)
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre(df_ct, 5, 40, optim.SGD, batch_size, lr, data_aug, None)
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre_oc(df_ct, 5, 40, optim.Adam, batch_size, lr, data_aug, None)

    # Devuelve el valor objetivo (en este caso, la pérdida promedio de validación)
    return avg_val_loss, avg_val_accuracy, avg_val_recall

import optuna
# Supongamos que tienes un dataset llamado 'dataset'
# Realiza una llamada a Optuna para optimizar los hiperparámetros
search_space = {"lr": [1e-5,1e-4,1e-3,1e-2, 1e-1],  'data_aug' : [True,False], 'batch_size' : [32, 16, 8],
                } ##Con este sampler se hacen todas las combinaciones posibles
study = optuna.create_study(directions = ['minimize','maximize','maximize'], sampler=optuna.samplers.GridSampler(search_space))#None, NSGA por defecto para multiobjetivo, sampler=optuna.samplers.GridSampler(search_space)
print(f"Sampler is {study.sampler.__class__.__name__}")
study.optimize(objective)#, n_trials=100

# Muestra los mejores hiperparámetros encontrados
print('Best trial:')
trial = study.best_trials
for t in trial:
    print('Value: {}'.format(t.values))
    print('Params: ')
    for key, value in t.params.items():
        print('{}: {}'.format(key, value))

save_study(study, 'ct_tl_adam')

[I 2024-04-22 10:49:07,290] A new study created in memory with name: no-name-d690e434-49e8-4961-ae92-c27605acd5c5


Sampler is GridSampler
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-22 10:50:54,942] Trial 0 finished with values: [1.909648277834058, 0.4781944481531779, 0.44591665267944336] and parameters: {'batch_size': 8, 'lr': 0.001, 'data_aug': False}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-22 10:52:31,170] Trial 1 finished with values: [1169999143.9410958, 0.4414583386480809, 0.4384375214576721] and parameters: {'batch_size': 16, 'lr': 0.1, 'data_aug': True}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


... (52 outputs left out)

[I 2024-04-22 11:36:40,462] Trial 28 finished with values: [551004.8517428569, 0.4567361146708329, 0.5595000386238098] and parameters: {'batch_size': 8, 'lr': 0.1, 'data_aug': False}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-22 11:38:16,978] Trial 29 finished with values: [1.5054048097878694, 0.459010421410203, 0.44718748331069946] and parameters: {'batch_size': 16, 'lr': 0.0001, 'data_aug': True}. 


5
Best trial:
Value: [1.082759075164795, 0.5118421098589898, 0.5757777690887451]
Params: 
batch_size: 32
lr: 0.0001
data_aug: False
Value: [0.7788366031646728, 0.48868421450257304, 0.4909444749355316]
Params: 
batch_size: 32
lr: 1e-05
data_aug: False
Value: [2.7862442538142203, 0.517368425577879, 0.6022777557373047]
Params: 
batch_size: 32
lr: 0.001
data_aug: False
Value: [0.7622942706942558, 0.5455263166129589, 0.3825555443763733]
Params: 
batch_size: 32
lr: 1e-05
data_aug: True
Value: [1.1676032510399819, 0.5339473709464073, 0.5135555863380432]
Params: 
batch_size: 32
lr: 0.0001
data_aug: True


In [14]:
def save_study(study, study_name="study"):
    # Crear una carpeta para guardar el estudio si no existe
    study_dir = os.path.join(os.getcwd(), study_name)
    if not os.path.exists(study_dir):
        os.makedirs(study_dir)
    
    # Guardar los resultados del estudio en un archivo CSV
    study_trials = study.trials_dataframe()
    study_trials.to_csv(os.path.join(study_dir, "study_results.csv"), index=False)

def objective(trial):
    # Define los hiperparámetros para optimizar
    # num_epochs_freeze = trial.suggest_categorical('num_epochs_freeze', [1,3,8,10,15])
    # num_epochs_all = trial.suggest_categorical('num_epochs_all', [25,30,35,40,45,50])
    batch_size = trial.suggest_categorical('batch_size', [32, 16, 8, 4, 48, 5])
    lr = trial.suggest_categorical('lr', [1e-5,1e-4,1e-3,1e-2,1e-1])
    data_aug = trial.suggest_categorical('data_aug', [True,False])
    # ini = trial.suggest_categorical('ini', ['G', 'H', 'N', 'U', 'O', 'Z', 'ON', 'None'])
    # optimizer_name = trial.suggest_categorical('optimizer_name', ['RMSprop', 'Adam', 'SGD'])

    # # Convertir el nombre del optimizador en una clase PyTorch
    # if optimizer_name == 'Adam':
    #     optimizer_class = torch.optim.Adam
    # elif optimizer_name == 'SGD':
    #     optimizer_class = torch.optim.SGD
    # elif optimizer_name == 'RMSprop':
    #     optimizer_class = torch.optim.RMSprop

    # Ejecuta la función de entrenamiento con los hiperparámetros seleccionados
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_pre_oc(df_a90, 5, 5, 35, optim.RMSprop, batch_size, lr, data_aug, None)
    avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_pre(df_ct, 5, 5, 35, optim.SGD, batch_size, lr, data_aug, None)
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre(df_ct, 5, 40, optim.SGD, batch_size, lr, data_aug, None)
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre_oc(df_ct, 5, 40, optim.Adam, batch_size, lr, data_aug, None)

    # Devuelve el valor objetivo (en este caso, la pérdida promedio de validación)
    return avg_val_loss, avg_val_accuracy, avg_val_recall

import optuna
# Supongamos que tienes un dataset llamado 'dataset'
# Realiza una llamada a Optuna para optimizar los hiperparámetros
search_space = {"lr": [1e-5,1e-4,1e-3,1e-2, 1e-1],  'data_aug' : [True,False], 'batch_size' : [32, 16, 8],
                } ##Con este sampler se hacen todas las combinaciones posibles
study = optuna.create_study(directions = ['minimize','maximize','maximize'], sampler=optuna.samplers.GridSampler(search_space))#None, NSGA por defecto para multiobjetivo, sampler=optuna.samplers.GridSampler(search_space)
print(f"Sampler is {study.sampler.__class__.__name__}")
study.optimize(objective)#, n_trials=100

# Muestra los mejores hiperparámetros encontrados
print('Best trial:')
trial = study.best_trials
for t in trial:
    print('Value: {}'.format(t.values))
    print('Params: ')
    for key, value in t.params.items():
        print('{}: {}'.format(key, value))

save_study(study, 'ct_tl_sgd')

[I 2024-04-22 11:38:17,019] A new study created in memory with name: no-name-df446dba-304e-463f-b8ab-5ddde775c370


Sampler is GridSampler
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-22 11:39:46,778] Trial 0 finished with values: [11.899190936982631, 0.4900000119209289, 0.464722216129303] and parameters: {'batch_size': 32, 'lr': 0.1, 'data_aug': False}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-22 11:41:17,952] Trial 1 finished with values: [0.9110795757174494, 0.5263157910108566, 0.5367223024368286] and parameters: {'batch_size': 32, 'lr': 0.01, 'data_aug': True}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


... (52 outputs left out)

[I 2024-04-22 12:25:04,422] Trial 28 finished with values: [0.8183020220696926, 0.36395833708345887, 0.3432639241218567] and parameters: {'batch_size': 16, 'lr': 0.001, 'data_aug': False}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-22 12:26:38,094] Trial 29 finished with values: [0.6997171908617019, 0.5400000014901162, 0.5571111440658569] and parameters: {'batch_size': 32, 'lr': 0.001, 'data_aug': True}. 


5
Best trial:
Value: [0.7077611491084099, 0.5121052706241608, 0.5746110677719116]
Params: 
batch_size: 32
lr: 1e-05
data_aug: True
Value: [0.7293263119459154, 0.5183333372573058, 0.6348611116409302]
Params: 
batch_size: 8
lr: 1e-05
data_aug: True
Value: [0.7308244447410107, 0.5556250055134295, 0.57072913646698]
Params: 
batch_size: 16
lr: 1e-05
data_aug: True
Value: [0.6997171908617019, 0.5400000014901162, 0.5571111440658569]
Params: 
batch_size: 32
lr: 0.001
data_aug: True


In [15]:
def save_study(study, study_name="study"):
    # Crear una carpeta para guardar el estudio si no existe
    study_dir = os.path.join(os.getcwd(), study_name)
    if not os.path.exists(study_dir):
        os.makedirs(study_dir)
    
    # Guardar los resultados del estudio en un archivo CSV
    study_trials = study.trials_dataframe()
    study_trials.to_csv(os.path.join(study_dir, "study_results.csv"), index=False)

def objective(trial):
    # Define los hiperparámetros para optimizar
    # num_epochs_freeze = trial.suggest_categorical('num_epochs_freeze', [1,3,8,10,15])
    # num_epochs_all = trial.suggest_categorical('num_epochs_all', [25,30,35,40,45,50])
    batch_size = trial.suggest_categorical('batch_size', [32, 16, 8, 4, 48, 5])
    lr = trial.suggest_categorical('lr', [1e-5,1e-4,1e-3,1e-2,1e-1])
    data_aug = trial.suggest_categorical('data_aug', [True,False])
    # ini = trial.suggest_categorical('ini', ['G', 'H', 'N', 'U', 'O', 'Z', 'ON', 'None'])
    # optimizer_name = trial.suggest_categorical('optimizer_name', ['RMSprop', 'Adam', 'SGD'])

    # # Convertir el nombre del optimizador en una clase PyTorch
    # if optimizer_name == 'Adam':
    #     optimizer_class = torch.optim.Adam
    # elif optimizer_name == 'SGD':
    #     optimizer_class = torch.optim.SGD
    # elif optimizer_name == 'RMSprop':
    #     optimizer_class = torch.optim.RMSprop

    # Ejecuta la función de entrenamiento con los hiperparámetros seleccionados
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_pre_oc(df_a90, 5, 5, 35, optim.RMSprop, batch_size, lr, data_aug, None)
    avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_pre(df_ct, 5, 5, 35, optim.RMSprop, batch_size, lr, data_aug, None)
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre(df_ct, 5, 40, optim.SGD, batch_size, lr, data_aug, None)
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre_oc(df_ct, 5, 40, optim.Adam, batch_size, lr, data_aug, None)

    # Devuelve el valor objetivo (en este caso, la pérdida promedio de validación)
    return avg_val_loss, avg_val_accuracy, avg_val_recall

import optuna
# Supongamos que tienes un dataset llamado 'dataset'
# Realiza una llamada a Optuna para optimizar los hiperparámetros
search_space = {"lr": [1e-5,1e-4,1e-3,1e-2, 1e-1],  'data_aug' : [True,False], 'batch_size' : [32, 16, 8],
                } ##Con este sampler se hacen todas las combinaciones posibles
study = optuna.create_study(directions = ['minimize','maximize','maximize'], sampler=optuna.samplers.GridSampler(search_space))#None, NSGA por defecto para multiobjetivo, sampler=optuna.samplers.GridSampler(search_space)
print(f"Sampler is {study.sampler.__class__.__name__}")
study.optimize(objective)#, n_trials=100

# Muestra los mejores hiperparámetros encontrados
print('Best trial:')
trial = study.best_trials
for t in trial:
    print('Value: {}'.format(t.values))
    print('Params: ')
    for key, value in t.params.items():
        print('{}: {}'.format(key, value))

save_study(study, 'ct_tl_rms')

[I 2024-04-22 12:26:38,123] A new study created in memory with name: no-name-97556ace-26fa-4e72-9726-4991cb9f1ec2


Sampler is GridSampler
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-22 12:28:14,942] Trial 0 finished with values: [112927.86641502913, 0.4963541727513075, 0.4571875035762787] and parameters: {'batch_size': 16, 'lr': 0.01, 'data_aug': True}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-22 12:29:45,544] Trial 1 finished with values: [1552083429.376304, 0.4876315882802009, 0.5618888735771179] and parameters: {'batch_size': 32, 'lr': 0.1, 'data_aug': False}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


... (52 outputs left out)

[I 2024-04-22 13:14:00,313] Trial 28 finished with values: [38.06910205403964, 0.4808333371082941, 0.49080556631088257] and parameters: {'batch_size': 8, 'lr': 0.1, 'data_aug': True}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-22 13:15:35,512] Trial 29 finished with values: [72477.04417359464, 0.46572917200624947, 0.5006250143051147] and parameters: {'batch_size': 16, 'lr': 0.001, 'data_aug': False}. 


5
Best trial:
Value: [0.8609581753611565, 0.5405263210833072, 0.48572224378585815]
Params: 
batch_size: 32
lr: 1e-05
data_aug: True
Value: [3378582910.864169, 0.48973685175180437, 0.6121110916137695]
Params: 
batch_size: 32
lr: 0.1
data_aug: True
Value: [326856.9826806143, 0.48842106178402905, 0.5843888521194458]
Params: 
batch_size: 32
lr: 0.001
data_aug: True
Value: [0.878397653400898, 0.5310526365041733, 0.51583331823349]
Params: 
batch_size: 32
lr: 1e-05
data_aug: False
Value: [1.179846449792385, 0.5589473721385001, 0.5812222361564636]
Params: 
batch_size: 32
lr: 0.0001
data_aug: False


In [16]:
def save_study(study, study_name="study"):
    # Crear una carpeta para guardar el estudio si no existe
    study_dir = os.path.join(os.getcwd(), study_name)
    if not os.path.exists(study_dir):
        os.makedirs(study_dir)
    
    # Guardar los resultados del estudio en un archivo CSV
    study_trials = study.trials_dataframe()
    study_trials.to_csv(os.path.join(study_dir, "study_results.csv"), index=False)

def objective(trial):
    # Define los hiperparámetros para optimizar
    # num_epochs_freeze = trial.suggest_categorical('num_epochs_freeze', [1,3,8,10,15])
    # num_epochs_all = trial.suggest_categorical('num_epochs_all', [25,30,35,40,45,50])
    batch_size = trial.suggest_categorical('batch_size', [32, 16, 8, 4, 48, 5])
    lr = trial.suggest_categorical('lr', [1e-5,1e-4,1e-3,1e-2,1e-1])
    data_aug = trial.suggest_categorical('data_aug', [True,False])
    # ini = trial.suggest_categorical('ini', ['G', 'H', 'N', 'U', 'O', 'Z', 'ON', 'None'])
    # optimizer_name = trial.suggest_categorical('optimizer_name', ['RMSprop', 'Adam', 'SGD'])

    # # Convertir el nombre del optimizador en una clase PyTorch
    # if optimizer_name == 'Adam':
    #     optimizer_class = torch.optim.Adam
    # elif optimizer_name == 'SGD':
    #     optimizer_class = torch.optim.SGD
    # elif optimizer_name == 'RMSprop':
    #     optimizer_class = torch.optim.RMSprop

    # Ejecuta la función de entrenamiento con los hiperparámetros seleccionados
    avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_pre_oc(df_ct, 5, 5, 35, optim.Adam, batch_size, lr, data_aug, None)
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_pre(df_ct, 5, 5, 35, optim.Adam, batch_size, lr, data_aug, None)
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre(df_ct, 5, 40, optim.SGD, batch_size, lr, data_aug, None)
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre_oc(df_ct, 5, 40, optim.Adam, batch_size, lr, data_aug, None)

    # Devuelve el valor objetivo (en este caso, la pérdida promedio de validación)
    return avg_val_loss, avg_val_accuracy, avg_val_recall

import optuna
# Supongamos que tienes un dataset llamado 'dataset'
# Realiza una llamada a Optuna para optimizar los hiperparámetros
search_space = {"lr": [1e-5,1e-4,1e-3,1e-2, 1e-1],  'data_aug' : [True,False], 'batch_size' : [32, 16, 8],
                } ##Con este sampler se hacen todas las combinaciones posibles
study = optuna.create_study(directions = ['minimize','maximize','maximize'], sampler=optuna.samplers.GridSampler(search_space))#None, NSGA por defecto para multiobjetivo, sampler=optuna.samplers.GridSampler(search_space)
print(f"Sampler is {study.sampler.__class__.__name__}")
study.optimize(objective)#, n_trials=100

# Muestra los mejores hiperparámetros encontrados
print('Best trial:')
trial = study.best_trials
for t in trial:
    print('Value: {}'.format(t.values))
    print('Params: ')
    for key, value in t.params.items():
        print('{}: {}'.format(key, value))

save_study(study, 'ct_tl_oc_adam')

[I 2024-04-22 13:15:35,542] A new study created in memory with name: no-name-9db8f16f-3e39-4176-bb41-6bd750e0c6e9


Sampler is GridSampler
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-22 13:17:24,279] Trial 0 finished with values: [1.1306193551669517, 0.5188888930777709, 0.4624166488647461] and parameters: {'batch_size': 8, 'lr': 0.01, 'data_aug': True}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-22 13:19:16,357] Trial 1 finished with values: [23.412263093193367, 0.4702777810394764, 0.49347224831581116] and parameters: {'batch_size': 8, 'lr': 0.1, 'data_aug': True}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


... (52 outputs left out)

[I 2024-04-22 14:06:01,614] Trial 28 finished with values: [0.9423464128871759, 0.46875000407298406, 0.30558332800865173] and parameters: {'batch_size': 8, 'lr': 0.0001, 'data_aug': False}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-22 14:07:33,848] Trial 29 finished with values: [2.9615772403776646, 0.5352631641924381, 0.5645555257797241] and parameters: {'batch_size': 32, 'lr': 0.01, 'data_aug': True}. 


5
Best trial:
Value: [0.7398436567187309, 0.46184211209416387, 0.7391111254692078]
Params: 
batch_size: 32
lr: 1e-05
data_aug: True
Value: [0.7068896095454693, 0.5562500063329934, 0.6359722018241882]
Params: 
batch_size: 16
lr: 1e-05
data_aug: True


In [17]:
def save_study(study, study_name="study"):
    # Crear una carpeta para guardar el estudio si no existe
    study_dir = os.path.join(os.getcwd(), study_name)
    if not os.path.exists(study_dir):
        os.makedirs(study_dir)
    
    # Guardar los resultados del estudio en un archivo CSV
    study_trials = study.trials_dataframe()
    study_trials.to_csv(os.path.join(study_dir, "study_results.csv"), index=False)

def objective(trial):
    # Define los hiperparámetros para optimizar
    # num_epochs_freeze = trial.suggest_categorical('num_epochs_freeze', [1,3,8,10,15])
    # num_epochs_all = trial.suggest_categorical('num_epochs_all', [25,30,35,40,45,50])
    batch_size = trial.suggest_categorical('batch_size', [32, 16, 8, 4, 48, 5])
    lr = trial.suggest_categorical('lr', [1e-5,1e-4,1e-3,1e-2,1e-1])
    data_aug = trial.suggest_categorical('data_aug', [True,False])
    # ini = trial.suggest_categorical('ini', ['G', 'H', 'N', 'U', 'O', 'Z', 'ON', 'None'])
    # optimizer_name = trial.suggest_categorical('optimizer_name', ['RMSprop', 'Adam', 'SGD'])

    # # Convertir el nombre del optimizador en una clase PyTorch
    # if optimizer_name == 'Adam':
    #     optimizer_class = torch.optim.Adam
    # elif optimizer_name == 'SGD':
    #     optimizer_class = torch.optim.SGD
    # elif optimizer_name == 'RMSprop':
    #     optimizer_class = torch.optim.RMSprop

    # Ejecuta la función de entrenamiento con los hiperparámetros seleccionados
    avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_pre_oc(df_ct, 5, 5, 35, optim.SGD, batch_size, lr, data_aug, None)
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_pre(df_ct, 5, 5, 35, optim.Adam, batch_size, lr, data_aug, None)
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre(df_ct, 5, 40, optim.SGD, batch_size, lr, data_aug, None)
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre_oc(df_ct, 5, 40, optim.Adam, batch_size, lr, data_aug, None)

    # Devuelve el valor objetivo (en este caso, la pérdida promedio de validación)
    return avg_val_loss, avg_val_accuracy, avg_val_recall

import optuna
# Supongamos que tienes un dataset llamado 'dataset'
# Realiza una llamada a Optuna para optimizar los hiperparámetros
search_space = {"lr": [1e-5,1e-4,1e-3,1e-2, 1e-1],  'data_aug' : [True,False], 'batch_size' : [32, 16, 8],
                } ##Con este sampler se hacen todas las combinaciones posibles
study = optuna.create_study(directions = ['minimize','maximize','maximize'], sampler=optuna.samplers.GridSampler(search_space))#None, NSGA por defecto para multiobjetivo, sampler=optuna.samplers.GridSampler(search_space)
print(f"Sampler is {study.sampler.__class__.__name__}")
study.optimize(objective)#, n_trials=100

# Muestra los mejores hiperparámetros encontrados
print('Best trial:')
trial = study.best_trials
for t in trial:
    print('Value: {}'.format(t.values))
    print('Params: ')
    for key, value in t.params.items():
        print('{}: {}'.format(key, value))

save_study(study, 'ct_tl_oc_sgd')

[I 2024-04-22 14:07:33,878] A new study created in memory with name: no-name-86c7da30-06d6-4fc0-81a7-fca9921ae13a


Sampler is GridSampler
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-22 14:09:09,740] Trial 0 finished with values: [0.7511515992879867, 0.4348437564820051, 0.1598263829946518] and parameters: {'batch_size': 16, 'lr': 0.0001, 'data_aug': True}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-22 14:10:53,020] Trial 1 finished with values: [2.9694119400158527, 0.5042361147205034, 0.4216388761997223] and parameters: {'batch_size': 8, 'lr': 0.1, 'data_aug': False}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


... (52 outputs left out)

[I 2024-04-22 14:54:15,483] Trial 28 finished with values: [0.7276895853877068, 0.49447369739413266, 0.699999988079071] and parameters: {'batch_size': 32, 'lr': 0.0001, 'data_aug': False}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-22 14:55:46,952] Trial 29 finished with values: [113.88419046729805, 0.5297368454933167, 0.5804444551467896] and parameters: {'batch_size': 32, 'lr': 0.1, 'data_aug': True}. 


5
Best trial:
Value: [0.7279311281442642, 0.5689473760128021, 0.6062778234481812]
Params: 
batch_size: 32
lr: 1e-05
data_aug: False
Value: [0.7078366279602051, 0.5679687558114528, 0.6302777528762817]
Params: 
batch_size: 16
lr: 1e-05
data_aug: True
Value: [0.7755181393027306, 0.4953125043213368, 0.8469097018241882]
Params: 
batch_size: 16
lr: 1e-05
data_aug: False
Value: [0.6976042342185973, 0.5649999982118606, 0.6145000457763672]
Params: 
batch_size: 32
lr: 1e-05
data_aug: True
Value: [0.688392043809096, 0.5344444489975769, 0.45980557799339294]
Params: 
batch_size: 8
lr: 0.0001
data_aug: True
Value: [0.7276895853877068, 0.49447369739413266, 0.699999988079071]
Params: 
batch_size: 32
lr: 0.0001
data_aug: False


In [11]:
def save_study(study, study_name="study"):
    # Crear una carpeta para guardar el estudio si no existe
    study_dir = os.path.join(os.getcwd(), study_name)
    if not os.path.exists(study_dir):
        os.makedirs(study_dir)
    
    # Guardar los resultados del estudio en un archivo CSV
    study_trials = study.trials_dataframe()
    study_trials.to_csv(os.path.join(study_dir, "study_results.csv"), index=False)

def objective(trial):
    # Define los hiperparámetros para optimizar
    # num_epochs_freeze = trial.suggest_categorical('num_epochs_freeze', [1,3,8,10,15])
    # num_epochs_all = trial.suggest_categorical('num_epochs_all', [25,30,35,40,45,50])
    batch_size = trial.suggest_categorical('batch_size', [32, 16, 8, 4, 48, 5])
    lr = trial.suggest_categorical('lr', [1e-5,1e-4,1e-3,1e-2,1e-1])
    data_aug = trial.suggest_categorical('data_aug', [True,False])
    # ini = trial.suggest_categorical('ini', ['G', 'H', 'N', 'U', 'O', 'Z', 'ON', 'None'])
    # optimizer_name = trial.suggest_categorical('optimizer_name', ['RMSprop', 'Adam', 'SGD'])

    # # Convertir el nombre del optimizador en una clase PyTorch
    # if optimizer_name == 'Adam':
    #     optimizer_class = torch.optim.Adam
    # elif optimizer_name == 'SGD':
    #     optimizer_class = torch.optim.SGD
    # elif optimizer_name == 'RMSprop':
    #     optimizer_class = torch.optim.RMSprop

    # Ejecuta la función de entrenamiento con los hiperparámetros seleccionados
    avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_pre_oc(df_ct, 5, 5, 35, optim.RMSprop, batch_size, lr, data_aug, None)
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_pre(df_ct, 5, 5, 35, optim.Adam, batch_size, lr, data_aug, None)
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre(df_ct, 5, 40, optim.SGD, batch_size, lr, data_aug, None)
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre_oc(df_ct, 5, 40, optim.Adam, batch_size, lr, data_aug, None)

    # Devuelve el valor objetivo (en este caso, la pérdida promedio de validación)
    return avg_val_loss, avg_val_accuracy, avg_val_recall

import optuna
# Supongamos que tienes un dataset llamado 'dataset'
# Realiza una llamada a Optuna para optimizar los hiperparámetros
search_space = {"lr": [1e-5,1e-4,1e-3,1e-2, 1e-1],  'data_aug' : [True,False], 'batch_size' : [32, 16, 8],
                } ##Con este sampler se hacen todas las combinaciones posibles
study = optuna.create_study(directions = ['minimize','maximize','maximize'], sampler=optuna.samplers.GridSampler(search_space))#None, NSGA por defecto para multiobjetivo, sampler=optuna.samplers.GridSampler(search_space)
print(f"Sampler is {study.sampler.__class__.__name__}")
study.optimize(objective)#, n_trials=100

# Muestra los mejores hiperparámetros encontrados
print('Best trial:')
trial = study.best_trials
for t in trial:
    print('Value: {}'.format(t.values))
    print('Params: ')
    for key, value in t.params.items():
        print('{}: {}'.format(key, value))

save_study(study, 'ct_tl_oc_rms')

c:\Users\da01m\anaconda3\envs\tfg_cnn_lv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2024-04-22 20:12:20,713] A new study created in memory with name: no-name-5ab8faa7-055e-4c7a-ac20-d3b7ba205964


Sampler is GridSampler
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-22 20:13:56,642] Trial 0 finished with values: [18379196.631303817, 0.49000000953674316, 0.4869999885559082] and parameters: {'batch_size': 32, 'lr': 0.01, 'data_aug': True}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-22 20:15:29,084] Trial 1 finished with values: [1.6937520381435753, 0.452968754991889, 0.3903124928474426] and parameters: {'batch_size': 16, 'lr': 0.0001, 'data_aug': False}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


... (52 outputs left out)

[I 2024-04-22 20:59:26,707] Trial 28 finished with values: [8.947455696314574, 0.4732812558859586, 0.35218751430511475] and parameters: {'batch_size': 16, 'lr': 0.001, 'data_aug': False}. 


5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2024-04-22 21:01:13,136] Trial 29 finished with values: [0.9247114451229572, 0.49590278133749965, 0.43022221326828003] and parameters: {'batch_size': 8, 'lr': 1e-05, 'data_aug': False}. 


5
Best trial:
Value: [0.8061868402361869, 0.5181579019129277, 0.4685555398464203]
Params: 
batch_size: 32
lr: 1e-05
data_aug: False
Value: [128.63646035909653, 0.513947376012802, 0.6248888969421387]
Params: 
batch_size: 32
lr: 0.001
data_aug: False
Value: [1.2834962236881255, 0.546842105537653, 0.6027222275733948]
Params: 
batch_size: 32
lr: 0.0001
data_aug: True
